# Immediate punishment and later damage persistence

Human move probabilities predict whether an opponent's immediate reply preserves a mistake's damage. Predicting the change in damage over the next 2, 4, 6 or 10 plies produces smaller improvements over no-change. Later moves by both players can alter the position, so preservation after one reply does not determine later persistence.

In game-grouped validation on four earlier archives, raw Maia-2 achieved 73.06% accuracy against 55.81% for the majority baseline, with ROC-AUC 0.7997. On all five archives, the fixed LightGBM reduced regression MAE by 0.14% at two plies and 2.48% at ten plies. Each task is assessed against its own baseline; classification accuracy and MAE reduction have different meanings.

Change in damage remains the primary research target. All results in this notebook are exploratory development estimates. The separate March 2025 final evaluation was completed on 21 November 2026; see [the final-results notebook](chess_final_results.ipynb). The tables are embedded in this notebook and are displayed without external data files or metric recalculation.

[Data and splits](#1.-Data-and-splits) · [Immediate reply](#2.-Immediate-opponent-reply) · [Later persistence](#3.-Longer-horizon-persistence) · [Supplementary analyses](#4.-Supplementary-development-analyses) · [Sources](#5.-Sources-and-reproducibility)


In [1]:
from IPython.display import HTML, Markdown, display

def show(title, table_html, note=""):
    label = title.replace("_", " ").replace("maia", "Maia").replace("lightgbm", "LightGBM").replace("xgboost", "XGBoost").replace("multipv", "MultiPV")
    display(Markdown(f"**{label[0].upper() + label[1:]}**"))
    if note:
        display(Markdown(note))
    display(HTML(table_html))


## 1. Data and splits

The five development archives are April, June and July 2017, February 2018 and February 2019. Maia-2 comparisons use the first four months, which predate its documented May 2018–November 2023 training period. This restriction is based on dates; individual external training-game or player membership has not been audited. The earlier five-month policy pilot is reported separately because February 2019 may overlap pretraining.

Eligibility is `10 <= win_before <= 90` and `initial_damage >20` winning-chance percentage points. Removing 510 overlapping observations from 23,677 rows left 23,167 mistakes from 6,512 games. Mistakes are identified by `(source_pgn, game_index, ply)` and games by verified `site`.

All evaluations use the original mistake-maker's perspective:

```text
initial_damage = win_before - win_after
remaining_damage_h = win_before - win_after_h
change_in_damage_h = win_after - win_after_h
persistence_ratio_h = 1 + change_in_damage_h / initial_damage
```

The regression horizons are 2, 4, 6 and 10 subsequent plies. Positive change means increasing damage; negative change means recovery. Ratios are unclipped. Descriptive categories are mostly recovered <0.25, partially persistent 0.25–<0.75, strongly persistent 0.75–1.0 and worsened >1.0. Binary recovery at ratio <0.75 is an exploratory project-specific definition.

The immediate-reply label is whether the actual next reply retains at least 75% of initial damage. Missing replies are excluded rather than labeled negative. Later horizons retain the implemented terminal-result targets. PGN results alone cannot identify abandonment or disconnection.

Core evaluations use depth 18 and `100 / (1 + exp(-0.00368208 * cp))`, with mates mapped to ±1000 cp. The historical core engine version was unrecorded. The reply analyses used Stockfish 19, one thread and 64 MiB hash. The legacy MultiPV extractor used ±100000 cp for mates; its saved features retain that convention. MultiPV-5 was evaluated after the mistake and before the reply. The 76 valid missing best–second gaps correspond to positions with one legal reply.

Validation uses three game-grouped folds. The player-disjoint protocol also removes whole training games involving either held-out player. Imputation, encoding and any inner selection use training folds only. Archive cohorts and eligibility populations are reported separately. Raw Maia probabilities are identical across the local protocols because the external policy is fixed; this does not establish disjointness from its training players.


In [2]:
show('data_by_archive',
'<table border="1" class="dataframe">\n  <thead>\n    <tr style="text-align: right;">\n      <th>archive_set</th>\n      <th>protocol</th>\n      <th>source_pgn</th>\n      <th>mistakes</th>\n      <th>games</th>\n    </tr>\n  </thead>\n  <tbody>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Canonical development, before validation</td>\n      <td>lichess_db_standard_rated_2017-04.pgn.zst</td>\n      <td>5254</td>\n      <td>1458</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Canonical development, before validation</td>\n      <td>lichess_db_standard_rated_2017-06.pgn.zst</td>\n      <td>3280</td>\n      <td>937</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Canonical development, before validation</td>\n      <td>lichess_db_standard_rated_2017-07.pgn.zst</td>\n      <td>3506</td>\n      <td>1008</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Canonical development, before validation</td>\n      <td>lichess_db_standard_rated_2018-02.pgn.zst</td>\n      <td>3913</td>\n      <td>1082</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Canonical development, before validation</td>\n      <td>lichess_db_standard_rated_2019-02.pgn.zst</td>\n      <td>7214</td>\n      <td>2027</td>\n    </tr>\n  </tbody>\n</table>',
'')

show('analysis_populations',
'<table border="1" class="dataframe">\n  <thead>\n    <tr style="text-align: right;">\n      <th>population</th>\n      <th>protocol</th>\n      <th>mistakes</th>\n      <th>games</th>\n      <th>excluded_missing_replies</th>\n      <th>archive_set</th>\n    </tr>\n  </thead>\n  <tbody>\n    <tr>\n      <td>Primary persistence</td>\n      <td>Counts only; archive_set specifies each population</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>unavailable / undefined</td>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n    </tr>\n    <tr>\n      <td>Four-archive Maia later horizons</td>\n      <td>Counts only; archive_set specifies each population</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>unavailable / undefined</td>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n    </tr>\n    <tr>\n      <td>Four-archive immediate reply</td>\n      <td>Counts only; archive_set specifies each population</td>\n      <td>15851</td>\n      <td>4472</td>\n      <td>102.000000</td>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n    </tr>\n    <tr>\n      <td>Five-archive continuous reply targets</td>\n      <td>Counts only; archive_set specifies each population</td>\n      <td>23023</td>\n      <td>6495</td>\n      <td>144.000000</td>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n    </tr>\n  </tbody>\n</table>',
'')

show('primary_outer_split_counts',
'<table border="1" class="dataframe">\n  <thead>\n    <tr style="text-align: right;">\n      <th>archive_set</th>\n      <th>protocol</th>\n      <th>scope</th>\n      <th>outer_fold</th>\n      <th>level</th>\n      <th>inner_fold</th>\n      <th>train_rows</th>\n      <th>valid_rows</th>\n      <th>train_games</th>\n      <th>valid_games</th>\n      <th>game_overlap</th>\n      <th>player_overlap</th>\n    </tr>\n  </thead>\n  <tbody>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>all</td>\n      <td>1</td>\n      <td>outer</td>\n      <td>0</td>\n      <td>15538</td>\n      <td>7629</td>\n      <td>4341</td>\n      <td>2171</td>\n      <td>0</td>\n      <td>unavailable / undefined</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>all</td>\n      <td>2</td>\n      <td>outer</td>\n      <td>0</td>\n      <td>15424</td>\n      <td>7743</td>\n      <td>4341</td>\n      <td>2171</td>\n      <td>0</td>\n      <td>unavailable / undefined</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>all</td>\n      <td>3</td>\n      <td>outer</td>\n      <td>0</td>\n      <td>15372</td>\n      <td>7795</td>\n      <td>4342</td>\n      <td>2170</td>\n      <td>0</td>\n      <td>unavailable / undefined</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>all</td>\n      <td>1</td>\n      <td>outer</td>\n      <td>0</td>\n      <td>12641</td>\n      <td>7629</td>\n      <td>3526</td>\n      <td>2171</td>\n      <td>0</td>\n      <td>0.000000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>all</td>\n      <td>2</td>\n      <td>outer</td>\n      <td>0</td>\n      <td>12504</td>\n      <td>7743</td>\n      <td>3511</td>\n      <td>2171</td>\n      <td>0</td>\n      <td>0.000000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>all</td>\n      <td>3</td>\n      <td>outer</td>\n      <td>0</td>\n      <td>12567</td>\n      <td>7795</td>\n      <td>3524</td>\n      <td>2170</td>\n      <td>0</td>\n      <td>0.000000</td>\n    </tr>\n  </tbody>\n</table>',
'')


**Data by archive**

archive_set,protocol,source_pgn,mistakes,games
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02","Canonical development, before validation",lichess_db_standard_rated_2017-04.pgn.zst,5254,1458
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02","Canonical development, before validation",lichess_db_standard_rated_2017-06.pgn.zst,3280,937
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02","Canonical development, before validation",lichess_db_standard_rated_2017-07.pgn.zst,3506,1008
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02","Canonical development, before validation",lichess_db_standard_rated_2018-02.pgn.zst,3913,1082
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02","Canonical development, before validation",lichess_db_standard_rated_2019-02.pgn.zst,7214,2027


**Analysis populations**

population,protocol,mistakes,games,excluded_missing_replies,archive_set
Primary persistence,Counts only; archive_set specifies each population,23167,6512,unavailable / undefined,"2017-04, 2017-06, 2017-07, 2018-02, 2019-02"
Four-archive Maia later horizons,Counts only; archive_set specifies each population,15953,4485,unavailable / undefined,"2017-04, 2017-06, 2017-07, 2018-02"
Four-archive immediate reply,Counts only; archive_set specifies each population,15851,4472,102.000000,"2017-04, 2017-06, 2017-07, 2018-02"
Five-archive continuous reply targets,Counts only; archive_set specifies each population,23023,6495,144.000000,"2017-04, 2017-06, 2017-07, 2018-02, 2019-02"


**Primary outer split counts**

archive_set,protocol,scope,outer_fold,level,inner_fold,train_rows,valid_rows,train_games,valid_games,game_overlap,player_overlap
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,all,1,outer,0,15538,7629,4341,2171,0,unavailable / undefined
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,all,2,outer,0,15424,7743,4341,2171,0,unavailable / undefined
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,all,3,outer,0,15372,7795,4342,2170,0,unavailable / undefined
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",player,all,1,outer,0,12641,7629,3526,2171,0,0.000000
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",player,all,2,outer,0,12504,7743,3511,2171,0,0.000000
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",player,all,3,outer,0,12567,7795,3524,2170,0,0.000000


## 2. Immediate opponent reply

The positive class is a reply that preserves at least 75% of the initial damage. Maia-2 estimates its probability by summing policy mass over qualifying hypothetical replies. The saved approximation evaluates a 97.5% policy-mass prefix and uses the midpoint of the residual bounds, without renormalizing the selected mass.

The LightGBM control uses `initial_damage` and `opponent_elo`; the augmented model adds only the Maia probability. Both use identical classifier settings, folds, imputation and a 0.5 cutoff. The analysis includes 15,851 valid replies from 4,472 games across four archives, excluding 102 missing replies.

On game folds, Brier score was 0.184310 for raw Maia, 0.244979 for the damage/Elo control and 0.180147 after adding Maia. Brier is the primary metric for the incremental probability comparison. Both class recalls accompany accuracy. The event measures the reply played, not whether the player consciously noticed the mistake. Logistic sensitivity results appear in Section 4.


In [3]:
show('immediate_reply_core_comparison',
'<table border="1" class="dataframe">\n  <thead>\n    <tr style="text-align: right;">\n      <th>archive_set</th>\n      <th>protocol</th>\n      <th>method</th>\n      <th>games</th>\n      <th>n</th>\n      <th>accuracy</th>\n      <th>balanced_accuracy</th>\n      <th>precision</th>\n      <th>preserving_recall</th>\n      <th>non_preserving_recall</th>\n      <th>tn</th>\n      <th>fp</th>\n      <th>fn</th>\n      <th>tp</th>\n      <th>roc_auc</th>\n      <th>brier</th>\n    </tr>\n  </thead>\n  <tbody>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>raw_maia</td>\n      <td>4472</td>\n      <td>15851</td>\n      <td>0.730553</td>\n      <td>0.721247</td>\n      <td>0.738284</td>\n      <td>0.801289</td>\n      <td>0.641205</td>\n      <td>4491</td>\n      <td>2513</td>\n      <td>1758</td>\n      <td>7089</td>\n      <td>0.799701</td>\n      <td>0.184310</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>simple_lightgbm</td>\n      <td>4472</td>\n      <td>15851</td>\n      <td>0.560280</td>\n      <td>0.527813</td>\n      <td>0.575667</td>\n      <td>0.807053</td>\n      <td>0.248572</td>\n      <td>1741</td>\n      <td>5263</td>\n      <td>1707</td>\n      <td>7140</td>\n      <td>0.556793</td>\n      <td>0.244979</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>simple_plus_maia_lightgbm</td>\n      <td>4472</td>\n      <td>15851</td>\n      <td>0.726768</td>\n      <td>0.716711</td>\n      <td>0.732880</td>\n      <td>0.803210</td>\n      <td>0.630211</td>\n      <td>4414</td>\n      <td>2590</td>\n      <td>1741</td>\n      <td>7106</td>\n      <td>0.798525</td>\n      <td>0.180147</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>training_majority_prevalence</td>\n      <td>4472</td>\n      <td>15851</td>\n      <td>0.558135</td>\n      <td>0.500000</td>\n      <td>0.558135</td>\n      <td>1.000000</td>\n      <td>0.000000</td>\n      <td>0</td>\n      <td>7004</td>\n      <td>0</td>\n      <td>8847</td>\n      <td>0.492869</td>\n      <td>0.246679</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>raw_maia</td>\n      <td>4472</td>\n      <td>15851</td>\n      <td>0.730553</td>\n      <td>0.721247</td>\n      <td>0.738284</td>\n      <td>0.801289</td>\n      <td>0.641205</td>\n      <td>4491</td>\n      <td>2513</td>\n      <td>1758</td>\n      <td>7089</td>\n      <td>0.799701</td>\n      <td>0.184310</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>simple_lightgbm</td>\n      <td>4472</td>\n      <td>15851</td>\n      <td>0.559523</td>\n      <td>0.526049</td>\n      <td>0.574380</td>\n      <td>0.813948</td>\n      <td>0.238150</td>\n      <td>1668</td>\n      <td>5336</td>\n      <td>1646</td>\n      <td>7201</td>\n      <td>0.552592</td>\n      <td>0.245956</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>simple_plus_maia_lightgbm</td>\n      <td>4472</td>\n      <td>15851</td>\n      <td>0.724308</td>\n      <td>0.712796</td>\n      <td>0.726408</td>\n      <td>0.811801</td>\n      <td>0.613792</td>\n      <td>4299</td>\n      <td>2705</td>\n      <td>1665</td>\n      <td>7182</td>\n      <td>0.796549</td>\n      <td>0.181277</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>training_majority_prevalence</td>\n      <td>4472</td>\n      <td>15851</td>\n      <td>0.558135</td>\n      <td>0.500000</td>\n      <td>0.558135</td>\n      <td>1.000000</td>\n      <td>0.000000</td>\n      <td>0</td>\n      <td>7004</td>\n      <td>0</td>\n      <td>8847</td>\n      <td>0.494322</td>\n      <td>0.246685</td>\n    </tr>\n  </tbody>\n</table>',
'Positive class: preserving. ROC-AUC uses continuous probabilities.')

show('immediate_reply_incremental_maia',
'<table border="1" class="dataframe">\n  <thead>\n    <tr style="text-align: right;">\n      <th>archive_set</th>\n      <th>protocol</th>\n      <th>initial_damage_threshold</th>\n      <th>augmented</th>\n      <th>baseline</th>\n      <th>metric</th>\n      <th>estimate</th>\n      <th>lower95</th>\n      <th>upper95</th>\n      <th>n</th>\n      <th>games</th>\n      <th>bootstrap_repeats</th>\n    </tr>\n  </thead>\n  <tbody>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>20</td>\n      <td>simple_plus_maia_lightgbm</td>\n      <td>simple_lightgbm</td>\n      <td>brier_improvement</td>\n      <td>0.064832</td>\n      <td>0.061650</td>\n      <td>0.067999</td>\n      <td>15851</td>\n      <td>4472</td>\n      <td>2000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>20</td>\n      <td>simple_plus_maia_lightgbm</td>\n      <td>simple_lightgbm</td>\n      <td>log_loss_improvement</td>\n      <td>0.146175</td>\n      <td>0.137675</td>\n      <td>0.154050</td>\n      <td>15851</td>\n      <td>4472</td>\n      <td>2000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>20</td>\n      <td>simple_plus_maia_lightgbm</td>\n      <td>simple_lightgbm</td>\n      <td>accuracy_gain_pp</td>\n      <td>16.648792</td>\n      <td>15.714662</td>\n      <td>17.627615</td>\n      <td>15851</td>\n      <td>4472</td>\n      <td>2000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>20</td>\n      <td>simple_plus_maia_lightgbm</td>\n      <td>simple_lightgbm</td>\n      <td>balanced_accuracy_gain_pp</td>\n      <td>18.889798</td>\n      <td>17.962601</td>\n      <td>19.871046</td>\n      <td>15851</td>\n      <td>4472</td>\n      <td>2000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>20</td>\n      <td>simple_plus_maia_lightgbm</td>\n      <td>simple_lightgbm</td>\n      <td>preserving_recall_gain_pp</td>\n      <td>-0.384311</td>\n      <td>-1.506613</td>\n      <td>0.822033</td>\n      <td>15851</td>\n      <td>4472</td>\n      <td>2000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>20</td>\n      <td>simple_plus_maia_lightgbm</td>\n      <td>simple_lightgbm</td>\n      <td>non_preserving_recall_gain_pp</td>\n      <td>38.163906</td>\n      <td>36.556944</td>\n      <td>39.890736</td>\n      <td>15851</td>\n      <td>4472</td>\n      <td>2000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>20</td>\n      <td>simple_plus_maia_lightgbm</td>\n      <td>simple_lightgbm</td>\n      <td>brier_improvement</td>\n      <td>0.064679</td>\n      <td>0.061271</td>\n      <td>0.067994</td>\n      <td>15851</td>\n      <td>4472</td>\n      <td>2000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>20</td>\n      <td>simple_plus_maia_lightgbm</td>\n      <td>simple_lightgbm</td>\n      <td>log_loss_improvement</td>\n      <td>0.145341</td>\n      <td>0.136445</td>\n      <td>0.153428</td>\n      <td>15851</td>\n      <td>4472</td>\n      <td>2000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>20</td>\n      <td>simple_plus_maia_lightgbm</td>\n      <td>simple_lightgbm</td>\n      <td>accuracy_gain_pp</td>\n      <td>16.478456</td>\n      <td>15.532895</td>\n      <td>17.451652</td>\n      <td>15851</td>\n      <td>4472</td>\n      <td>2000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>20</td>\n      <td>simple_plus_maia_lightgbm</td>\n      <td>simple_lightgbm</td>\n      <td>balanced_accuracy_gain_pp</td>\n      <td>18.674743</td>\n      <td>17.778098</td>\n      <td>19.696048</td>\n      <td>15851</td>\n      <td>4472</td>\n      <td>2000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>20</td>\n      <td>simple_plus_maia_lightgbm</td>\n      <td>simple_lightgbm</td>\n      <td>preserving_recall_gain_pp</td>\n      <td>-0.214762</td>\n      <td>-1.376754</td>\n      <td>0.926239</td>\n      <td>15851</td>\n      <td>4472</td>\n      <td>2000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>20</td>\n      <td>simple_plus_maia_lightgbm</td>\n      <td>simple_lightgbm</td>\n      <td>non_preserving_recall_gain_pp</td>\n      <td>37.564249</td>\n      <td>35.980348</td>\n      <td>39.199687</td>\n      <td>15851</td>\n      <td>4472</td>\n      <td>2000</td>\n    </tr>\n  </tbody>\n</table>',
'Positive values favor adding Maia; intervals use paired game resampling.')


**Immediate reply core comparison**

Positive class: preserving. ROC-AUC uses continuous probabilities.

archive_set,protocol,method,games,n,accuracy,balanced_accuracy,precision,preserving_recall,non_preserving_recall,tn,fp,fn,tp,roc_auc,brier
"2017-04, 2017-06, 2017-07, 2018-02",game,raw_maia,4472,15851,0.730553,0.721247,0.738284,0.801289,0.641205,4491,2513,1758,7089,0.799701,0.184310
"2017-04, 2017-06, 2017-07, 2018-02",game,simple_lightgbm,4472,15851,0.560280,0.527813,0.575667,0.807053,0.248572,1741,5263,1707,7140,0.556793,0.244979
"2017-04, 2017-06, 2017-07, 2018-02",game,simple_plus_maia_lightgbm,4472,15851,0.726768,0.716711,0.732880,0.803210,0.630211,4414,2590,1741,7106,0.798525,0.180147
"2017-04, 2017-06, 2017-07, 2018-02",game,training_majority_prevalence,4472,15851,0.558135,0.500000,0.558135,1.000000,0.000000,0,7004,0,8847,0.492869,0.246679
"2017-04, 2017-06, 2017-07, 2018-02",player,raw_maia,4472,15851,0.730553,0.721247,0.738284,0.801289,0.641205,4491,2513,1758,7089,0.799701,0.184310
"2017-04, 2017-06, 2017-07, 2018-02",player,simple_lightgbm,4472,15851,0.559523,0.526049,0.574380,0.813948,0.238150,1668,5336,1646,7201,0.552592,0.245956
"2017-04, 2017-06, 2017-07, 2018-02",player,simple_plus_maia_lightgbm,4472,15851,0.724308,0.712796,0.726408,0.811801,0.613792,4299,2705,1665,7182,0.796549,0.181277
"2017-04, 2017-06, 2017-07, 2018-02",player,training_majority_prevalence,4472,15851,0.558135,0.500000,0.558135,1.000000,0.000000,0,7004,0,8847,0.494322,0.246685


**Immediate reply incremental Maia**

Positive values favor adding Maia; intervals use paired game resampling.

archive_set,protocol,initial_damage_threshold,augmented,baseline,metric,estimate,lower95,upper95,n,games,bootstrap_repeats
"2017-04, 2017-06, 2017-07, 2018-02",game,20,simple_plus_maia_lightgbm,simple_lightgbm,brier_improvement,0.064832,0.061650,0.067999,15851,4472,2000
"2017-04, 2017-06, 2017-07, 2018-02",game,20,simple_plus_maia_lightgbm,simple_lightgbm,log_loss_improvement,0.146175,0.137675,0.154050,15851,4472,2000
"2017-04, 2017-06, 2017-07, 2018-02",game,20,simple_plus_maia_lightgbm,simple_lightgbm,accuracy_gain_pp,16.648792,15.714662,17.627615,15851,4472,2000
"2017-04, 2017-06, 2017-07, 2018-02",game,20,simple_plus_maia_lightgbm,simple_lightgbm,balanced_accuracy_gain_pp,18.889798,17.962601,19.871046,15851,4472,2000
"2017-04, 2017-06, 2017-07, 2018-02",game,20,simple_plus_maia_lightgbm,simple_lightgbm,preserving_recall_gain_pp,-0.384311,-1.506613,0.822033,15851,4472,2000
"2017-04, 2017-06, 2017-07, 2018-02",game,20,simple_plus_maia_lightgbm,simple_lightgbm,non_preserving_recall_gain_pp,38.163906,36.556944,39.890736,15851,4472,2000
"2017-04, 2017-06, 2017-07, 2018-02",player,20,simple_plus_maia_lightgbm,simple_lightgbm,brier_improvement,0.064679,0.061271,0.067994,15851,4472,2000
"2017-04, 2017-06, 2017-07, 2018-02",player,20,simple_plus_maia_lightgbm,simple_lightgbm,log_loss_improvement,0.145341,0.136445,0.153428,15851,4472,2000
"2017-04, 2017-06, 2017-07, 2018-02",player,20,simple_plus_maia_lightgbm,simple_lightgbm,accuracy_gain_pp,16.478456,15.532895,17.451652,15851,4472,2000
"2017-04, 2017-06, 2017-07, 2018-02",player,20,simple_plus_maia_lightgbm,simple_lightgbm,balanced_accuracy_gain_pp,18.674743,17.778098,19.696048,15851,4472,2000


### Subgroup limitations

UltraBullet contains 274 replies from 67 games, including 122 preserving and 152 non-preserving replies. Its class recalls are descriptive estimates from a small, clustered, out-of-domain sample. They do not establish a stable population difference or a causal effect of time control. Maia does not use live clocks, and no subgroup-specific cutoff was fitted.


In [4]:
show('immediate_reply_speed_recalls',
'<table border="1" class="dataframe">\n  <thead>\n    <tr style="text-align: right;">\n      <th>archive_set</th>\n      <th>protocol</th>\n      <th>subgroup</th>\n      <th>reply_count</th>\n      <th>game_count</th>\n      <th>preserving_count</th>\n      <th>non_preserving_count</th>\n      <th>preserving_recall_percent</th>\n      <th>non_preserving_recall_percent</th>\n      <th>balanced_accuracy_percent</th>\n      <th>tn</th>\n      <th>fp</th>\n      <th>fn</th>\n      <th>tp</th>\n    </tr>\n  </thead>\n  <tbody>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>Fixed external policy; local game/player scores identical</td>\n      <td>UltraBullet</td>\n      <td>274</td>\n      <td>67</td>\n      <td>122</td>\n      <td>152</td>\n      <td>91.803279</td>\n      <td>30.263158</td>\n      <td>61.033218</td>\n      <td>46</td>\n      <td>106</td>\n      <td>10</td>\n      <td>112</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>Fixed external policy; local game/player scores identical</td>\n      <td>Bullet</td>\n      <td>4807</td>\n      <td>1331</td>\n      <td>2589</td>\n      <td>2218</td>\n      <td>83.043646</td>\n      <td>57.348963</td>\n      <td>70.196305</td>\n      <td>1272</td>\n      <td>946</td>\n      <td>439</td>\n      <td>2150</td>\n    </tr>\n  </tbody>\n</table>',
'Columns explicitly marked percent use 0–100 units.')

show('immediate_reply_subgroups',
'<table border="1" class="dataframe">\n  <thead>\n    <tr style="text-align: right;">\n      <th>archive_set</th>\n      <th>protocol</th>\n      <th>subgroup_type</th>\n      <th>subgroup</th>\n      <th>reply_count</th>\n      <th>game_count</th>\n      <th>share_of_all_replies_percent</th>\n      <th>preserving_count</th>\n      <th>non_preserving_count</th>\n      <th>preserving_share_percent</th>\n      <th>balanced_accuracy_percent</th>\n      <th>preserving_recall_percent</th>\n      <th>non_preserving_recall_percent</th>\n      <th>accuracy_percent</th>\n      <th>tn</th>\n      <th>fp</th>\n      <th>fn</th>\n      <th>tp</th>\n      <th>balanced_accuracy_defined</th>\n      <th>small_group_under_100_replies</th>\n    </tr>\n  </thead>\n  <tbody>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>Fixed external policy; descriptive subgroup summaries</td>\n      <td>reply_maker_elo</td>\n      <td>1600–&lt;2000</td>\n      <td>6859</td>\n      <td>2205</td>\n      <td>43.271718</td>\n      <td>3918</td>\n      <td>2941</td>\n      <td>57.122029</td>\n      <td>72.441103</td>\n      <td>79.428280</td>\n      <td>65.453927</td>\n      <td>73.436361</td>\n      <td>1925</td>\n      <td>1016</td>\n      <td>806</td>\n      <td>3112</td>\n      <td>True</td>\n      <td>False</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>Fixed external policy; descriptive subgroup summaries</td>\n      <td>reply_maker_elo</td>\n      <td>&lt;1600</td>\n      <td>7272</td>\n      <td>2172</td>\n      <td>45.877232</td>\n      <td>3923</td>\n      <td>3349</td>\n      <td>53.946645</td>\n      <td>71.899963</td>\n      <td>81.034922</td>\n      <td>62.765004</td>\n      <td>72.621012</td>\n      <td>2102</td>\n      <td>1247</td>\n      <td>744</td>\n      <td>3179</td>\n      <td>True</td>\n      <td>False</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>Fixed external policy; descriptive subgroup summaries</td>\n      <td>reply_maker_elo</td>\n      <td>&gt;=2000</td>\n      <td>1720</td>\n      <td>587</td>\n      <td>10.851050</td>\n      <td>1006</td>\n      <td>714</td>\n      <td>58.488372</td>\n      <td>72.155025</td>\n      <td>79.324056</td>\n      <td>64.985994</td>\n      <td>73.372093</td>\n      <td>464</td>\n      <td>250</td>\n      <td>208</td>\n      <td>798</td>\n      <td>True</td>\n      <td>False</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>Fixed external policy; descriptive subgroup summaries</td>\n      <td>archive</td>\n      <td>2017-04</td>\n      <td>5217</td>\n      <td>1454</td>\n      <td>32.912750</td>\n      <td>2929</td>\n      <td>2288</td>\n      <td>56.143377</td>\n      <td>73.421604</td>\n      <td>80.846705</td>\n      <td>65.996503</td>\n      <td>74.333908</td>\n      <td>1510</td>\n      <td>778</td>\n      <td>561</td>\n      <td>2368</td>\n      <td>True</td>\n      <td>False</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>Fixed external policy; descriptive subgroup summaries</td>\n      <td>archive</td>\n      <td>2017-06</td>\n      <td>3259</td>\n      <td>935</td>\n      <td>20.560217</td>\n      <td>1790</td>\n      <td>1469</td>\n      <td>54.924824</td>\n      <td>72.242433</td>\n      <td>80.223464</td>\n      <td>64.261402</td>\n      <td>73.028536</td>\n      <td>944</td>\n      <td>525</td>\n      <td>354</td>\n      <td>1436</td>\n      <td>True</td>\n      <td>False</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>Fixed external policy; descriptive subgroup summaries</td>\n      <td>archive</td>\n      <td>2017-07</td>\n      <td>3484</td>\n      <td>1006</td>\n      <td>21.979686</td>\n      <td>1968</td>\n      <td>1516</td>\n      <td>56.486797</td>\n      <td>71.152021</td>\n      <td>79.573171</td>\n      <td>62.730871</td>\n      <td>72.244546</td>\n      <td>951</td>\n      <td>565</td>\n      <td>402</td>\n      <td>1566</td>\n      <td>True</td>\n      <td>False</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>Fixed external policy; descriptive subgroup summaries</td>\n      <td>archive</td>\n      <td>2018-02</td>\n      <td>3891</td>\n      <td>1077</td>\n      <td>24.547347</td>\n      <td>2160</td>\n      <td>1731</td>\n      <td>55.512722</td>\n      <td>71.160817</td>\n      <td>79.583333</td>\n      <td>62.738302</td>\n      <td>72.089437</td>\n      <td>1086</td>\n      <td>645</td>\n      <td>441</td>\n      <td>1719</td>\n      <td>True</td>\n      <td>False</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>Fixed external policy; descriptive subgroup summaries</td>\n      <td>archive_speed_label</td>\n      <td>Blitz</td>\n      <td>7224</td>\n      <td>2095</td>\n      <td>45.574412</td>\n      <td>4108</td>\n      <td>3116</td>\n      <td>56.866002</td>\n      <td>73.780800</td>\n      <td>78.627069</td>\n      <td>68.934531</td>\n      <td>74.446290</td>\n      <td>2148</td>\n      <td>968</td>\n      <td>878</td>\n      <td>3230</td>\n      <td>True</td>\n      <td>False</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>Fixed external policy; descriptive subgroup summaries</td>\n      <td>archive_speed_label</td>\n      <td>Bullet</td>\n      <td>4807</td>\n      <td>1331</td>\n      <td>30.326162</td>\n      <td>2589</td>\n      <td>2218</td>\n      <td>53.858956</td>\n      <td>70.196305</td>\n      <td>83.043646</td>\n      <td>57.348963</td>\n      <td>71.187851</td>\n      <td>1272</td>\n      <td>946</td>\n      <td>439</td>\n      <td>2150</td>\n      <td>True</td>\n      <td>False</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>Fixed external policy; descriptive subgroup summaries</td>\n      <td>archive_speed_label</td>\n      <td>Classical</td>\n      <td>2887</td>\n      <td>803</td>\n      <td>18.213362</td>\n      <td>1666</td>\n      <td>1221</td>\n      <td>57.706962</td>\n      <td>73.478998</td>\n      <td>78.571429</td>\n      <td>68.386568</td>\n      <td>74.263942</td>\n      <td>835</td>\n      <td>386</td>\n      <td>357</td>\n      <td>1309</td>\n      <td>True</td>\n      <td>False</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>Fixed external policy; descriptive subgroup summaries</td>\n      <td>archive_speed_label</td>\n      <td>Correspondence</td>\n      <td>51</td>\n      <td>14</td>\n      <td>0.321746</td>\n      <td>33</td>\n      <td>18</td>\n      <td>64.705882</td>\n      <td>77.272727</td>\n      <td>87.878788</td>\n      <td>66.666667</td>\n      <td>80.392157</td>\n      <td>12</td>\n      <td>6</td>\n      <td>4</td>\n      <td>29</td>\n      <td>True</td>\n      <td>True</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>Fixed external policy; descriptive subgroup summaries</td>\n      <td>archive_speed_label</td>\n      <td>Standard/unspecified</td>\n      <td>608</td>\n      <td>162</td>\n      <td>3.835720</td>\n      <td>329</td>\n      <td>279</td>\n      <td>54.111842</td>\n      <td>71.261344</td>\n      <td>78.723404</td>\n      <td>63.799283</td>\n      <td>71.875000</td>\n      <td>178</td>\n      <td>101</td>\n      <td>70</td>\n      <td>259</td>\n      <td>True</td>\n      <td>False</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>Fixed external policy; descriptive subgroup summaries</td>\n      <td>archive_speed_label</td>\n      <td>UltraBullet</td>\n      <td>274</td>\n      <td>67</td>\n      <td>1.728598</td>\n      <td>122</td>\n      <td>152</td>\n      <td>44.525547</td>\n      <td>61.033218</td>\n      <td>91.803279</td>\n      <td>30.263158</td>\n      <td>57.664234</td>\n      <td>46</td>\n      <td>106</td>\n      <td>10</td>\n      <td>112</td>\n      <td>True</td>\n      <td>False</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>Fixed external policy; descriptive subgroup summaries</td>\n      <td>game_phase</td>\n      <td>endgame</td>\n      <td>1776</td>\n      <td>657</td>\n      <td>11.204340</td>\n      <td>1021</td>\n      <td>755</td>\n      <td>57.488739</td>\n      <td>71.291942</td>\n      <td>83.643487</td>\n      <td>58.940397</td>\n      <td>73.141892</td>\n      <td>445</td>\n      <td>310</td>\n      <td>167</td>\n      <td>854</td>\n      <td>True</td>\n      <td>False</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>Fixed external policy; descriptive subgroup summaries</td>\n      <td>game_phase</td>\n      <td>middlegame</td>\n      <td>3468</td>\n      <td>1508</td>\n      <td>21.878746</td>\n      <td>2034</td>\n      <td>1434</td>\n      <td>58.650519</td>\n      <td>70.952901</td>\n      <td>80.678466</td>\n      <td>61.227336</td>\n      <td>72.635525</td>\n      <td>878</td>\n      <td>556</td>\n      <td>393</td>\n      <td>1641</td>\n      <td>True</td>\n      <td>False</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>Fixed external policy; descriptive subgroup summaries</td>\n      <td>game_phase</td>\n      <td>opening</td>\n      <td>10607</td>\n      <td>3927</td>\n      <td>66.916914</td>\n      <td>5792</td>\n      <td>4815</td>\n      <td>54.605449</td>\n      <td>72.555345</td>\n      <td>79.316298</td>\n      <td>65.794393</td>\n      <td>73.178090</td>\n      <td>3168</td>\n      <td>1647</td>\n      <td>1198</td>\n      <td>4594</td>\n      <td>True</td>\n      <td>False</td>\n    </tr>\n  </tbody>\n</table>',
'')


**Immediate reply speed recalls**

Columns explicitly marked percent use 0–100 units.

archive_set,protocol,subgroup,reply_count,game_count,preserving_count,non_preserving_count,preserving_recall_percent,non_preserving_recall_percent,balanced_accuracy_percent,tn,fp,fn,tp
"2017-04, 2017-06, 2017-07, 2018-02",Fixed external policy; local game/player scores identical,UltraBullet,274,67,122,152,91.803279,30.263158,61.033218,46,106,10,112
"2017-04, 2017-06, 2017-07, 2018-02",Fixed external policy; local game/player scores identical,Bullet,4807,1331,2589,2218,83.043646,57.348963,70.196305,1272,946,439,2150


**Immediate reply subgroups**

archive_set,protocol,subgroup_type,subgroup,reply_count,game_count,share_of_all_replies_percent,preserving_count,non_preserving_count,preserving_share_percent,balanced_accuracy_percent,preserving_recall_percent,non_preserving_recall_percent,accuracy_percent,tn,fp,fn,tp,balanced_accuracy_defined,small_group_under_100_replies
"2017-04, 2017-06, 2017-07, 2018-02",Fixed external policy; descriptive subgroup summaries,reply_maker_elo,1600–<2000,6859,2205,43.271718,3918,2941,57.122029,72.441103,79.428280,65.453927,73.436361,1925,1016,806,3112,True,False
"2017-04, 2017-06, 2017-07, 2018-02",Fixed external policy; descriptive subgroup summaries,reply_maker_elo,<1600,7272,2172,45.877232,3923,3349,53.946645,71.899963,81.034922,62.765004,72.621012,2102,1247,744,3179,True,False
"2017-04, 2017-06, 2017-07, 2018-02",Fixed external policy; descriptive subgroup summaries,reply_maker_elo,>=2000,1720,587,10.851050,1006,714,58.488372,72.155025,79.324056,64.985994,73.372093,464,250,208,798,True,False
"2017-04, 2017-06, 2017-07, 2018-02",Fixed external policy; descriptive subgroup summaries,archive,2017-04,5217,1454,32.912750,2929,2288,56.143377,73.421604,80.846705,65.996503,74.333908,1510,778,561,2368,True,False
"2017-04, 2017-06, 2017-07, 2018-02",Fixed external policy; descriptive subgroup summaries,archive,2017-06,3259,935,20.560217,1790,1469,54.924824,72.242433,80.223464,64.261402,73.028536,944,525,354,1436,True,False
"2017-04, 2017-06, 2017-07, 2018-02",Fixed external policy; descriptive subgroup summaries,archive,2017-07,3484,1006,21.979686,1968,1516,56.486797,71.152021,79.573171,62.730871,72.244546,951,565,402,1566,True,False
"2017-04, 2017-06, 2017-07, 2018-02",Fixed external policy; descriptive subgroup summaries,archive,2018-02,3891,1077,24.547347,2160,1731,55.512722,71.160817,79.583333,62.738302,72.089437,1086,645,441,1719,True,False
"2017-04, 2017-06, 2017-07, 2018-02",Fixed external policy; descriptive subgroup summaries,archive_speed_label,Blitz,7224,2095,45.574412,4108,3116,56.866002,73.780800,78.627069,68.934531,74.446290,2148,968,878,3230,True,False
"2017-04, 2017-06, 2017-07, 2018-02",Fixed external policy; descriptive subgroup summaries,archive_speed_label,Bullet,4807,1331,30.326162,2589,2218,53.858956,70.196305,83.043646,57.348963,71.187851,1272,946,439,2150,True,False
"2017-04, 2017-06, 2017-07, 2018-02",Fixed external policy; descriptive subgroup summaries,archive_speed_label,Classical,2887,803,18.213362,1666,1221,57.706962,73.478998,78.571429,68.386568,74.263942,835,386,357,1309,True,False


## 3. Longer-horizon persistence

The fixed five-archive LightGBM reduced ten-ply MAE by 0.586 winning-chance pp (2.48%) over no-change, with Spearman 0.123 on game folds. At two plies the MAE reduction was 0.018 pp. The gains are modest even where paired intervals exclude zero.

All models below share the same eligible held-out rows within each protocol. The fixed reference uses 24 `full_multipv` predictors. No-change predicts zero; mean and median baselines use training-fold targets. Tree, forest and engine-control models retain their specified settings. Optuna LightGBM and XGBoost use 12 trials per outer fold and horizon, selected by inner-fold MAE; `model_selection` distinguishes them from fixed models. Historical `selected.json` configurations are reported separately in Section 5.

MAE and Spearman were checked against saved predictions. Spearman is undefined for constant and training-constant baselines. The 95% percentile intervals use 2,000 paired whole-game bootstrap resamples, seed 1701, conditional on fitted OOF predictions. They exclude full training/selection uncertainty and are unadjusted for multiple comparisons.


In [5]:
show('primary_regression',
'<table border="1" class="dataframe">\n  <thead>\n    <tr style="text-align: right;">\n      <th>archive_set</th>\n      <th>protocol</th>\n      <th>target</th>\n      <th>model</th>\n      <th>model_selection</th>\n      <th>n</th>\n      <th>games</th>\n      <th>mae</th>\n      <th>spearman</th>\n      <th>estimate_mae_improvement_pp</th>\n      <th>lower95_mae_improvement_pp</th>\n      <th>upper95_mae_improvement_pp</th>\n      <th>estimate_mae_improvement_percent</th>\n      <th>lower95_mae_improvement_percent</th>\n      <th>upper95_mae_improvement_percent</th>\n    </tr>\n  </thead>\n  <tbody>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>2</td>\n      <td>decision_tree</td>\n      <td>Fixed settings</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>14.586629</td>\n      <td>0.108037</td>\n      <td>-2.010071</td>\n      <td>-2.105561</td>\n      <td>-1.911686</td>\n      <td>-15.982677</td>\n      <td>-16.955489</td>\n      <td>-15.040742</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>2</td>\n      <td>engine_only</td>\n      <td>Fixed settings</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>12.583891</td>\n      <td>0.011851</td>\n      <td>-0.007333</td>\n      <td>-0.010593</td>\n      <td>-0.003754</td>\n      <td>-0.058305</td>\n      <td>-0.084208</td>\n      <td>-0.029775</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>2</td>\n      <td>engine_ratings</td>\n      <td>Fixed settings</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>12.587247</td>\n      <td>0.011108</td>\n      <td>-0.010689</td>\n      <td>-0.014384</td>\n      <td>-0.007150</td>\n      <td>-0.084991</td>\n      <td>-0.114226</td>\n      <td>-0.056612</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>2</td>\n      <td>fixed_lightgbm</td>\n      <td>Fixed settings</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>12.558943</td>\n      <td>0.116483</td>\n      <td>0.017615</td>\n      <td>0.011389</td>\n      <td>0.023895</td>\n      <td>0.140066</td>\n      <td>0.090622</td>\n      <td>0.189788</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>2</td>\n      <td>no_change</td>\n      <td>Prespecified baseline</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>12.576558</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>2</td>\n      <td>optuna_lightgbm</td>\n      <td>Inner-fold Optuna</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>12.554643</td>\n      <td>0.117968</td>\n      <td>0.021915</td>\n      <td>0.004240</td>\n      <td>0.039136</td>\n      <td>0.174255</td>\n      <td>0.033772</td>\n      <td>0.311095</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>2</td>\n      <td>optuna_xgboost</td>\n      <td>Inner-fold Optuna</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>12.544558</td>\n      <td>0.116243</td>\n      <td>0.032000</td>\n      <td>0.018283</td>\n      <td>0.046740</td>\n      <td>0.254443</td>\n      <td>0.146196</td>\n      <td>0.369665</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>2</td>\n      <td>random_forest</td>\n      <td>Fixed settings</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>14.520221</td>\n      <td>0.129608</td>\n      <td>-1.943663</td>\n      <td>-2.041528</td>\n      <td>-1.842074</td>\n      <td>-15.454649</td>\n      <td>-16.435487</td>\n      <td>-14.510469</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>2</td>\n      <td>training_mean</td>\n      <td>Prespecified baseline</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>14.650068</td>\n      <td>unavailable / undefined</td>\n      <td>-2.073509</td>\n      <td>-2.136868</td>\n      <td>-2.007414</td>\n      <td>-16.487095</td>\n      <td>-17.203540</td>\n      <td>-15.756583</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>2</td>\n      <td>training_median</td>\n      <td>Prespecified baseline</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>12.576602</td>\n      <td>unavailable / undefined</td>\n      <td>-0.000043</td>\n      <td>-0.000232</td>\n      <td>0.000145</td>\n      <td>-0.000345</td>\n      <td>-0.001836</td>\n      <td>0.001154</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>4</td>\n      <td>decision_tree</td>\n      <td>Fixed settings</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>18.939973</td>\n      <td>0.119005</td>\n      <td>-2.004108</td>\n      <td>-2.128867</td>\n      <td>-1.878648</td>\n      <td>-11.833517</td>\n      <td>-12.701231</td>\n      <td>-11.002933</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>4</td>\n      <td>engine_only</td>\n      <td>Fixed settings</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>16.953035</td>\n      <td>0.008783</td>\n      <td>-0.017171</td>\n      <td>-0.029863</td>\n      <td>-0.004868</td>\n      <td>-0.101386</td>\n      <td>-0.176295</td>\n      <td>-0.028724</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>4</td>\n      <td>engine_ratings</td>\n      <td>Fixed settings</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>16.953861</td>\n      <td>0.032067</td>\n      <td>-0.017996</td>\n      <td>-0.031367</td>\n      <td>-0.004996</td>\n      <td>-0.106261</td>\n      <td>-0.185237</td>\n      <td>-0.029228</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>4</td>\n      <td>fixed_lightgbm</td>\n      <td>Fixed settings</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>16.843501</td>\n      <td>0.104874</td>\n      <td>0.092363</td>\n      <td>0.040911</td>\n      <td>0.146109</td>\n      <td>0.545371</td>\n      <td>0.241712</td>\n      <td>0.862825</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>4</td>\n      <td>no_change</td>\n      <td>Prespecified baseline</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>16.935864</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>4</td>\n      <td>optuna_lightgbm</td>\n      <td>Inner-fold Optuna</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>16.811599</td>\n      <td>0.098739</td>\n      <td>0.124265</td>\n      <td>0.060563</td>\n      <td>0.187495</td>\n      <td>0.733739</td>\n      <td>0.359209</td>\n      <td>1.103989</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>4</td>\n      <td>optuna_xgboost</td>\n      <td>Inner-fold Optuna</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>16.820052</td>\n      <td>0.098783</td>\n      <td>0.115812</td>\n      <td>0.058706</td>\n      <td>0.171249</td>\n      <td>0.683827</td>\n      <td>0.343927</td>\n      <td>1.008342</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>4</td>\n      <td>random_forest</td>\n      <td>Fixed settings</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>18.760332</td>\n      <td>0.142511</td>\n      <td>-1.824467</td>\n      <td>-1.949886</td>\n      <td>-1.697673</td>\n      <td>-10.772803</td>\n      <td>-11.648864</td>\n      <td>-9.936768</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>4</td>\n      <td>training_mean</td>\n      <td>Prespecified baseline</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>18.868808</td>\n      <td>unavailable / undefined</td>\n      <td>-1.932944</td>\n      <td>-2.021222</td>\n      <td>-1.845488</td>\n      <td>-11.413316</td>\n      <td>-12.060805</td>\n      <td>-10.785601</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>4</td>\n      <td>training_median</td>\n      <td>Prespecified baseline</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>16.936451</td>\n      <td>unavailable / undefined</td>\n      <td>-0.000587</td>\n      <td>-0.001155</td>\n      <td>-0.000013</td>\n      <td>-0.003463</td>\n      <td>-0.006795</td>\n      <td>-0.000077</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>decision_tree</td>\n      <td>Fixed settings</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>21.725772</td>\n      <td>0.122735</td>\n      <td>-1.891280</td>\n      <td>-2.045362</td>\n      <td>-1.748592</td>\n      <td>-9.535308</td>\n      <td>-10.406760</td>\n      <td>-8.741724</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>engine_only</td>\n      <td>Fixed settings</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>19.858545</td>\n      <td>-0.010585</td>\n      <td>-0.024053</td>\n      <td>-0.043668</td>\n      <td>-0.004234</td>\n      <td>-0.121270</td>\n      <td>-0.221475</td>\n      <td>-0.021118</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>engine_ratings</td>\n      <td>Fixed settings</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>19.838905</td>\n      <td>0.035348</td>\n      <td>-0.004414</td>\n      <td>-0.027209</td>\n      <td>0.017622</td>\n      <td>-0.022252</td>\n      <td>-0.137138</td>\n      <td>0.087814</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>fixed_lightgbm</td>\n      <td>Fixed settings</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>19.579173</td>\n      <td>0.099988</td>\n      <td>0.255319</td>\n      <td>0.172802</td>\n      <td>0.338477</td>\n      <td>1.287246</td>\n      <td>0.869333</td>\n      <td>1.698471</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>no_change</td>\n      <td>Prespecified baseline</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>19.834492</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>optuna_lightgbm</td>\n      <td>Inner-fold Optuna</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>19.565276</td>\n      <td>0.100014</td>\n      <td>0.269216</td>\n      <td>0.184484</td>\n      <td>0.349004</td>\n      <td>1.357313</td>\n      <td>0.933535</td>\n      <td>1.754331</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>optuna_xgboost</td>\n      <td>Inner-fold Optuna</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>19.546584</td>\n      <td>0.101415</td>\n      <td>0.287908</td>\n      <td>0.209031</td>\n      <td>0.364718</td>\n      <td>1.451551</td>\n      <td>1.059301</td>\n      <td>1.830258</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>random_forest</td>\n      <td>Fixed settings</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>21.626793</td>\n      <td>0.154500</td>\n      <td>-1.792301</td>\n      <td>-1.950726</td>\n      <td>-1.642144</td>\n      <td>-9.036283</td>\n      <td>-9.924903</td>\n      <td>-8.215530</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>training_mean</td>\n      <td>Prespecified baseline</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>21.768507</td>\n      <td>unavailable / undefined</td>\n      <td>-1.934015</td>\n      <td>-2.034342</td>\n      <td>-1.838665</td>\n      <td>-9.750766</td>\n      <td>-10.358015</td>\n      <td>-9.193557</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>training_median</td>\n      <td>Prespecified baseline</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>19.834672</td>\n      <td>unavailable / undefined</td>\n      <td>-0.000180</td>\n      <td>-0.000703</td>\n      <td>0.000325</td>\n      <td>-0.000908</td>\n      <td>-0.003540</td>\n      <td>0.001640</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>10</td>\n      <td>decision_tree</td>\n      <td>Fixed settings</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>25.438442</td>\n      <td>0.144455</td>\n      <td>-1.803031</td>\n      <td>-1.992981</td>\n      <td>-1.618005</td>\n      <td>-7.628514</td>\n      <td>-8.523277</td>\n      <td>-6.785121</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>10</td>\n      <td>engine_only</td>\n      <td>Fixed settings</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>23.664575</td>\n      <td>-0.037379</td>\n      <td>-0.029164</td>\n      <td>-0.057396</td>\n      <td>-0.000426</td>\n      <td>-0.123391</td>\n      <td>-0.243032</td>\n      <td>-0.001802</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>10</td>\n      <td>engine_ratings</td>\n      <td>Fixed settings</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>23.593498</td>\n      <td>0.045582</td>\n      <td>0.041913</td>\n      <td>-0.007140</td>\n      <td>0.089363</td>\n      <td>0.177331</td>\n      <td>-0.030365</td>\n      <td>0.379967</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>10</td>\n      <td>fixed_lightgbm</td>\n      <td>Fixed settings</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>23.049053</td>\n      <td>0.123120</td>\n      <td>0.586358</td>\n      <td>0.462247</td>\n      <td>0.707210</td>\n      <td>2.480846</td>\n      <td>1.960367</td>\n      <td>2.970119</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>10</td>\n      <td>no_change</td>\n      <td>Prespecified baseline</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>23.635411</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>10</td>\n      <td>optuna_lightgbm</td>\n      <td>Inner-fold Optuna</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>23.060939</td>\n      <td>0.120210</td>\n      <td>0.574472</td>\n      <td>0.461893</td>\n      <td>0.685609</td>\n      <td>2.430555</td>\n      <td>1.951446</td>\n      <td>2.888818</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>10</td>\n      <td>optuna_xgboost</td>\n      <td>Inner-fold Optuna</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>23.047485</td>\n      <td>0.119099</td>\n      <td>0.587926</td>\n      <td>0.473892</td>\n      <td>0.696195</td>\n      <td>2.487479</td>\n      <td>2.009239</td>\n      <td>2.934362</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>10</td>\n      <td>random_forest</td>\n      <td>Fixed settings</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>25.312098</td>\n      <td>0.182780</td>\n      <td>-1.676687</td>\n      <td>-1.870842</td>\n      <td>-1.488656</td>\n      <td>-7.093962</td>\n      <td>-7.982035</td>\n      <td>-6.240520</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>10</td>\n      <td>training_mean</td>\n      <td>Prespecified baseline</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>25.600246</td>\n      <td>unavailable / undefined</td>\n      <td>-1.964835</td>\n      <td>-2.082417</td>\n      <td>-1.844914</td>\n      <td>-8.313097</td>\n      <td>-8.883831</td>\n      <td>-7.751246</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>10</td>\n      <td>training_median</td>\n      <td>Prespecified baseline</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>23.634855</td>\n      <td>unavailable / undefined</td>\n      <td>0.000556</td>\n      <td>-0.001288</td>\n      <td>0.002458</td>\n      <td>0.002351</td>\n      <td>-0.005420</td>\n      <td>0.010479</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>2</td>\n      <td>decision_tree</td>\n      <td>Fixed settings</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>14.580921</td>\n      <td>0.102168</td>\n      <td>-2.004362</td>\n      <td>-2.098916</td>\n      <td>-1.907139</td>\n      <td>-15.937288</td>\n      <td>-16.889137</td>\n      <td>-15.015262</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>2</td>\n      <td>engine_only</td>\n      <td>Fixed settings</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>12.585793</td>\n      <td>0.007524</td>\n      <td>-0.009234</td>\n      <td>-0.012918</td>\n      <td>-0.005260</td>\n      <td>-0.073426</td>\n      <td>-0.103078</td>\n      <td>-0.042015</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>2</td>\n      <td>engine_ratings</td>\n      <td>Fixed settings</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>12.585963</td>\n      <td>0.009533</td>\n      <td>-0.009405</td>\n      <td>-0.013419</td>\n      <td>-0.005405</td>\n      <td>-0.074779</td>\n      <td>-0.106788</td>\n      <td>-0.043045</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>2</td>\n      <td>fixed_lightgbm</td>\n      <td>Fixed settings</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>12.559600</td>\n      <td>0.107507</td>\n      <td>0.016959</td>\n      <td>0.010862</td>\n      <td>0.022967</td>\n      <td>0.134843</td>\n      <td>0.086108</td>\n      <td>0.183473</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>2</td>\n      <td>no_change</td>\n      <td>Prespecified baseline</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>12.576558</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>2</td>\n      <td>optuna_lightgbm</td>\n      <td>Inner-fold Optuna</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>12.557639</td>\n      <td>0.110772</td>\n      <td>0.018919</td>\n      <td>0.014369</td>\n      <td>0.023428</td>\n      <td>0.150431</td>\n      <td>0.114981</td>\n      <td>0.186228</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>2</td>\n      <td>optuna_xgboost</td>\n      <td>Inner-fold Optuna</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>12.558491</td>\n      <td>0.112125</td>\n      <td>0.018067</td>\n      <td>0.006076</td>\n      <td>0.030062</td>\n      <td>0.143656</td>\n      <td>0.048501</td>\n      <td>0.237705</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>2</td>\n      <td>random_forest</td>\n      <td>Fixed settings</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>14.534985</td>\n      <td>0.127331</td>\n      <td>-1.958427</td>\n      <td>-2.055090</td>\n      <td>-1.860644</td>\n      <td>-15.572042</td>\n      <td>-16.560591</td>\n      <td>-14.636172</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>2</td>\n      <td>training_mean</td>\n      <td>Prespecified baseline</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>14.603817</td>\n      <td>unavailable / undefined</td>\n      <td>-2.027259</td>\n      <td>-2.089657</td>\n      <td>-1.962118</td>\n      <td>-16.119347</td>\n      <td>-16.822375</td>\n      <td>-15.401572</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>2</td>\n      <td>training_median</td>\n      <td>Prespecified baseline</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>12.576599</td>\n      <td>unavailable / undefined</td>\n      <td>-0.000041</td>\n      <td>-0.000340</td>\n      <td>0.000228</td>\n      <td>-0.000325</td>\n      <td>-0.002688</td>\n      <td>0.001815</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>4</td>\n      <td>decision_tree</td>\n      <td>Fixed settings</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>18.934775</td>\n      <td>0.119664</td>\n      <td>-1.998910</td>\n      <td>-2.120723</td>\n      <td>-1.874152</td>\n      <td>-11.802823</td>\n      <td>-12.669442</td>\n      <td>-10.988477</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>4</td>\n      <td>engine_only</td>\n      <td>Fixed settings</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>16.958340</td>\n      <td>0.009330</td>\n      <td>-0.022476</td>\n      <td>-0.038076</td>\n      <td>-0.007645</td>\n      <td>-0.132711</td>\n      <td>-0.222645</td>\n      <td>-0.045139</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>4</td>\n      <td>engine_ratings</td>\n      <td>Fixed settings</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>16.965110</td>\n      <td>0.028914</td>\n      <td>-0.029245</td>\n      <td>-0.045114</td>\n      <td>-0.013780</td>\n      <td>-0.172681</td>\n      <td>-0.266163</td>\n      <td>-0.081246</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>4</td>\n      <td>fixed_lightgbm</td>\n      <td>Fixed settings</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>16.878519</td>\n      <td>0.093913</td>\n      <td>0.057345</td>\n      <td>0.013943</td>\n      <td>0.101248</td>\n      <td>0.338601</td>\n      <td>0.083215</td>\n      <td>0.596083</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>4</td>\n      <td>no_change</td>\n      <td>Prespecified baseline</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>16.935864</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>4</td>\n      <td>optuna_lightgbm</td>\n      <td>Inner-fold Optuna</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>16.880082</td>\n      <td>0.089291</td>\n      <td>0.055783</td>\n      <td>0.013433</td>\n      <td>0.099117</td>\n      <td>0.329375</td>\n      <td>0.078692</td>\n      <td>0.583538</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>4</td>\n      <td>optuna_xgboost</td>\n      <td>Inner-fold Optuna</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>16.822227</td>\n      <td>0.096662</td>\n      <td>0.113637</td>\n      <td>0.054803</td>\n      <td>0.175512</td>\n      <td>0.670987</td>\n      <td>0.325142</td>\n      <td>1.026969</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>4</td>\n      <td>random_forest</td>\n      <td>Fixed settings</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>18.826968</td>\n      <td>0.140754</td>\n      <td>-1.891104</td>\n      <td>-2.015503</td>\n      <td>-1.765072</td>\n      <td>-11.166266</td>\n      <td>-12.032127</td>\n      <td>-10.326600</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>4</td>\n      <td>training_mean</td>\n      <td>Prespecified baseline</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>18.834331</td>\n      <td>unavailable / undefined</td>\n      <td>-1.898466</td>\n      <td>-1.985887</td>\n      <td>-1.812036</td>\n      <td>-11.209739</td>\n      <td>-11.850256</td>\n      <td>-10.592672</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>4</td>\n      <td>training_median</td>\n      <td>Prespecified baseline</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>16.936563</td>\n      <td>unavailable / undefined</td>\n      <td>-0.000698</td>\n      <td>-0.001343</td>\n      <td>-0.000044</td>\n      <td>-0.004122</td>\n      <td>-0.007975</td>\n      <td>-0.000257</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>decision_tree</td>\n      <td>Fixed settings</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>21.754933</td>\n      <td>0.123731</td>\n      <td>-1.920441</td>\n      <td>-2.073142</td>\n      <td>-1.785306</td>\n      <td>-9.682332</td>\n      <td>-10.569842</td>\n      <td>-8.927119</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>engine_only</td>\n      <td>Fixed settings</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>19.862167</td>\n      <td>-0.011331</td>\n      <td>-0.027675</td>\n      <td>-0.049471</td>\n      <td>-0.006164</td>\n      <td>-0.139529</td>\n      <td>-0.250098</td>\n      <td>-0.031075</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>engine_ratings</td>\n      <td>Fixed settings</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>19.846700</td>\n      <td>0.035090</td>\n      <td>-0.012208</td>\n      <td>-0.035921</td>\n      <td>0.012497</td>\n      <td>-0.061548</td>\n      <td>-0.180215</td>\n      <td>0.063354</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>fixed_lightgbm</td>\n      <td>Fixed settings</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>19.617191</td>\n      <td>0.094367</td>\n      <td>0.217301</td>\n      <td>0.138916</td>\n      <td>0.293363</td>\n      <td>1.095572</td>\n      <td>0.702708</td>\n      <td>1.471529</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>no_change</td>\n      <td>Prespecified baseline</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>19.834492</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>optuna_lightgbm</td>\n      <td>Inner-fold Optuna</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>19.637148</td>\n      <td>0.087821</td>\n      <td>0.197344</td>\n      <td>0.115810</td>\n      <td>0.278306</td>\n      <td>0.994952</td>\n      <td>0.584809</td>\n      <td>1.395551</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>optuna_xgboost</td>\n      <td>Inner-fold Optuna</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>19.611260</td>\n      <td>0.088723</td>\n      <td>0.223232</td>\n      <td>0.149032</td>\n      <td>0.292199</td>\n      <td>1.125472</td>\n      <td>0.754222</td>\n      <td>1.468406</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>random_forest</td>\n      <td>Fixed settings</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>21.692143</td>\n      <td>0.149266</td>\n      <td>-1.857651</td>\n      <td>-2.013116</td>\n      <td>-1.711515</td>\n      <td>-9.365761</td>\n      <td>-10.246925</td>\n      <td>-8.566633</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>training_mean</td>\n      <td>Prespecified baseline</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>21.732476</td>\n      <td>unavailable / undefined</td>\n      <td>-1.897984</td>\n      <td>-1.996971</td>\n      <td>-1.804016</td>\n      <td>-9.569107</td>\n      <td>-10.167310</td>\n      <td>-9.021386</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>training_median</td>\n      <td>Prespecified baseline</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>19.834651</td>\n      <td>unavailable / undefined</td>\n      <td>-0.000159</td>\n      <td>-0.000780</td>\n      <td>0.000406</td>\n      <td>-0.000803</td>\n      <td>-0.003919</td>\n      <td>0.002053</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>10</td>\n      <td>decision_tree</td>\n      <td>Fixed settings</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>25.532606</td>\n      <td>0.154599</td>\n      <td>-1.897195</td>\n      <td>-2.092082</td>\n      <td>-1.716594</td>\n      <td>-8.026917</td>\n      <td>-8.928639</td>\n      <td>-7.193399</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>10</td>\n      <td>engine_only</td>\n      <td>Fixed settings</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>23.678162</td>\n      <td>-0.041141</td>\n      <td>-0.042750</td>\n      <td>-0.071398</td>\n      <td>-0.013961</td>\n      <td>-0.180874</td>\n      <td>-0.301055</td>\n      <td>-0.059059</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>10</td>\n      <td>engine_ratings</td>\n      <td>Fixed settings</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>23.611332</td>\n      <td>0.034767</td>\n      <td>0.024079</td>\n      <td>-0.026630</td>\n      <td>0.072711</td>\n      <td>0.101877</td>\n      <td>-0.112050</td>\n      <td>0.305500</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>10</td>\n      <td>fixed_lightgbm</td>\n      <td>Fixed settings</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>23.133595</td>\n      <td>0.111968</td>\n      <td>0.501816</td>\n      <td>0.383311</td>\n      <td>0.620378</td>\n      <td>2.123154</td>\n      <td>1.630685</td>\n      <td>2.610606</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>10</td>\n      <td>no_change</td>\n      <td>Prespecified baseline</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>23.635411</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>10</td>\n      <td>optuna_lightgbm</td>\n      <td>Inner-fold Optuna</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>23.112520</td>\n      <td>0.107460</td>\n      <td>0.522891</td>\n      <td>0.411050</td>\n      <td>0.636792</td>\n      <td>2.212322</td>\n      <td>1.740975</td>\n      <td>2.687532</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>10</td>\n      <td>optuna_xgboost</td>\n      <td>Inner-fold Optuna</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>23.115777</td>\n      <td>0.110181</td>\n      <td>0.519634</td>\n      <td>0.410886</td>\n      <td>0.625239</td>\n      <td>2.198539</td>\n      <td>1.742565</td>\n      <td>2.628760</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>10</td>\n      <td>random_forest</td>\n      <td>Fixed settings</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>25.403635</td>\n      <td>0.176490</td>\n      <td>-1.768223</td>\n      <td>-1.957273</td>\n      <td>-1.579465</td>\n      <td>-7.481247</td>\n      <td>-8.373122</td>\n      <td>-6.631918</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>10</td>\n      <td>training_mean</td>\n      <td>Prespecified baseline</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>25.559365</td>\n      <td>unavailable / undefined</td>\n      <td>-1.923954</td>\n      <td>-2.039752</td>\n      <td>-1.805820</td>\n      <td>-8.140132</td>\n      <td>-8.702703</td>\n      <td>-7.585848</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>10</td>\n      <td>training_median</td>\n      <td>Prespecified baseline</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>23.634709</td>\n      <td>unavailable / undefined</td>\n      <td>0.000702</td>\n      <td>-0.001447</td>\n      <td>0.002993</td>\n      <td>0.002972</td>\n      <td>-0.006108</td>\n      <td>0.012645</td>\n    </tr>\n  </tbody>\n</table>',
'Horizon is in plies. MAE gains and intervals are relative to no-change.')


**Primary regression**

Horizon is in plies. MAE gains and intervals are relative to no-change.

archive_set,protocol,target,model,model_selection,n,games,mae,spearman,estimate_mae_improvement_pp,lower95_mae_improvement_pp,upper95_mae_improvement_pp,estimate_mae_improvement_percent,lower95_mae_improvement_percent,upper95_mae_improvement_percent
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,2,decision_tree,Fixed settings,23167,6512,14.586629,0.108037,-2.010071,-2.105561,-1.911686,-15.982677,-16.955489,-15.040742
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,2,engine_only,Fixed settings,23167,6512,12.583891,0.011851,-0.007333,-0.010593,-0.003754,-0.058305,-0.084208,-0.029775
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,2,engine_ratings,Fixed settings,23167,6512,12.587247,0.011108,-0.010689,-0.014384,-0.007150,-0.084991,-0.114226,-0.056612
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,2,fixed_lightgbm,Fixed settings,23167,6512,12.558943,0.116483,0.017615,0.011389,0.023895,0.140066,0.090622,0.189788
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,2,no_change,Prespecified baseline,23167,6512,12.576558,unavailable / undefined,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,2,optuna_lightgbm,Inner-fold Optuna,23167,6512,12.554643,0.117968,0.021915,0.004240,0.039136,0.174255,0.033772,0.311095
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,2,optuna_xgboost,Inner-fold Optuna,23167,6512,12.544558,0.116243,0.032000,0.018283,0.046740,0.254443,0.146196,0.369665
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,2,random_forest,Fixed settings,23167,6512,14.520221,0.129608,-1.943663,-2.041528,-1.842074,-15.454649,-16.435487,-14.510469
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,2,training_mean,Prespecified baseline,23167,6512,14.650068,unavailable / undefined,-2.073509,-2.136868,-2.007414,-16.487095,-17.203540,-15.756583
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,2,training_median,Prespecified baseline,23167,6512,12.576602,unavailable / undefined,-0.000043,-0.000232,0.000145,-0.000345,-0.001836,0.001154


### Maia-2 and ten-ply change in damage

The matched comparison uses 15,953 mistakes from 4,485 games in four archives. Both regressors use identical folds, preprocessing and LightGBM settings. The augmented model adds the immediate-reply Maia probability to the original 24 predictors.

Adding Maia reduced game-fold MAE by 0.198 pp (0.86%) relative to the matched reference. Its contribution to later damage prediction is smaller than its immediate-reply value. This comparison used four-archive training data; the five-archive baseline is a separate result. Ten-ply analysis followed a prespecified immediate-regret gate, so it remains exploratory development evidence.


In [6]:
show('matched_ten_ply_maia_regression',
'<table border="1" class="dataframe">\n  <thead>\n    <tr style="text-align: right;">\n      <th>archive_set</th>\n      <th>protocol</th>\n      <th>variant</th>\n      <th>target</th>\n      <th>n</th>\n      <th>games</th>\n      <th>mae</th>\n      <th>spearman</th>\n      <th>recovered_n_diagnostic</th>\n      <th>recovered_only_mae_diagnostic</th>\n    </tr>\n  </thead>\n  <tbody>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>fixed_lightgbm</td>\n      <td>change_in_damage_10ply</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>22.930386</td>\n      <td>0.133253</td>\n      <td>6101</td>\n      <td>40.483837</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>plus_maia</td>\n      <td>change_in_damage_10ply</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>22.732183</td>\n      <td>0.178333</td>\n      <td>6101</td>\n      <td>38.738184</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>training_mean</td>\n      <td>change_in_damage_10ply</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>25.372458</td>\n      <td>unavailable / undefined</td>\n      <td>6101</td>\n      <td>33.281956</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>zero</td>\n      <td>change_in_damage_10ply</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>23.454479</td>\n      <td>unavailable / undefined</td>\n      <td>6101</td>\n      <td>43.552549</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>fixed_lightgbm</td>\n      <td>change_in_damage_10ply</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>23.056998</td>\n      <td>0.123293</td>\n      <td>6101</td>\n      <td>40.572835</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>plus_maia</td>\n      <td>change_in_damage_10ply</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>22.823851</td>\n      <td>0.174504</td>\n      <td>6101</td>\n      <td>38.848907</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>training_mean</td>\n      <td>change_in_damage_10ply</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>25.312923</td>\n      <td>unavailable / undefined</td>\n      <td>6101</td>\n      <td>33.474108</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>zero</td>\n      <td>change_in_damage_10ply</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>23.454479</td>\n      <td>unavailable / undefined</td>\n      <td>6101</td>\n      <td>43.552549</td>\n    </tr>\n  </tbody>\n</table>',
'Both models were fitted on the same four-archive training folds.')

show('matched_ten_ply_maia_increment',
'<table border="1" class="dataframe">\n  <thead>\n    <tr style="text-align: right;">\n      <th>archive_set</th>\n      <th>protocol</th>\n      <th>target</th>\n      <th>variant</th>\n      <th>comparator</th>\n      <th>initial_damage_threshold</th>\n      <th>n</th>\n      <th>games</th>\n      <th>mae_improvement_pp</th>\n      <th>lower95</th>\n      <th>upper95</th>\n      <th>improvement_percent</th>\n      <th>percent_lower95</th>\n      <th>percent_upper95</th>\n      <th>bootstrap_repeats</th>\n    </tr>\n  </thead>\n  <tbody>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>change_in_damage_10ply</td>\n      <td>plus_maia</td>\n      <td>fixed_lightgbm</td>\n      <td>20</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>0.198204</td>\n      <td>0.116301</td>\n      <td>0.281079</td>\n      <td>0.864371</td>\n      <td>0.506563</td>\n      <td>1.222197</td>\n      <td>2000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>change_in_damage_10ply</td>\n      <td>plus_maia</td>\n      <td>fixed_lightgbm</td>\n      <td>20</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>0.233147</td>\n      <td>0.155220</td>\n      <td>0.308837</td>\n      <td>1.011178</td>\n      <td>0.671928</td>\n      <td>1.342351</td>\n      <td>2000</td>\n    </tr>\n  </tbody>\n</table>',
'Paired MAE improvement relative to the matched reference.')


**Matched ten ply Maia regression**

Both models were fitted on the same four-archive training folds.

archive_set,protocol,variant,target,n,games,mae,spearman,recovered_n_diagnostic,recovered_only_mae_diagnostic
"2017-04, 2017-06, 2017-07, 2018-02",game,fixed_lightgbm,change_in_damage_10ply,15953,4485,22.930386,0.133253,6101,40.483837
"2017-04, 2017-06, 2017-07, 2018-02",game,plus_maia,change_in_damage_10ply,15953,4485,22.732183,0.178333,6101,38.738184
"2017-04, 2017-06, 2017-07, 2018-02",game,training_mean,change_in_damage_10ply,15953,4485,25.372458,unavailable / undefined,6101,33.281956
"2017-04, 2017-06, 2017-07, 2018-02",game,zero,change_in_damage_10ply,15953,4485,23.454479,unavailable / undefined,6101,43.552549
"2017-04, 2017-06, 2017-07, 2018-02",player,fixed_lightgbm,change_in_damage_10ply,15953,4485,23.056998,0.123293,6101,40.572835
"2017-04, 2017-06, 2017-07, 2018-02",player,plus_maia,change_in_damage_10ply,15953,4485,22.823851,0.174504,6101,38.848907
"2017-04, 2017-06, 2017-07, 2018-02",player,training_mean,change_in_damage_10ply,15953,4485,25.312923,unavailable / undefined,6101,33.474108
"2017-04, 2017-06, 2017-07, 2018-02",player,zero,change_in_damage_10ply,15953,4485,23.454479,unavailable / undefined,6101,43.552549


**Matched ten ply Maia increment**

Paired MAE improvement relative to the matched reference.

archive_set,protocol,target,variant,comparator,initial_damage_threshold,n,games,mae_improvement_pp,lower95,upper95,improvement_percent,percent_lower95,percent_upper95,bootstrap_repeats
"2017-04, 2017-06, 2017-07, 2018-02",game,change_in_damage_10ply,plus_maia,fixed_lightgbm,20,15953,4485,0.198204,0.116301,0.281079,0.864371,0.506563,1.222197,2000
"2017-04, 2017-06, 2017-07, 2018-02",player,change_in_damage_10ply,plus_maia,fixed_lightgbm,20,15953,4485,0.233147,0.155220,0.308837,1.011178,0.671928,1.342351,2000


### MultiPV features

The historical feature-group comparison evaluates context, context+MultiPV, full and full+MultiPV inputs using the same candidate grid. Each entry is the lowest mean fold MAE selected within that feature group. Differences between these minima combine feature and configuration selection; they do not isolate a controlled MultiPV effect. Matched OOF predictions, Spearman and paired intervals for this comparison are unavailable.

The two added features describe the number of strong replies within 10 winning-chance points of the best and the best–second gap. They measure available engine replies, not the human's awareness or choice.


In [7]:
show('multipv_feature_group_selection',
'<table border="1" class="dataframe">\n  <thead>\n    <tr style="text-align: right;">\n      <th>feature_set</th>\n      <th>archive_set</th>\n      <th>protocol</th>\n      <th>horizon</th>\n      <th>context</th>\n      <th>context_multipv</th>\n      <th>full</th>\n      <th>full_multipv</th>\n      <th>context_minus_context_multipv</th>\n      <th>full_minus_full_multipv</th>\n    </tr>\n  </thead>\n  <tbody>\n    <tr>\n      <th></th>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Historical 3-fold game CV used for selection</td>\n      <td>2</td>\n      <td>12.562096</td>\n      <td>12.548324</td>\n      <td>12.550759</td>\n      <td>12.546288</td>\n      <td>0.013772</td>\n      <td>0.004472</td>\n    </tr>\n    <tr>\n      <th></th>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Historical 3-fold game CV used for selection</td>\n      <td>4</td>\n      <td>16.799494</td>\n      <td>16.795291</td>\n      <td>16.808617</td>\n      <td>16.786366</td>\n      <td>0.004203</td>\n      <td>0.022252</td>\n    </tr>\n    <tr>\n      <th></th>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Historical 3-fold game CV used for selection</td>\n      <td>6</td>\n      <td>19.581606</td>\n      <td>19.575286</td>\n      <td>19.571595</td>\n      <td>19.555519</td>\n      <td>0.006320</td>\n      <td>0.016076</td>\n    </tr>\n    <tr>\n      <th></th>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Historical 3-fold game CV used for selection</td>\n      <td>10</td>\n      <td>23.053305</td>\n      <td>23.052400</td>\n      <td>23.069729</td>\n      <td>23.071354</td>\n      <td>0.000905</td>\n      <td>-0.001625</td>\n    </tr>\n  </tbody>\n</table>',
'Selected mean fold MAE; matched OOF predictions and paired intervals unavailable.')


**MultiPV feature group selection**

Selected mean fold MAE; matched OOF predictions and paired intervals unavailable.

feature_set,archive_set,protocol,horizon,context,context_multipv,full,full_multipv,context_minus_context_multipv,full_minus_full_multipv
,"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",Historical 3-fold game CV used for selection,2,12.562096,12.548324,12.550759,12.546288,0.013772,0.004472
,"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",Historical 3-fold game CV used for selection,4,16.799494,16.795291,16.808617,16.786366,0.004203,0.022252
,"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",Historical 3-fold game CV used for selection,6,19.581606,19.575286,19.571595,19.555519,0.006320,0.016076
,"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",Historical 3-fold game CV used for selection,10,23.053305,23.052400,23.069729,23.071354,0.000905,-0.001625


### Clock time and later persistence

The five-archive feature study compares existing clock-seconds/time-control inputs with normalized clocks, depth-8/depth-18 best-reply agreement and both additions. Normalized clocks divide the pre-mistake clock by initial time; they are unclipped and missing when initial time is invalid. Agreement compares engine-selected replies rather than the observed human move.

The six- and ten-ply results are followed by fixed-model errors grouped at 30 seconds. These groups are descriptive. Clock associations and isolated feature gains do not establish causal effects of time pressure. The additional feature variants remain development-only.


In [8]:
show('longer_horizon_clock_and_agreement',
'<table border="1" class="dataframe">\n  <thead>\n    <tr style="text-align: right;">\n      <th>archive_set</th>\n      <th>protocol</th>\n      <th>target</th>\n      <th>variant</th>\n      <th>n</th>\n      <th>games</th>\n      <th>mae</th>\n      <th>spearman</th>\n      <th>mae_improvement_pp</th>\n      <th>lower95</th>\n      <th>upper95</th>\n    </tr>\n  </thead>\n  <tbody>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>change_in_damage_10ply</td>\n      <td>baseline</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>23.049053</td>\n      <td>0.123120</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>change_in_damage_10ply</td>\n      <td>both</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>23.053643</td>\n      <td>0.118963</td>\n      <td>-0.004590</td>\n      <td>-0.045439</td>\n      <td>0.039041</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>change_in_damage_10ply</td>\n      <td>normalized_clocks</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>23.054470</td>\n      <td>0.123463</td>\n      <td>-0.005417</td>\n      <td>-0.047356</td>\n      <td>0.037955</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>change_in_damage_10ply</td>\n      <td>reply_agreement</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>23.040167</td>\n      <td>0.122135</td>\n      <td>0.008886</td>\n      <td>-0.003307</td>\n      <td>0.020195</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>change_in_damage_6ply</td>\n      <td>baseline</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>19.579173</td>\n      <td>0.099988</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>change_in_damage_6ply</td>\n      <td>both</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>19.543005</td>\n      <td>0.106833</td>\n      <td>0.036168</td>\n      <td>-0.004298</td>\n      <td>0.077922</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>change_in_damage_6ply</td>\n      <td>normalized_clocks</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>19.544598</td>\n      <td>0.102725</td>\n      <td>0.034576</td>\n      <td>-0.006820</td>\n      <td>0.076168</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>change_in_damage_6ply</td>\n      <td>reply_agreement</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>19.586882</td>\n      <td>0.101255</td>\n      <td>-0.007709</td>\n      <td>-0.018230</td>\n      <td>0.003011</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>change_in_damage_10ply</td>\n      <td>baseline</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>23.133595</td>\n      <td>0.111968</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>change_in_damage_10ply</td>\n      <td>both</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>23.118334</td>\n      <td>0.113275</td>\n      <td>0.015261</td>\n      <td>-0.020694</td>\n      <td>0.054451</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>change_in_damage_10ply</td>\n      <td>normalized_clocks</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>23.119203</td>\n      <td>0.113263</td>\n      <td>0.014392</td>\n      <td>-0.023591</td>\n      <td>0.052907</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>change_in_damage_10ply</td>\n      <td>reply_agreement</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>23.145713</td>\n      <td>0.111570</td>\n      <td>-0.012118</td>\n      <td>-0.024947</td>\n      <td>-0.000283</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>change_in_damage_6ply</td>\n      <td>baseline</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>19.617191</td>\n      <td>0.094367</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>change_in_damage_6ply</td>\n      <td>both</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>19.605579</td>\n      <td>0.097475</td>\n      <td>0.011611</td>\n      <td>-0.012791</td>\n      <td>0.036025</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>change_in_damage_6ply</td>\n      <td>normalized_clocks</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>19.609958</td>\n      <td>0.095814</td>\n      <td>0.007233</td>\n      <td>-0.018022</td>\n      <td>0.032995</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>change_in_damage_6ply</td>\n      <td>reply_agreement</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>19.598597</td>\n      <td>0.102162</td>\n      <td>0.018593</td>\n      <td>0.007540</td>\n      <td>0.030048</td>\n    </tr>\n  </tbody>\n</table>',
'MAE gains are relative to the original fixed LightGBM.')

show('longer_horizon_clock_descriptive',
'<table border="1" class="dataframe">\n  <thead>\n    <tr style="text-align: right;">\n      <th>archive_set</th>\n      <th>protocol</th>\n      <th>horizon</th>\n      <th>clock</th>\n      <th>clock_group</th>\n      <th>model</th>\n      <th>n</th>\n      <th>games</th>\n      <th>mae</th>\n      <th>spearman</th>\n    </tr>\n  </thead>\n  <tbody>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>10</td>\n      <td>opponent_clock_before_move</td>\n      <td>&lt;=30 seconds</td>\n      <td>fixed_lightgbm</td>\n      <td>4658</td>\n      <td>1688</td>\n      <td>25.940683</td>\n      <td>0.323991</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>10</td>\n      <td>opponent_clock_before_move</td>\n      <td>&gt;30 seconds</td>\n      <td>fixed_lightgbm</td>\n      <td>18440</td>\n      <td>5930</td>\n      <td>22.335585</td>\n      <td>0.056293</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>10</td>\n      <td>opponent_clock_before_move</td>\n      <td>Missing</td>\n      <td>fixed_lightgbm</td>\n      <td>69</td>\n      <td>21</td>\n      <td>18.514872</td>\n      <td>-0.002667</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>10</td>\n      <td>opponent_clock_before_move</td>\n      <td>&lt;=30 seconds</td>\n      <td>no_change</td>\n      <td>4658</td>\n      <td>1688</td>\n      <td>28.631110</td>\n      <td>unavailable / undefined</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>10</td>\n      <td>opponent_clock_before_move</td>\n      <td>&gt;30 seconds</td>\n      <td>no_change</td>\n      <td>18440</td>\n      <td>5930</td>\n      <td>22.392859</td>\n      <td>unavailable / undefined</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>10</td>\n      <td>opponent_clock_before_move</td>\n      <td>Missing</td>\n      <td>no_change</td>\n      <td>69</td>\n      <td>21</td>\n      <td>18.457144</td>\n      <td>unavailable / undefined</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>opponent_clock_before_move</td>\n      <td>&lt;=30 seconds</td>\n      <td>fixed_lightgbm</td>\n      <td>4658</td>\n      <td>1688</td>\n      <td>22.688692</td>\n      <td>0.220980</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>opponent_clock_before_move</td>\n      <td>&gt;30 seconds</td>\n      <td>fixed_lightgbm</td>\n      <td>18440</td>\n      <td>5930</td>\n      <td>18.807950</td>\n      <td>0.063158</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>opponent_clock_before_move</td>\n      <td>Missing</td>\n      <td>fixed_lightgbm</td>\n      <td>69</td>\n      <td>21</td>\n      <td>15.770742</td>\n      <td>-0.109207</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>opponent_clock_before_move</td>\n      <td>&lt;=30 seconds</td>\n      <td>no_change</td>\n      <td>4658</td>\n      <td>1688</td>\n      <td>23.785083</td>\n      <td>unavailable / undefined</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>opponent_clock_before_move</td>\n      <td>&gt;30 seconds</td>\n      <td>no_change</td>\n      <td>18440</td>\n      <td>5930</td>\n      <td>18.852195</td>\n      <td>unavailable / undefined</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>opponent_clock_before_move</td>\n      <td>Missing</td>\n      <td>no_change</td>\n      <td>69</td>\n      <td>21</td>\n      <td>15.656275</td>\n      <td>unavailable / undefined</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>10</td>\n      <td>opponent_clock_before_move</td>\n      <td>&lt;=30 seconds</td>\n      <td>fixed_lightgbm</td>\n      <td>4658</td>\n      <td>1688</td>\n      <td>26.283278</td>\n      <td>0.294241</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>10</td>\n      <td>opponent_clock_before_move</td>\n      <td>&gt;30 seconds</td>\n      <td>fixed_lightgbm</td>\n      <td>18440</td>\n      <td>5930</td>\n      <td>22.356026</td>\n      <td>0.052243</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>10</td>\n      <td>opponent_clock_before_move</td>\n      <td>Missing</td>\n      <td>fixed_lightgbm</td>\n      <td>69</td>\n      <td>21</td>\n      <td>18.309473</td>\n      <td>0.050786</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>10</td>\n      <td>opponent_clock_before_move</td>\n      <td>&lt;=30 seconds</td>\n      <td>no_change</td>\n      <td>4658</td>\n      <td>1688</td>\n      <td>28.631110</td>\n      <td>unavailable / undefined</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>10</td>\n      <td>opponent_clock_before_move</td>\n      <td>&gt;30 seconds</td>\n      <td>no_change</td>\n      <td>18440</td>\n      <td>5930</td>\n      <td>22.392859</td>\n      <td>unavailable / undefined</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>10</td>\n      <td>opponent_clock_before_move</td>\n      <td>Missing</td>\n      <td>no_change</td>\n      <td>69</td>\n      <td>21</td>\n      <td>18.457144</td>\n      <td>unavailable / undefined</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>opponent_clock_before_move</td>\n      <td>&lt;=30 seconds</td>\n      <td>fixed_lightgbm</td>\n      <td>4658</td>\n      <td>1688</td>\n      <td>22.807210</td>\n      <td>0.218104</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>opponent_clock_before_move</td>\n      <td>&gt;30 seconds</td>\n      <td>fixed_lightgbm</td>\n      <td>18440</td>\n      <td>5930</td>\n      <td>18.825699</td>\n      <td>0.056819</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>opponent_clock_before_move</td>\n      <td>Missing</td>\n      <td>fixed_lightgbm</td>\n      <td>69</td>\n      <td>21</td>\n      <td>15.791080</td>\n      <td>-0.106577</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>opponent_clock_before_move</td>\n      <td>&lt;=30 seconds</td>\n      <td>no_change</td>\n      <td>4658</td>\n      <td>1688</td>\n      <td>23.785083</td>\n      <td>unavailable / undefined</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>opponent_clock_before_move</td>\n      <td>&gt;30 seconds</td>\n      <td>no_change</td>\n      <td>18440</td>\n      <td>5930</td>\n      <td>18.852195</td>\n      <td>unavailable / undefined</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>opponent_clock_before_move</td>\n      <td>Missing</td>\n      <td>no_change</td>\n      <td>69</td>\n      <td>21</td>\n      <td>15.656275</td>\n      <td>unavailable / undefined</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>10</td>\n      <td>player_clock_before_move</td>\n      <td>&lt;=30 seconds</td>\n      <td>fixed_lightgbm</td>\n      <td>4506</td>\n      <td>1688</td>\n      <td>24.473566</td>\n      <td>0.309330</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>10</td>\n      <td>player_clock_before_move</td>\n      <td>&gt;30 seconds</td>\n      <td>fixed_lightgbm</td>\n      <td>18592</td>\n      <td>5941</td>\n      <td>22.720632</td>\n      <td>0.069344</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>10</td>\n      <td>player_clock_before_move</td>\n      <td>Missing</td>\n      <td>fixed_lightgbm</td>\n      <td>69</td>\n      <td>21</td>\n      <td>18.514872</td>\n      <td>-0.002667</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>10</td>\n      <td>player_clock_before_move</td>\n      <td>&lt;=30 seconds</td>\n      <td>no_change</td>\n      <td>4506</td>\n      <td>1688</td>\n      <td>26.623620</td>\n      <td>unavailable / undefined</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>10</td>\n      <td>player_clock_before_move</td>\n      <td>&gt;30 seconds</td>\n      <td>no_change</td>\n      <td>18592</td>\n      <td>5941</td>\n      <td>22.930400</td>\n      <td>unavailable / undefined</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>10</td>\n      <td>player_clock_before_move</td>\n      <td>Missing</td>\n      <td>no_change</td>\n      <td>69</td>\n      <td>21</td>\n      <td>18.457144</td>\n      <td>unavailable / undefined</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>player_clock_before_move</td>\n      <td>&lt;=30 seconds</td>\n      <td>fixed_lightgbm</td>\n      <td>4506</td>\n      <td>1688</td>\n      <td>21.773149</td>\n      <td>0.207945</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>player_clock_before_move</td>\n      <td>&gt;30 seconds</td>\n      <td>fixed_lightgbm</td>\n      <td>18592</td>\n      <td>5941</td>\n      <td>19.061570</td>\n      <td>0.071948</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>player_clock_before_move</td>\n      <td>Missing</td>\n      <td>fixed_lightgbm</td>\n      <td>69</td>\n      <td>21</td>\n      <td>15.770742</td>\n      <td>-0.109207</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>player_clock_before_move</td>\n      <td>&lt;=30 seconds</td>\n      <td>no_change</td>\n      <td>4506</td>\n      <td>1688</td>\n      <td>22.572389</td>\n      <td>unavailable / undefined</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>player_clock_before_move</td>\n      <td>&gt;30 seconds</td>\n      <td>no_change</td>\n      <td>18592</td>\n      <td>5941</td>\n      <td>19.186435</td>\n      <td>unavailable / undefined</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>player_clock_before_move</td>\n      <td>Missing</td>\n      <td>no_change</td>\n      <td>69</td>\n      <td>21</td>\n      <td>15.656275</td>\n      <td>unavailable / undefined</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>10</td>\n      <td>player_clock_before_move</td>\n      <td>&lt;=30 seconds</td>\n      <td>fixed_lightgbm</td>\n      <td>4506</td>\n      <td>1688</td>\n      <td>24.827405</td>\n      <td>0.277526</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>10</td>\n      <td>player_clock_before_move</td>\n      <td>&gt;30 seconds</td>\n      <td>fixed_lightgbm</td>\n      <td>18592</td>\n      <td>5941</td>\n      <td>22.740983</td>\n      <td>0.065714</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>10</td>\n      <td>player_clock_before_move</td>\n      <td>Missing</td>\n      <td>fixed_lightgbm</td>\n      <td>69</td>\n      <td>21</td>\n      <td>18.309473</td>\n      <td>0.050786</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>10</td>\n      <td>player_clock_before_move</td>\n      <td>&lt;=30 seconds</td>\n      <td>no_change</td>\n      <td>4506</td>\n      <td>1688</td>\n      <td>26.623620</td>\n      <td>unavailable / undefined</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>10</td>\n      <td>player_clock_before_move</td>\n      <td>&gt;30 seconds</td>\n      <td>no_change</td>\n      <td>18592</td>\n      <td>5941</td>\n      <td>22.930400</td>\n      <td>unavailable / undefined</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>10</td>\n      <td>player_clock_before_move</td>\n      <td>Missing</td>\n      <td>no_change</td>\n      <td>69</td>\n      <td>21</td>\n      <td>18.457144</td>\n      <td>unavailable / undefined</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>player_clock_before_move</td>\n      <td>&lt;=30 seconds</td>\n      <td>fixed_lightgbm</td>\n      <td>4506</td>\n      <td>1688</td>\n      <td>21.876802</td>\n      <td>0.210959</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>player_clock_before_move</td>\n      <td>&gt;30 seconds</td>\n      <td>fixed_lightgbm</td>\n      <td>18592</td>\n      <td>5941</td>\n      <td>19.083746</td>\n      <td>0.063687</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>player_clock_before_move</td>\n      <td>Missing</td>\n      <td>fixed_lightgbm</td>\n      <td>69</td>\n      <td>21</td>\n      <td>15.791080</td>\n      <td>-0.106577</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>player_clock_before_move</td>\n      <td>&lt;=30 seconds</td>\n      <td>no_change</td>\n      <td>4506</td>\n      <td>1688</td>\n      <td>22.572389</td>\n      <td>unavailable / undefined</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>player_clock_before_move</td>\n      <td>&gt;30 seconds</td>\n      <td>no_change</td>\n      <td>18592</td>\n      <td>5941</td>\n      <td>19.186435</td>\n      <td>unavailable / undefined</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>player_clock_before_move</td>\n      <td>Missing</td>\n      <td>no_change</td>\n      <td>69</td>\n      <td>21</td>\n      <td>15.656275</td>\n      <td>unavailable / undefined</td>\n    </tr>\n  </tbody>\n</table>',
'Descriptive fixed-model OOF errors; no subgroup intervals.')


**Longer horizon clock and agreement**

MAE gains are relative to the original fixed LightGBM.

archive_set,protocol,target,variant,n,games,mae,spearman,mae_improvement_pp,lower95,upper95
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,change_in_damage_10ply,baseline,23167,6512,23.049053,0.123120,0.000000,0.000000,0.000000
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,change_in_damage_10ply,both,23167,6512,23.053643,0.118963,-0.004590,-0.045439,0.039041
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,change_in_damage_10ply,normalized_clocks,23167,6512,23.054470,0.123463,-0.005417,-0.047356,0.037955
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,change_in_damage_10ply,reply_agreement,23167,6512,23.040167,0.122135,0.008886,-0.003307,0.020195
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,change_in_damage_6ply,baseline,23167,6512,19.579173,0.099988,0.000000,0.000000,0.000000
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,change_in_damage_6ply,both,23167,6512,19.543005,0.106833,0.036168,-0.004298,0.077922
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,change_in_damage_6ply,normalized_clocks,23167,6512,19.544598,0.102725,0.034576,-0.006820,0.076168
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,change_in_damage_6ply,reply_agreement,23167,6512,19.586882,0.101255,-0.007709,-0.018230,0.003011
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",player,change_in_damage_10ply,baseline,23167,6512,23.133595,0.111968,0.000000,0.000000,0.000000
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",player,change_in_damage_10ply,both,23167,6512,23.118334,0.113275,0.015261,-0.020694,0.054451


**Longer horizon clock descriptive**

Descriptive fixed-model OOF errors; no subgroup intervals.

archive_set,protocol,horizon,clock,clock_group,model,n,games,mae,spearman
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,10,opponent_clock_before_move,<=30 seconds,fixed_lightgbm,4658,1688,25.940683,0.323991
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,10,opponent_clock_before_move,>30 seconds,fixed_lightgbm,18440,5930,22.335585,0.056293
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,10,opponent_clock_before_move,Missing,fixed_lightgbm,69,21,18.514872,-0.002667
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,10,opponent_clock_before_move,<=30 seconds,no_change,4658,1688,28.631110,unavailable / undefined
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,10,opponent_clock_before_move,>30 seconds,no_change,18440,5930,22.392859,unavailable / undefined
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,10,opponent_clock_before_move,Missing,no_change,69,21,18.457144,unavailable / undefined
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,6,opponent_clock_before_move,<=30 seconds,fixed_lightgbm,4658,1688,22.688692,0.220980
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,6,opponent_clock_before_move,>30 seconds,fixed_lightgbm,18440,5930,18.807950,0.063158
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,6,opponent_clock_before_move,Missing,fixed_lightgbm,69,21,15.770742,-0.109207
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,6,opponent_clock_before_move,<=30 seconds,no_change,4658,1688,23.785083,unavailable / undefined


### Later recovered-versus-persistent decisions

Recovery is the positive class, defined by actual persistence ratio <0.75. Regression-derived decisions apply the same boundary to predicted ratio. ROC-AUC uses the negative continuous predicted ratio, a ranking score rather than a calibrated probability. The majority baseline is estimated from training labels.

Results are separated by five-archive and four-archive cohorts. Accuracy can remain near the majority baseline while recovery recall is low. These labels concern later damage and differ from immediate-reply preservation. Separate classifier experiments are reported in Section 4.


In [9]:
show('later_regression_derived_five_archives',
'<table border="1" class="dataframe">\n  <thead>\n    <tr style="text-align: right;">\n      <th>archive_set</th>\n      <th>protocol</th>\n      <th>horizon</th>\n      <th>model</th>\n      <th>kind</th>\n      <th>games</th>\n      <th>majority_accuracy</th>\n      <th>n</th>\n      <th>accuracy</th>\n      <th>balanced_accuracy</th>\n      <th>precision</th>\n      <th>recall</th>\n      <th>other_class_recall</th>\n      <th>tn</th>\n      <th>fp</th>\n      <th>fn</th>\n      <th>tp</th>\n      <th>roc_auc</th>\n    </tr>\n  </thead>\n  <tbody>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>10</td>\n      <td>decision_tree</td>\n      <td>regression-derived</td>\n      <td>6512</td>\n      <td>0.617041</td>\n      <td>23167</td>\n      <td>0.461001</td>\n      <td>0.501922</td>\n      <td>0.384305</td>\n      <td>0.676736</td>\n      <td>0.327107</td>\n      <td>4676</td>\n      <td>9619</td>\n      <td>2868</td>\n      <td>6004</td>\n      <td>0.523015</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>10</td>\n      <td>engine_only</td>\n      <td>regression-derived</td>\n      <td>6512</td>\n      <td>0.617041</td>\n      <td>23167</td>\n      <td>0.614408</td>\n      <td>0.501864</td>\n      <td>0.429885</td>\n      <td>0.021078</td>\n      <td>0.982651</td>\n      <td>14047</td>\n      <td>248</td>\n      <td>8685</td>\n      <td>187</td>\n      <td>0.526795</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>10</td>\n      <td>engine_ratings</td>\n      <td>regression-derived</td>\n      <td>6512</td>\n      <td>0.617041</td>\n      <td>23167</td>\n      <td>0.619416</td>\n      <td>0.512464</td>\n      <td>0.529538</td>\n      <td>0.055568</td>\n      <td>0.969360</td>\n      <td>13857</td>\n      <td>438</td>\n      <td>8379</td>\n      <td>493</td>\n      <td>0.549214</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>10</td>\n      <td>fixed_lightgbm</td>\n      <td>regression-derived</td>\n      <td>6512</td>\n      <td>0.617041</td>\n      <td>23167</td>\n      <td>0.629128</td>\n      <td>0.525016</td>\n      <td>0.622378</td>\n      <td>0.080252</td>\n      <td>0.969780</td>\n      <td>13863</td>\n      <td>432</td>\n      <td>8160</td>\n      <td>712</td>\n      <td>0.582195</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>10</td>\n      <td>no_change</td>\n      <td>regression-derived</td>\n      <td>6512</td>\n      <td>0.617041</td>\n      <td>23167</td>\n      <td>0.617041</td>\n      <td>0.500000</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>1.000000</td>\n      <td>14295</td>\n      <td>0</td>\n      <td>8872</td>\n      <td>0</td>\n      <td>0.500000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>10</td>\n      <td>random_forest</td>\n      <td>regression-derived</td>\n      <td>6512</td>\n      <td>0.617041</td>\n      <td>23167</td>\n      <td>0.484137</td>\n      <td>0.515282</td>\n      <td>0.394432</td>\n      <td>0.648332</td>\n      <td>0.382232</td>\n      <td>5464</td>\n      <td>8831</td>\n      <td>3120</td>\n      <td>5752</td>\n      <td>0.532677</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>10</td>\n      <td>training_mean</td>\n      <td>regression-derived</td>\n      <td>6512</td>\n      <td>0.617041</td>\n      <td>23167</td>\n      <td>0.454828</td>\n      <td>0.522383</td>\n      <td>0.396462</td>\n      <td>0.810978</td>\n      <td>0.233788</td>\n      <td>3342</td>\n      <td>10953</td>\n      <td>1677</td>\n      <td>7195</td>\n      <td>0.541111</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>10</td>\n      <td>training_median</td>\n      <td>regression-derived</td>\n      <td>6512</td>\n      <td>0.617041</td>\n      <td>23167</td>\n      <td>0.617041</td>\n      <td>0.500000</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>1.000000</td>\n      <td>14295</td>\n      <td>0</td>\n      <td>8872</td>\n      <td>0</td>\n      <td>0.464303</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>2</td>\n      <td>decision_tree</td>\n      <td>regression-derived</td>\n      <td>6512</td>\n      <td>0.724263</td>\n      <td>23167</td>\n      <td>0.637329</td>\n      <td>0.540759</td>\n      <td>0.336844</td>\n      <td>0.325454</td>\n      <td>0.756064</td>\n      <td>12686</td>\n      <td>4093</td>\n      <td>4309</td>\n      <td>2079</td>\n      <td>0.562099</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>2</td>\n      <td>engine_only</td>\n      <td>regression-derived</td>\n      <td>6512</td>\n      <td>0.724263</td>\n      <td>23167</td>\n      <td>0.724263</td>\n      <td>0.500000</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>1.000000</td>\n      <td>16779</td>\n      <td>0</td>\n      <td>6388</td>\n      <td>0</td>\n      <td>0.497877</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>2</td>\n      <td>engine_ratings</td>\n      <td>regression-derived</td>\n      <td>6512</td>\n      <td>0.724263</td>\n      <td>23167</td>\n      <td>0.724263</td>\n      <td>0.500000</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>1.000000</td>\n      <td>16779</td>\n      <td>0</td>\n      <td>6388</td>\n      <td>0</td>\n      <td>0.502524</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>2</td>\n      <td>fixed_lightgbm</td>\n      <td>regression-derived</td>\n      <td>6512</td>\n      <td>0.724263</td>\n      <td>23167</td>\n      <td>0.724263</td>\n      <td>0.500000</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>1.000000</td>\n      <td>16779</td>\n      <td>0</td>\n      <td>6388</td>\n      <td>0</td>\n      <td>0.593282</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>2</td>\n      <td>no_change</td>\n      <td>regression-derived</td>\n      <td>6512</td>\n      <td>0.724263</td>\n      <td>23167</td>\n      <td>0.724263</td>\n      <td>0.500000</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>1.000000</td>\n      <td>16779</td>\n      <td>0</td>\n      <td>6388</td>\n      <td>0</td>\n      <td>0.500000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>2</td>\n      <td>random_forest</td>\n      <td>regression-derived</td>\n      <td>6512</td>\n      <td>0.724263</td>\n      <td>23167</td>\n      <td>0.647300</td>\n      <td>0.550987</td>\n      <td>0.353348</td>\n      <td>0.336255</td>\n      <td>0.765719</td>\n      <td>12848</td>\n      <td>3931</td>\n      <td>4240</td>\n      <td>2148</td>\n      <td>0.575998</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>2</td>\n      <td>training_mean</td>\n      <td>regression-derived</td>\n      <td>6512</td>\n      <td>0.724263</td>\n      <td>23167</td>\n      <td>0.622782</td>\n      <td>0.517871</td>\n      <td>0.303395</td>\n      <td>0.283970</td>\n      <td>0.751773</td>\n      <td>12614</td>\n      <td>4165</td>\n      <td>4574</td>\n      <td>1814</td>\n      <td>0.530395</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>2</td>\n      <td>training_median</td>\n      <td>regression-derived</td>\n      <td>6512</td>\n      <td>0.724263</td>\n      <td>23167</td>\n      <td>0.724263</td>\n      <td>0.500000</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>1.000000</td>\n      <td>16779</td>\n      <td>0</td>\n      <td>6388</td>\n      <td>0</td>\n      <td>0.490478</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>4</td>\n      <td>decision_tree</td>\n      <td>regression-derived</td>\n      <td>6512</td>\n      <td>0.667242</td>\n      <td>23167</td>\n      <td>0.551474</td>\n      <td>0.532704</td>\n      <td>0.366301</td>\n      <td>0.476586</td>\n      <td>0.588821</td>\n      <td>9102</td>\n      <td>6356</td>\n      <td>4035</td>\n      <td>3674</td>\n      <td>0.532280</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>4</td>\n      <td>engine_only</td>\n      <td>regression-derived</td>\n      <td>6512</td>\n      <td>0.667242</td>\n      <td>23167</td>\n      <td>0.667026</td>\n      <td>0.500001</td>\n      <td>0.333333</td>\n      <td>0.000649</td>\n      <td>0.999353</td>\n      <td>15448</td>\n      <td>10</td>\n      <td>7704</td>\n      <td>5</td>\n      <td>0.527356</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>4</td>\n      <td>engine_ratings</td>\n      <td>regression-derived</td>\n      <td>6512</td>\n      <td>0.667242</td>\n      <td>23167</td>\n      <td>0.666897</td>\n      <td>0.500294</td>\n      <td>0.404762</td>\n      <td>0.002205</td>\n      <td>0.998383</td>\n      <td>15433</td>\n      <td>25</td>\n      <td>7692</td>\n      <td>17</td>\n      <td>0.531626</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>4</td>\n      <td>fixed_lightgbm</td>\n      <td>regression-derived</td>\n      <td>6512</td>\n      <td>0.667242</td>\n      <td>23167</td>\n      <td>0.668580</td>\n      <td>0.505847</td>\n      <td>0.558052</td>\n      <td>0.019328</td>\n      <td>0.992366</td>\n      <td>15340</td>\n      <td>118</td>\n      <td>7560</td>\n      <td>149</td>\n      <td>0.585270</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>4</td>\n      <td>no_change</td>\n      <td>regression-derived</td>\n      <td>6512</td>\n      <td>0.667242</td>\n      <td>23167</td>\n      <td>0.667242</td>\n      <td>0.500000</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>1.000000</td>\n      <td>15458</td>\n      <td>0</td>\n      <td>7709</td>\n      <td>0</td>\n      <td>0.500000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>4</td>\n      <td>random_forest</td>\n      <td>regression-derived</td>\n      <td>6512</td>\n      <td>0.667242</td>\n      <td>23167</td>\n      <td>0.560755</td>\n      <td>0.539560</td>\n      <td>0.374248</td>\n      <td>0.476197</td>\n      <td>0.602924</td>\n      <td>9320</td>\n      <td>6138</td>\n      <td>4038</td>\n      <td>3671</td>\n      <td>0.546314</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>4</td>\n      <td>training_mean</td>\n      <td>regression-derived</td>\n      <td>6512</td>\n      <td>0.667242</td>\n      <td>23167</td>\n      <td>0.522251</td>\n      <td>0.526932</td>\n      <td>0.356441</td>\n      <td>0.540926</td>\n      <td>0.512938</td>\n      <td>7929</td>\n      <td>7529</td>\n      <td>3539</td>\n      <td>4170</td>\n      <td>0.532278</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>4</td>\n      <td>training_median</td>\n      <td>regression-derived</td>\n      <td>6512</td>\n      <td>0.667242</td>\n      <td>23167</td>\n      <td>0.667242</td>\n      <td>0.500000</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>1.000000</td>\n      <td>15458</td>\n      <td>0</td>\n      <td>7709</td>\n      <td>0</td>\n      <td>0.488401</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>decision_tree</td>\n      <td>regression-derived</td>\n      <td>6512</td>\n      <td>0.641689</td>\n      <td>23167</td>\n      <td>0.499460</td>\n      <td>0.517016</td>\n      <td>0.372356</td>\n      <td>0.578966</td>\n      <td>0.455065</td>\n      <td>6765</td>\n      <td>8101</td>\n      <td>3495</td>\n      <td>4806</td>\n      <td>0.524545</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>engine_only</td>\n      <td>regression-derived</td>\n      <td>6512</td>\n      <td>0.641689</td>\n      <td>23167</td>\n      <td>0.641602</td>\n      <td>0.501050</td>\n      <td>0.488372</td>\n      <td>0.005060</td>\n      <td>0.997040</td>\n      <td>14822</td>\n      <td>44</td>\n      <td>8259</td>\n      <td>42</td>\n      <td>0.527543</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>engine_ratings</td>\n      <td>regression-derived</td>\n      <td>6512</td>\n      <td>0.641689</td>\n      <td>23167</td>\n      <td>0.641343</td>\n      <td>0.501965</td>\n      <td>0.477273</td>\n      <td>0.010119</td>\n      <td>0.993811</td>\n      <td>14774</td>\n      <td>92</td>\n      <td>8217</td>\n      <td>84</td>\n      <td>0.546217</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>fixed_lightgbm</td>\n      <td>regression-derived</td>\n      <td>6512</td>\n      <td>0.641689</td>\n      <td>23167</td>\n      <td>0.646307</td>\n      <td>0.510914</td>\n      <td>0.620767</td>\n      <td>0.033129</td>\n      <td>0.988699</td>\n      <td>14698</td>\n      <td>168</td>\n      <td>8026</td>\n      <td>275</td>\n      <td>0.581350</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>no_change</td>\n      <td>regression-derived</td>\n      <td>6512</td>\n      <td>0.641689</td>\n      <td>23167</td>\n      <td>0.641689</td>\n      <td>0.500000</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>1.000000</td>\n      <td>14866</td>\n      <td>0</td>\n      <td>8301</td>\n      <td>0</td>\n      <td>0.500000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>random_forest</td>\n      <td>regression-derived</td>\n      <td>6512</td>\n      <td>0.641689</td>\n      <td>23167</td>\n      <td>0.511892</td>\n      <td>0.524707</td>\n      <td>0.379421</td>\n      <td>0.569931</td>\n      <td>0.479483</td>\n      <td>7128</td>\n      <td>7738</td>\n      <td>3570</td>\n      <td>4731</td>\n      <td>0.533632</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>training_mean</td>\n      <td>regression-derived</td>\n      <td>6512</td>\n      <td>0.641689</td>\n      <td>23167</td>\n      <td>0.486856</td>\n      <td>0.529831</td>\n      <td>0.379639</td>\n      <td>0.681484</td>\n      <td>0.378178</td>\n      <td>5622</td>\n      <td>9244</td>\n      <td>2644</td>\n      <td>5657</td>\n      <td>0.539536</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>training_median</td>\n      <td>regression-derived</td>\n      <td>6512</td>\n      <td>0.641689</td>\n      <td>23167</td>\n      <td>0.641689</td>\n      <td>0.500000</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>1.000000</td>\n      <td>14866</td>\n      <td>0</td>\n      <td>8301</td>\n      <td>0</td>\n      <td>0.487226</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>10</td>\n      <td>decision_tree</td>\n      <td>regression-derived</td>\n      <td>6512</td>\n      <td>0.617041</td>\n      <td>23167</td>\n      <td>0.461087</td>\n      <td>0.500794</td>\n      <td>0.383519</td>\n      <td>0.670424</td>\n      <td>0.331165</td>\n      <td>4734</td>\n      <td>9561</td>\n      <td>2924</td>\n      <td>5948</td>\n      <td>0.518184</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>10</td>\n      <td>engine_only</td>\n      <td>regression-derived</td>\n      <td>6512</td>\n      <td>0.617041</td>\n      <td>23167</td>\n      <td>0.614797</td>\n      <td>0.503141</td>\n      <td>0.449612</td>\n      <td>0.026150</td>\n      <td>0.980133</td>\n      <td>14011</td>\n      <td>284</td>\n      <td>8640</td>\n      <td>232</td>\n      <td>0.520183</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>10</td>\n      <td>engine_ratings</td>\n      <td>regression-derived</td>\n      <td>6512</td>\n      <td>0.617041</td>\n      <td>23167</td>\n      <td>0.619372</td>\n      <td>0.512985</td>\n      <td>0.527439</td>\n      <td>0.058499</td>\n      <td>0.967471</td>\n      <td>13830</td>\n      <td>465</td>\n      <td>8353</td>\n      <td>519</td>\n      <td>0.544300</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>10</td>\n      <td>fixed_lightgbm</td>\n      <td>regression-derived</td>\n      <td>6512</td>\n      <td>0.617041</td>\n      <td>23167</td>\n      <td>0.626581</td>\n      <td>0.524064</td>\n      <td>0.584545</td>\n      <td>0.086114</td>\n      <td>0.962015</td>\n      <td>13752</td>\n      <td>543</td>\n      <td>8108</td>\n      <td>764</td>\n      <td>0.575963</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>10</td>\n      <td>no_change</td>\n      <td>regression-derived</td>\n      <td>6512</td>\n      <td>0.617041</td>\n      <td>23167</td>\n      <td>0.617041</td>\n      <td>0.500000</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>1.000000</td>\n      <td>14295</td>\n      <td>0</td>\n      <td>8872</td>\n      <td>0</td>\n      <td>0.500000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>10</td>\n      <td>random_forest</td>\n      <td>regression-derived</td>\n      <td>6512</td>\n      <td>0.617041</td>\n      <td>23167</td>\n      <td>0.478482</td>\n      <td>0.510871</td>\n      <td>0.391039</td>\n      <td>0.649234</td>\n      <td>0.372508</td>\n      <td>5325</td>\n      <td>8970</td>\n      <td>3112</td>\n      <td>5760</td>\n      <td>0.528471</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>10</td>\n      <td>training_mean</td>\n      <td>regression-derived</td>\n      <td>6512</td>\n      <td>0.617041</td>\n      <td>23167</td>\n      <td>0.458152</td>\n      <td>0.523003</td>\n      <td>0.397046</td>\n      <td>0.800045</td>\n      <td>0.245960</td>\n      <td>3516</td>\n      <td>10779</td>\n      <td>1774</td>\n      <td>7098</td>\n      <td>0.541113</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>10</td>\n      <td>training_median</td>\n      <td>regression-derived</td>\n      <td>6512</td>\n      <td>0.617041</td>\n      <td>23167</td>\n      <td>0.617041</td>\n      <td>0.500000</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>1.000000</td>\n      <td>14295</td>\n      <td>0</td>\n      <td>8872</td>\n      <td>0</td>\n      <td>0.459195</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>2</td>\n      <td>decision_tree</td>\n      <td>regression-derived</td>\n      <td>6512</td>\n      <td>0.724263</td>\n      <td>23167</td>\n      <td>0.633142</td>\n      <td>0.538935</td>\n      <td>0.332805</td>\n      <td>0.328898</td>\n      <td>0.748972</td>\n      <td>12567</td>\n      <td>4212</td>\n      <td>4287</td>\n      <td>2101</td>\n      <td>0.565826</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>2</td>\n      <td>engine_only</td>\n      <td>regression-derived</td>\n      <td>6512</td>\n      <td>0.724263</td>\n      <td>23167</td>\n      <td>0.724263</td>\n      <td>0.500000</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>1.000000</td>\n      <td>16779</td>\n      <td>0</td>\n      <td>6388</td>\n      <td>0</td>\n      <td>0.501815</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>2</td>\n      <td>engine_ratings</td>\n      <td>regression-derived</td>\n      <td>6512</td>\n      <td>0.724263</td>\n      <td>23167</td>\n      <td>0.724263</td>\n      <td>0.500000</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>1.000000</td>\n      <td>16779</td>\n      <td>0</td>\n      <td>6388</td>\n      <td>0</td>\n      <td>0.507454</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>2</td>\n      <td>fixed_lightgbm</td>\n      <td>regression-derived</td>\n      <td>6512</td>\n      <td>0.724263</td>\n      <td>23167</td>\n      <td>0.724263</td>\n      <td>0.500000</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>1.000000</td>\n      <td>16779</td>\n      <td>0</td>\n      <td>6388</td>\n      <td>0</td>\n      <td>0.590543</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>2</td>\n      <td>no_change</td>\n      <td>regression-derived</td>\n      <td>6512</td>\n      <td>0.724263</td>\n      <td>23167</td>\n      <td>0.724263</td>\n      <td>0.500000</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>1.000000</td>\n      <td>16779</td>\n      <td>0</td>\n      <td>6388</td>\n      <td>0</td>\n      <td>0.500000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>2</td>\n      <td>random_forest</td>\n      <td>regression-derived</td>\n      <td>6512</td>\n      <td>0.724263</td>\n      <td>23167</td>\n      <td>0.645444</td>\n      <td>0.553341</td>\n      <td>0.354432</td>\n      <td>0.347996</td>\n      <td>0.758686</td>\n      <td>12730</td>\n      <td>4049</td>\n      <td>4165</td>\n      <td>2223</td>\n      <td>0.577504</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>2</td>\n      <td>training_mean</td>\n      <td>regression-derived</td>\n      <td>6512</td>\n      <td>0.724263</td>\n      <td>23167</td>\n      <td>0.630336</td>\n      <td>0.516785</td>\n      <td>0.303752</td>\n      <td>0.263619</td>\n      <td>0.769951</td>\n      <td>12919</td>\n      <td>3860</td>\n      <td>4704</td>\n      <td>1684</td>\n      <td>0.530396</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>2</td>\n      <td>training_median</td>\n      <td>regression-derived</td>\n      <td>6512</td>\n      <td>0.724263</td>\n      <td>23167</td>\n      <td>0.724263</td>\n      <td>0.500000</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>1.000000</td>\n      <td>16779</td>\n      <td>0</td>\n      <td>6388</td>\n      <td>0</td>\n      <td>0.487642</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>4</td>\n      <td>decision_tree</td>\n      <td>regression-derived</td>\n      <td>6512</td>\n      <td>0.667242</td>\n      <td>23167</td>\n      <td>0.550265</td>\n      <td>0.537618</td>\n      <td>0.369912</td>\n      <td>0.499805</td>\n      <td>0.575430</td>\n      <td>8895</td>\n      <td>6563</td>\n      <td>3856</td>\n      <td>3853</td>\n      <td>0.538627</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>4</td>\n      <td>engine_only</td>\n      <td>regression-derived</td>\n      <td>6512</td>\n      <td>0.667242</td>\n      <td>23167</td>\n      <td>0.667070</td>\n      <td>0.500066</td>\n      <td>0.375000</td>\n      <td>0.000778</td>\n      <td>0.999353</td>\n      <td>15448</td>\n      <td>10</td>\n      <td>7703</td>\n      <td>6</td>\n      <td>0.528918</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>4</td>\n      <td>engine_ratings</td>\n      <td>regression-derived</td>\n      <td>6512</td>\n      <td>0.667242</td>\n      <td>23167</td>\n      <td>0.667113</td>\n      <td>0.500716</td>\n      <td>0.471698</td>\n      <td>0.003243</td>\n      <td>0.998189</td>\n      <td>15430</td>\n      <td>28</td>\n      <td>7684</td>\n      <td>25</td>\n      <td>0.531756</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>4</td>\n      <td>fixed_lightgbm</td>\n      <td>regression-derived</td>\n      <td>6512</td>\n      <td>0.667242</td>\n      <td>23167</td>\n      <td>0.668278</td>\n      <td>0.506206</td>\n      <td>0.538710</td>\n      <td>0.021663</td>\n      <td>0.990749</td>\n      <td>15315</td>\n      <td>143</td>\n      <td>7542</td>\n      <td>167</td>\n      <td>0.577368</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>4</td>\n      <td>no_change</td>\n      <td>regression-derived</td>\n      <td>6512</td>\n      <td>0.667242</td>\n      <td>23167</td>\n      <td>0.667242</td>\n      <td>0.500000</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>1.000000</td>\n      <td>15458</td>\n      <td>0</td>\n      <td>7709</td>\n      <td>0</td>\n      <td>0.500000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>4</td>\n      <td>random_forest</td>\n      <td>regression-derived</td>\n      <td>6512</td>\n      <td>0.667242</td>\n      <td>23167</td>\n      <td>0.556395</td>\n      <td>0.539577</td>\n      <td>0.373022</td>\n      <td>0.489298</td>\n      <td>0.589856</td>\n      <td>9118</td>\n      <td>6340</td>\n      <td>3937</td>\n      <td>3772</td>\n      <td>0.545142</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>4</td>\n      <td>training_mean</td>\n      <td>regression-derived</td>\n      <td>6512</td>\n      <td>0.667242</td>\n      <td>23167</td>\n      <td>0.525575</td>\n      <td>0.526172</td>\n      <td>0.356330</td>\n      <td>0.527954</td>\n      <td>0.524389</td>\n      <td>8106</td>\n      <td>7352</td>\n      <td>3639</td>\n      <td>4070</td>\n      <td>0.532172</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>4</td>\n      <td>training_median</td>\n      <td>regression-derived</td>\n      <td>6512</td>\n      <td>0.667242</td>\n      <td>23167</td>\n      <td>0.667242</td>\n      <td>0.500000</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>1.000000</td>\n      <td>15458</td>\n      <td>0</td>\n      <td>7709</td>\n      <td>0</td>\n      <td>0.487664</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>decision_tree</td>\n      <td>regression-derived</td>\n      <td>6512</td>\n      <td>0.641689</td>\n      <td>23167</td>\n      <td>0.490353</td>\n      <td>0.520692</td>\n      <td>0.374138</td>\n      <td>0.627756</td>\n      <td>0.413628</td>\n      <td>6149</td>\n      <td>8717</td>\n      <td>3090</td>\n      <td>5211</td>\n      <td>0.525533</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>engine_only</td>\n      <td>regression-derived</td>\n      <td>6512</td>\n      <td>0.641689</td>\n      <td>23167</td>\n      <td>0.640653</td>\n      <td>0.501081</td>\n      <td>0.427711</td>\n      <td>0.008553</td>\n      <td>0.993610</td>\n      <td>14771</td>\n      <td>95</td>\n      <td>8230</td>\n      <td>71</td>\n      <td>0.530252</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>engine_ratings</td>\n      <td>regression-derived</td>\n      <td>6512</td>\n      <td>0.641689</td>\n      <td>23167</td>\n      <td>0.641127</td>\n      <td>0.502941</td>\n      <td>0.475655</td>\n      <td>0.015299</td>\n      <td>0.990583</td>\n      <td>14726</td>\n      <td>140</td>\n      <td>8174</td>\n      <td>127</td>\n      <td>0.542922</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>fixed_lightgbm</td>\n      <td>regression-derived</td>\n      <td>6512</td>\n      <td>0.641689</td>\n      <td>23167</td>\n      <td>0.645358</td>\n      <td>0.510360</td>\n      <td>0.588727</td>\n      <td>0.033972</td>\n      <td>0.986748</td>\n      <td>14669</td>\n      <td>197</td>\n      <td>8019</td>\n      <td>282</td>\n      <td>0.574481</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>no_change</td>\n      <td>regression-derived</td>\n      <td>6512</td>\n      <td>0.641689</td>\n      <td>23167</td>\n      <td>0.641689</td>\n      <td>0.500000</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>1.000000</td>\n      <td>14866</td>\n      <td>0</td>\n      <td>8301</td>\n      <td>0</td>\n      <td>0.500000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>random_forest</td>\n      <td>regression-derived</td>\n      <td>6512</td>\n      <td>0.641689</td>\n      <td>23167</td>\n      <td>0.505201</td>\n      <td>0.519680</td>\n      <td>0.374901</td>\n      <td>0.570775</td>\n      <td>0.468586</td>\n      <td>6966</td>\n      <td>7900</td>\n      <td>3563</td>\n      <td>4738</td>\n      <td>0.529517</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>training_mean</td>\n      <td>regression-derived</td>\n      <td>6512</td>\n      <td>0.641689</td>\n      <td>23167</td>\n      <td>0.490353</td>\n      <td>0.530162</td>\n      <td>0.380260</td>\n      <td>0.670642</td>\n      <td>0.389681</td>\n      <td>5793</td>\n      <td>9073</td>\n      <td>2734</td>\n      <td>5567</td>\n      <td>0.539553</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>training_median</td>\n      <td>regression-derived</td>\n      <td>6512</td>\n      <td>0.641689</td>\n      <td>23167</td>\n      <td>0.641689</td>\n      <td>0.500000</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>1.000000</td>\n      <td>14866</td>\n      <td>0</td>\n      <td>8301</td>\n      <td>0</td>\n      <td>0.484066</td>\n    </tr>\n  </tbody>\n</table>',
'Positive class: recovered. ROC-AUC uses negative continuous predicted ratio.')

show('later_regression_derived_four_archives',
'<table border="1" class="dataframe">\n  <thead>\n    <tr style="text-align: right;">\n      <th>archive_set</th>\n      <th>protocol</th>\n      <th>horizon</th>\n      <th>model</th>\n      <th>kind</th>\n      <th>games</th>\n      <th>majority_accuracy</th>\n      <th>n</th>\n      <th>accuracy</th>\n      <th>balanced_accuracy</th>\n      <th>precision</th>\n      <th>recall</th>\n      <th>other_class_recall</th>\n      <th>tn</th>\n      <th>fp</th>\n      <th>fn</th>\n      <th>tp</th>\n      <th>roc_auc</th>\n    </tr>\n  </thead>\n  <tbody>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>2</td>\n      <td>fixed_lightgbm</td>\n      <td>regression-derived</td>\n      <td>4485</td>\n      <td>0.725130</td>\n      <td>15953</td>\n      <td>0.725130</td>\n      <td>0.500000</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>1.000000</td>\n      <td>11568</td>\n      <td>0</td>\n      <td>4385</td>\n      <td>0</td>\n      <td>0.590628</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>2</td>\n      <td>original_plus_maia</td>\n      <td>regression-derived</td>\n      <td>4485</td>\n      <td>0.725130</td>\n      <td>15953</td>\n      <td>0.738168</td>\n      <td>0.567119</td>\n      <td>0.572524</td>\n      <td>0.187229</td>\n      <td>0.947009</td>\n      <td>10955</td>\n      <td>613</td>\n      <td>3564</td>\n      <td>821</td>\n      <td>0.697473</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>2</td>\n      <td>training_majority</td>\n      <td>training majority</td>\n      <td>4485</td>\n      <td>0.725130</td>\n      <td>15953</td>\n      <td>0.725130</td>\n      <td>0.500000</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>1.000000</td>\n      <td>11568</td>\n      <td>0</td>\n      <td>4385</td>\n      <td>0</td>\n      <td>unavailable / undefined</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>4</td>\n      <td>fixed_lightgbm</td>\n      <td>regression-derived</td>\n      <td>4485</td>\n      <td>0.666207</td>\n      <td>15953</td>\n      <td>0.666207</td>\n      <td>0.505388</td>\n      <td>0.500000</td>\n      <td>0.021596</td>\n      <td>0.989180</td>\n      <td>10513</td>\n      <td>115</td>\n      <td>5210</td>\n      <td>115</td>\n      <td>0.584590</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>4</td>\n      <td>original_plus_maia</td>\n      <td>regression-derived</td>\n      <td>4485</td>\n      <td>0.666207</td>\n      <td>15953</td>\n      <td>0.678932</td>\n      <td>0.558885</td>\n      <td>0.553337</td>\n      <td>0.197746</td>\n      <td>0.920023</td>\n      <td>9778</td>\n      <td>850</td>\n      <td>4272</td>\n      <td>1053</td>\n      <td>0.647609</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>4</td>\n      <td>training_majority</td>\n      <td>training majority</td>\n      <td>4485</td>\n      <td>0.666207</td>\n      <td>15953</td>\n      <td>0.666207</td>\n      <td>0.500000</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>1.000000</td>\n      <td>10628</td>\n      <td>0</td>\n      <td>5325</td>\n      <td>0</td>\n      <td>unavailable / undefined</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>fixed_lightgbm</td>\n      <td>regression-derived</td>\n      <td>4485</td>\n      <td>0.643891</td>\n      <td>15953</td>\n      <td>0.647841</td>\n      <td>0.512429</td>\n      <td>0.576271</td>\n      <td>0.041894</td>\n      <td>0.982963</td>\n      <td>10097</td>\n      <td>175</td>\n      <td>5443</td>\n      <td>238</td>\n      <td>0.585757</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>original_plus_maia</td>\n      <td>regression-derived</td>\n      <td>4485</td>\n      <td>0.643891</td>\n      <td>15953</td>\n      <td>0.660377</td>\n      <td>0.557764</td>\n      <td>0.565002</td>\n      <td>0.201197</td>\n      <td>0.914330</td>\n      <td>9392</td>\n      <td>880</td>\n      <td>4538</td>\n      <td>1143</td>\n      <td>0.636537</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>training_majority</td>\n      <td>training majority</td>\n      <td>4485</td>\n      <td>0.643891</td>\n      <td>15953</td>\n      <td>0.643891</td>\n      <td>0.500000</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>1.000000</td>\n      <td>10272</td>\n      <td>0</td>\n      <td>5681</td>\n      <td>0</td>\n      <td>unavailable / undefined</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>10</td>\n      <td>fixed_lightgbm</td>\n      <td>regression-derived</td>\n      <td>4485</td>\n      <td>0.617564</td>\n      <td>15953</td>\n      <td>0.629725</td>\n      <td>0.528349</td>\n      <td>0.597782</td>\n      <td>0.097197</td>\n      <td>0.959501</td>\n      <td>9453</td>\n      <td>399</td>\n      <td>5508</td>\n      <td>593</td>\n      <td>0.581283</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>10</td>\n      <td>original_plus_maia</td>\n      <td>regression-derived</td>\n      <td>4485</td>\n      <td>0.617564</td>\n      <td>15953</td>\n      <td>0.640444</td>\n      <td>0.559431</td>\n      <td>0.580860</td>\n      <td>0.214883</td>\n      <td>0.903979</td>\n      <td>8906</td>\n      <td>946</td>\n      <td>4790</td>\n      <td>1311</td>\n      <td>0.614539</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>10</td>\n      <td>training_majority</td>\n      <td>training majority</td>\n      <td>4485</td>\n      <td>0.617564</td>\n      <td>15953</td>\n      <td>0.617564</td>\n      <td>0.500000</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>1.000000</td>\n      <td>9852</td>\n      <td>0</td>\n      <td>6101</td>\n      <td>0</td>\n      <td>unavailable / undefined</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>2</td>\n      <td>fixed_lightgbm</td>\n      <td>regression-derived</td>\n      <td>4485</td>\n      <td>0.725130</td>\n      <td>15953</td>\n      <td>0.725130</td>\n      <td>0.500000</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>1.000000</td>\n      <td>11568</td>\n      <td>0</td>\n      <td>4385</td>\n      <td>0</td>\n      <td>0.587507</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>2</td>\n      <td>original_plus_maia</td>\n      <td>regression-derived</td>\n      <td>4485</td>\n      <td>0.725130</td>\n      <td>15953</td>\n      <td>0.738983</td>\n      <td>0.568743</td>\n      <td>0.576154</td>\n      <td>0.190650</td>\n      <td>0.946836</td>\n      <td>10953</td>\n      <td>615</td>\n      <td>3549</td>\n      <td>836</td>\n      <td>0.696262</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>2</td>\n      <td>training_majority</td>\n      <td>training majority</td>\n      <td>4485</td>\n      <td>0.725130</td>\n      <td>15953</td>\n      <td>0.725130</td>\n      <td>0.500000</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>1.000000</td>\n      <td>11568</td>\n      <td>0</td>\n      <td>4385</td>\n      <td>0</td>\n      <td>unavailable / undefined</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>4</td>\n      <td>fixed_lightgbm</td>\n      <td>regression-derived</td>\n      <td>4485</td>\n      <td>0.666207</td>\n      <td>15953</td>\n      <td>0.665517</td>\n      <td>0.503418</td>\n      <td>0.469274</td>\n      <td>0.015775</td>\n      <td>0.991061</td>\n      <td>10533</td>\n      <td>95</td>\n      <td>5241</td>\n      <td>84</td>\n      <td>0.578726</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>4</td>\n      <td>original_plus_maia</td>\n      <td>regression-derived</td>\n      <td>4485</td>\n      <td>0.666207</td>\n      <td>15953</td>\n      <td>0.680436</td>\n      <td>0.557531</td>\n      <td>0.564016</td>\n      <td>0.187793</td>\n      <td>0.927268</td>\n      <td>9855</td>\n      <td>773</td>\n      <td>4325</td>\n      <td>1000</td>\n      <td>0.642069</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>4</td>\n      <td>training_majority</td>\n      <td>training majority</td>\n      <td>4485</td>\n      <td>0.666207</td>\n      <td>15953</td>\n      <td>0.666207</td>\n      <td>0.500000</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>1.000000</td>\n      <td>10628</td>\n      <td>0</td>\n      <td>5325</td>\n      <td>0</td>\n      <td>unavailable / undefined</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>fixed_lightgbm</td>\n      <td>regression-derived</td>\n      <td>4485</td>\n      <td>0.643891</td>\n      <td>15953</td>\n      <td>0.647778</td>\n      <td>0.513639</td>\n      <td>0.564854</td>\n      <td>0.047527</td>\n      <td>0.979751</td>\n      <td>10064</td>\n      <td>208</td>\n      <td>5411</td>\n      <td>270</td>\n      <td>0.579641</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>original_plus_maia</td>\n      <td>regression-derived</td>\n      <td>4485</td>\n      <td>0.643891</td>\n      <td>15953</td>\n      <td>0.660001</td>\n      <td>0.557904</td>\n      <td>0.562652</td>\n      <td>0.203133</td>\n      <td>0.912675</td>\n      <td>9375</td>\n      <td>897</td>\n      <td>4527</td>\n      <td>1154</td>\n      <td>0.630438</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>training_majority</td>\n      <td>training majority</td>\n      <td>4485</td>\n      <td>0.643891</td>\n      <td>15953</td>\n      <td>0.643891</td>\n      <td>0.500000</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>1.000000</td>\n      <td>10272</td>\n      <td>0</td>\n      <td>5681</td>\n      <td>0</td>\n      <td>unavailable / undefined</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>10</td>\n      <td>fixed_lightgbm</td>\n      <td>regression-derived</td>\n      <td>4485</td>\n      <td>0.617564</td>\n      <td>15953</td>\n      <td>0.627594</td>\n      <td>0.526249</td>\n      <td>0.579840</td>\n      <td>0.095230</td>\n      <td>0.957268</td>\n      <td>9431</td>\n      <td>421</td>\n      <td>5520</td>\n      <td>581</td>\n      <td>0.577812</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>10</td>\n      <td>original_plus_maia</td>\n      <td>regression-derived</td>\n      <td>4485</td>\n      <td>0.617564</td>\n      <td>15953</td>\n      <td>0.637059</td>\n      <td>0.557283</td>\n      <td>0.566198</td>\n      <td>0.217997</td>\n      <td>0.896569</td>\n      <td>8833</td>\n      <td>1019</td>\n      <td>4771</td>\n      <td>1330</td>\n      <td>0.607163</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>10</td>\n      <td>training_majority</td>\n      <td>training majority</td>\n      <td>4485</td>\n      <td>0.617564</td>\n      <td>15953</td>\n      <td>0.617564</td>\n      <td>0.500000</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>1.000000</td>\n      <td>9852</td>\n      <td>0</td>\n      <td>6101</td>\n      <td>0</td>\n      <td>unavailable / undefined</td>\n    </tr>\n  </tbody>\n</table>',
'Positive class: recovered. Fixed ratio boundary 0.75; continuous-score ROC-AUC.')


**Later regression derived five archives**

Positive class: recovered. ROC-AUC uses negative continuous predicted ratio.

archive_set,protocol,horizon,model,kind,games,majority_accuracy,n,accuracy,balanced_accuracy,precision,recall,other_class_recall,tn,fp,fn,tp,roc_auc
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,10,decision_tree,regression-derived,6512,0.617041,23167,0.461001,0.501922,0.384305,0.676736,0.327107,4676,9619,2868,6004,0.523015
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,10,engine_only,regression-derived,6512,0.617041,23167,0.614408,0.501864,0.429885,0.021078,0.982651,14047,248,8685,187,0.526795
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,10,engine_ratings,regression-derived,6512,0.617041,23167,0.619416,0.512464,0.529538,0.055568,0.969360,13857,438,8379,493,0.549214
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,10,fixed_lightgbm,regression-derived,6512,0.617041,23167,0.629128,0.525016,0.622378,0.080252,0.969780,13863,432,8160,712,0.582195
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,10,no_change,regression-derived,6512,0.617041,23167,0.617041,0.500000,unavailable / undefined,0.000000,1.000000,14295,0,8872,0,0.500000
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,10,random_forest,regression-derived,6512,0.617041,23167,0.484137,0.515282,0.394432,0.648332,0.382232,5464,8831,3120,5752,0.532677
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,10,training_mean,regression-derived,6512,0.617041,23167,0.454828,0.522383,0.396462,0.810978,0.233788,3342,10953,1677,7195,0.541111
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,10,training_median,regression-derived,6512,0.617041,23167,0.617041,0.500000,unavailable / undefined,0.000000,1.000000,14295,0,8872,0,0.464303
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,2,decision_tree,regression-derived,6512,0.724263,23167,0.637329,0.540759,0.336844,0.325454,0.756064,12686,4093,4309,2079,0.562099
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,2,engine_only,regression-derived,6512,0.724263,23167,0.724263,0.500000,unavailable / undefined,0.000000,1.000000,16779,0,6388,0,0.497877


**Later regression derived four archives**

Positive class: recovered. Fixed ratio boundary 0.75; continuous-score ROC-AUC.

archive_set,protocol,horizon,model,kind,games,majority_accuracy,n,accuracy,balanced_accuracy,precision,recall,other_class_recall,tn,fp,fn,tp,roc_auc
"2017-04, 2017-06, 2017-07, 2018-02",game,2,fixed_lightgbm,regression-derived,4485,0.725130,15953,0.725130,0.500000,unavailable / undefined,0.000000,1.000000,11568,0,4385,0,0.590628
"2017-04, 2017-06, 2017-07, 2018-02",game,2,original_plus_maia,regression-derived,4485,0.725130,15953,0.738168,0.567119,0.572524,0.187229,0.947009,10955,613,3564,821,0.697473
"2017-04, 2017-06, 2017-07, 2018-02",game,2,training_majority,training majority,4485,0.725130,15953,0.725130,0.500000,unavailable / undefined,0.000000,1.000000,11568,0,4385,0,unavailable / undefined
"2017-04, 2017-06, 2017-07, 2018-02",game,4,fixed_lightgbm,regression-derived,4485,0.666207,15953,0.666207,0.505388,0.500000,0.021596,0.989180,10513,115,5210,115,0.584590
"2017-04, 2017-06, 2017-07, 2018-02",game,4,original_plus_maia,regression-derived,4485,0.666207,15953,0.678932,0.558885,0.553337,0.197746,0.920023,9778,850,4272,1053,0.647609
"2017-04, 2017-06, 2017-07, 2018-02",game,4,training_majority,training majority,4485,0.666207,15953,0.666207,0.500000,unavailable / undefined,0.000000,1.000000,10628,0,5325,0,unavailable / undefined
"2017-04, 2017-06, 2017-07, 2018-02",game,6,fixed_lightgbm,regression-derived,4485,0.643891,15953,0.647841,0.512429,0.576271,0.041894,0.982963,10097,175,5443,238,0.585757
"2017-04, 2017-06, 2017-07, 2018-02",game,6,original_plus_maia,regression-derived,4485,0.643891,15953,0.660377,0.557764,0.565002,0.201197,0.914330,9392,880,4538,1143,0.636537
"2017-04, 2017-06, 2017-07, 2018-02",game,6,training_majority,training majority,4485,0.643891,15953,0.643891,0.500000,unavailable / undefined,0.000000,1.000000,10272,0,5681,0,unavailable / undefined
"2017-04, 2017-06, 2017-07, 2018-02",game,10,fixed_lightgbm,regression-derived,4485,0.617564,15953,0.629725,0.528349,0.597782,0.097197,0.959501,9453,399,5508,593,0.581283


### Recovered-only MAE

This diagnostic restricts evaluation to mistakes that actually recovered. Membership is determined by the outcome and is unavailable at prediction time. Full-sample MAE remains the primary regression metric.


In [10]:
show('recovered_only_mae_diagnostic',
'<table border="1" class="dataframe">\n  <thead>\n    <tr style="text-align: right;">\n      <th>archive_set</th>\n      <th>protocol</th>\n      <th>target</th>\n      <th>model</th>\n      <th>recovered_n</th>\n      <th>recovered_only_mae_diagnostic</th>\n    </tr>\n  </thead>\n  <tbody>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>2</td>\n      <td>fixed_lightgbm</td>\n      <td>6388.000000</td>\n      <td>32.582466</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>2</td>\n      <td>no_change</td>\n      <td>6388.000000</td>\n      <td>32.711581</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>4</td>\n      <td>fixed_lightgbm</td>\n      <td>7709.000000</td>\n      <td>35.222624</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>4</td>\n      <td>no_change</td>\n      <td>7709.000000</td>\n      <td>36.277953</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>fixed_lightgbm</td>\n      <td>8301.000000</td>\n      <td>37.749732</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>no_change</td>\n      <td>8301.000000</td>\n      <td>39.524744</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>10</td>\n      <td>fixed_lightgbm</td>\n      <td>8872.000000</td>\n      <td>41.096974</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>10</td>\n      <td>no_change</td>\n      <td>8872.000000</td>\n      <td>44.018921</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>2</td>\n      <td>fixed_lightgbm</td>\n      <td>6388.000000</td>\n      <td>32.588178</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>2</td>\n      <td>no_change</td>\n      <td>6388.000000</td>\n      <td>32.711581</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>4</td>\n      <td>fixed_lightgbm</td>\n      <td>7709.000000</td>\n      <td>35.228577</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>4</td>\n      <td>no_change</td>\n      <td>7709.000000</td>\n      <td>36.277953</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>fixed_lightgbm</td>\n      <td>8301.000000</td>\n      <td>37.683545</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>no_change</td>\n      <td>8301.000000</td>\n      <td>39.524744</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>10</td>\n      <td>fixed_lightgbm</td>\n      <td>8872.000000</td>\n      <td>41.059256</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>10</td>\n      <td>no_change</td>\n      <td>8872.000000</td>\n      <td>44.018921</td>\n    </tr>\n  </tbody>\n</table>',
'Outcome-defined diagnostic subset.')


**Recovered only mae diagnostic**

Outcome-defined diagnostic subset.

archive_set,protocol,target,model,recovered_n,recovered_only_mae_diagnostic
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,2,fixed_lightgbm,6388.000000,32.582466
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,2,no_change,6388.000000,32.711581
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,4,fixed_lightgbm,7709.000000,35.222624
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,4,no_change,7709.000000,36.277953
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,6,fixed_lightgbm,8301.000000,37.749732
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,6,no_change,8301.000000,39.524744
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,10,fixed_lightgbm,8872.000000,41.096974
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,10,no_change,8872.000000,44.018921
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",player,2,fixed_lightgbm,6388.000000,32.588178
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",player,2,no_change,6388.000000,32.711581


## 4. Supplementary development analyses

These analyses document sensitivity to models, features and thresholds. They reuse development data and are not independent replications or additions to the final evaluation scope.

### Immediate-reply logistic regression

The logistic models use the same damage/Elo controls, with or without Maia, and the fixed 0.5 cutoff.


In [11]:
show('immediate_reply_logistic_sensitivity',
'<table border="1" class="dataframe">\n  <thead>\n    <tr style="text-align: right;">\n      <th>archive_set</th>\n      <th>protocol</th>\n      <th>method</th>\n      <th>games</th>\n      <th>n</th>\n      <th>accuracy</th>\n      <th>balanced_accuracy</th>\n      <th>precision</th>\n      <th>preserving_recall</th>\n      <th>non_preserving_recall</th>\n      <th>tn</th>\n      <th>fp</th>\n      <th>fn</th>\n      <th>tp</th>\n      <th>roc_auc</th>\n      <th>brier</th>\n    </tr>\n  </thead>\n  <tbody>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>simple_logistic</td>\n      <td>4472</td>\n      <td>15851</td>\n      <td>0.566337</td>\n      <td>0.520151</td>\n      <td>0.569184</td>\n      <td>0.917373</td>\n      <td>0.122930</td>\n      <td>861</td>\n      <td>6143</td>\n      <td>731</td>\n      <td>8116</td>\n      <td>0.563848</td>\n      <td>0.243910</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>simple_plus_maia_logistic</td>\n      <td>4472</td>\n      <td>15851</td>\n      <td>0.730427</td>\n      <td>0.721907</td>\n      <td>0.740838</td>\n      <td>0.795185</td>\n      <td>0.648629</td>\n      <td>4543</td>\n      <td>2461</td>\n      <td>1812</td>\n      <td>7035</td>\n      <td>0.799638</td>\n      <td>0.180654</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>simple_logistic</td>\n      <td>4472</td>\n      <td>15851</td>\n      <td>0.566021</td>\n      <td>0.518649</td>\n      <td>0.568248</td>\n      <td>0.926077</td>\n      <td>0.111222</td>\n      <td>779</td>\n      <td>6225</td>\n      <td>654</td>\n      <td>8193</td>\n      <td>0.563731</td>\n      <td>0.243947</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>simple_plus_maia_logistic</td>\n      <td>4472</td>\n      <td>15851</td>\n      <td>0.730869</td>\n      <td>0.721113</td>\n      <td>0.737038</td>\n      <td>0.805019</td>\n      <td>0.637207</td>\n      <td>4463</td>\n      <td>2541</td>\n      <td>1725</td>\n      <td>7122</td>\n      <td>0.799360</td>\n      <td>0.180780</td>\n    </tr>\n  </tbody>\n</table>',
'')


**Immediate reply logistic sensitivity**

archive_set,protocol,method,games,n,accuracy,balanced_accuracy,precision,preserving_recall,non_preserving_recall,tn,fp,fn,tp,roc_auc,brier
"2017-04, 2017-06, 2017-07, 2018-02",game,simple_logistic,4472,15851,0.566337,0.520151,0.569184,0.917373,0.122930,861,6143,731,8116,0.563848,0.243910
"2017-04, 2017-06, 2017-07, 2018-02",game,simple_plus_maia_logistic,4472,15851,0.730427,0.721907,0.740838,0.795185,0.648629,4543,2461,1812,7035,0.799638,0.180654
"2017-04, 2017-06, 2017-07, 2018-02",player,simple_logistic,4472,15851,0.566021,0.518649,0.568248,0.926077,0.111222,779,6225,654,8193,0.563731,0.243947
"2017-04, 2017-06, 2017-07, 2018-02",player,simple_plus_maia_logistic,4472,15851,0.730869,0.721113,0.737038,0.805019,0.637207,4463,2541,1725,7122,0.799360,0.180780


### Later-horizon classifiers and cutoff strategies

The direct classifiers and inner-selected thresholds are separate from the damage regressors. Their precision, recall and accuracy show the tradeoffs associated with different decision rules. These experiments remain development-only.


In [12]:
show('direct_classifiers_five_archives',
'<table border="1" class="dataframe">\n  <thead>\n    <tr style="text-align: right;">\n      <th>archive_set</th>\n      <th>protocol</th>\n      <th>target</th>\n      <th>model</th>\n      <th>strategy</th>\n      <th>n</th>\n      <th>games</th>\n      <th>accuracy</th>\n      <th>majority_baseline_accuracy</th>\n      <th>balanced_accuracy</th>\n      <th>roc_auc</th>\n      <th>recovered_precision</th>\n      <th>recovered_recall</th>\n      <th>tn</th>\n      <th>fp</th>\n      <th>fn</th>\n      <th>tp</th>\n    </tr>\n  </thead>\n  <tbody>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>decision_tree</td>\n      <td>default_0.5</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>0.643933</td>\n      <td>0.641689</td>\n      <td>0.506883</td>\n      <td>0.577693</td>\n      <td>0.577844</td>\n      <td>0.023250</td>\n      <td>14725.000000</td>\n      <td>141.000000</td>\n      <td>8108.000000</td>\n      <td>193.000000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>decision_tree</td>\n      <td>inner_balanced_accuracy</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>0.528726</td>\n      <td>0.641689</td>\n      <td>0.555221</td>\n      <td>0.577693</td>\n      <td>0.402256</td>\n      <td>0.648717</td>\n      <td>6864.000000</td>\n      <td>8002.000000</td>\n      <td>2916.000000</td>\n      <td>5385.000000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>engine_only</td>\n      <td>default_0.5</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>0.640092</td>\n      <td>0.641689</td>\n      <td>0.501469</td>\n      <td>0.577342</td>\n      <td>0.423237</td>\n      <td>0.012288</td>\n      <td>14727.000000</td>\n      <td>139.000000</td>\n      <td>8199.000000</td>\n      <td>102.000000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>engine_only</td>\n      <td>inner_balanced_accuracy</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>0.507316</td>\n      <td>0.641689</td>\n      <td>0.553275</td>\n      <td>0.577342</td>\n      <td>0.396171</td>\n      <td>0.715456</td>\n      <td>5814.000000</td>\n      <td>9052.000000</td>\n      <td>2362.000000</td>\n      <td>5939.000000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>engine_ratings</td>\n      <td>default_0.5</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>0.638538</td>\n      <td>0.641689</td>\n      <td>0.506456</td>\n      <td>0.583407</td>\n      <td>0.450875</td>\n      <td>0.040357</td>\n      <td>14458.000000</td>\n      <td>408.000000</td>\n      <td>7966.000000</td>\n      <td>335.000000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>engine_ratings</td>\n      <td>inner_balanced_accuracy</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>0.544136</td>\n      <td>0.641689</td>\n      <td>0.558264</td>\n      <td>0.583407</td>\n      <td>0.408546</td>\n      <td>0.608120</td>\n      <td>7558.000000</td>\n      <td>7308.000000</td>\n      <td>3253.000000</td>\n      <td>5048.000000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>fixed_lightgbm</td>\n      <td>default_0.5</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>0.646523</td>\n      <td>0.641689</td>\n      <td>0.523744</td>\n      <td>0.617453</td>\n      <td>0.540288</td>\n      <td>0.090471</td>\n      <td>14227.000000</td>\n      <td>639.000000</td>\n      <td>7550.000000</td>\n      <td>751.000000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>fixed_lightgbm</td>\n      <td>inner_balanced_accuracy</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>0.580567</td>\n      <td>0.641689</td>\n      <td>0.582156</td>\n      <td>0.617453</td>\n      <td>0.436639</td>\n      <td>0.587761</td>\n      <td>8571.000000</td>\n      <td>6295.000000</td>\n      <td>3422.000000</td>\n      <td>4879.000000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>optuna_lightgbm</td>\n      <td>default_0.5</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>0.646005</td>\n      <td>0.641689</td>\n      <td>0.524085</td>\n      <td>0.618699</td>\n      <td>0.534294</td>\n      <td>0.093844</td>\n      <td>14187.000000</td>\n      <td>679.000000</td>\n      <td>7522.000000</td>\n      <td>779.000000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>optuna_lightgbm</td>\n      <td>inner_balanced_accuracy</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>0.583718</td>\n      <td>0.641689</td>\n      <td>0.581605</td>\n      <td>0.618699</td>\n      <td>0.438253</td>\n      <td>0.574148</td>\n      <td>8757.000000</td>\n      <td>6109.000000</td>\n      <td>3535.000000</td>\n      <td>4766.000000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>optuna_xgboost</td>\n      <td>default_0.5</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>0.646091</td>\n      <td>0.641689</td>\n      <td>0.517555</td>\n      <td>0.616463</td>\n      <td>0.553125</td>\n      <td>0.063968</td>\n      <td>14437.000000</td>\n      <td>429.000000</td>\n      <td>7770.000000</td>\n      <td>531.000000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>optuna_xgboost</td>\n      <td>inner_balanced_accuracy</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>0.563862</td>\n      <td>0.641689</td>\n      <td>0.580896</td>\n      <td>0.616463</td>\n      <td>0.427561</td>\n      <td>0.641007</td>\n      <td>7742.000000</td>\n      <td>7124.000000</td>\n      <td>2980.000000</td>\n      <td>5321.000000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>random_forest</td>\n      <td>default_0.5</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>0.645573</td>\n      <td>0.641689</td>\n      <td>0.520317</td>\n      <td>0.610379</td>\n      <td>0.537190</td>\n      <td>0.078304</td>\n      <td>14306.000000</td>\n      <td>560.000000</td>\n      <td>7651.000000</td>\n      <td>650.000000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>random_forest</td>\n      <td>inner_balanced_accuracy</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>0.559201</td>\n      <td>0.641689</td>\n      <td>0.575907</td>\n      <td>0.610379</td>\n      <td>0.423259</td>\n      <td>0.634863</td>\n      <td>7685.000000</td>\n      <td>7181.000000</td>\n      <td>3031.000000</td>\n      <td>5270.000000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>training_majority</td>\n      <td>majority</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>0.641689</td>\n      <td>0.641689</td>\n      <td>0.500000</td>\n      <td>unavailable / undefined</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>14866.000000</td>\n      <td>0.000000</td>\n      <td>8301.000000</td>\n      <td>0.000000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>decision_tree</td>\n      <td>default_0.5</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>0.638322</td>\n      <td>0.641689</td>\n      <td>0.500781</td>\n      <td>0.573906</td>\n      <td>0.383234</td>\n      <td>0.015420</td>\n      <td>14660.000000</td>\n      <td>206.000000</td>\n      <td>8173.000000</td>\n      <td>128.000000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>decision_tree</td>\n      <td>inner_balanced_accuracy</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>0.545517</td>\n      <td>0.641689</td>\n      <td>0.558010</td>\n      <td>0.573906</td>\n      <td>0.408868</td>\n      <td>0.602096</td>\n      <td>7640.000000</td>\n      <td>7226.000000</td>\n      <td>3303.000000</td>\n      <td>4998.000000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>engine_only</td>\n      <td>default_0.5</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>0.640437</td>\n      <td>0.641689</td>\n      <td>0.503148</td>\n      <td>0.575687</td>\n      <td>0.457227</td>\n      <td>0.018672</td>\n      <td>14682.000000</td>\n      <td>184.000000</td>\n      <td>8146.000000</td>\n      <td>155.000000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>engine_only</td>\n      <td>inner_balanced_accuracy</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>0.538697</td>\n      <td>0.641689</td>\n      <td>0.554718</td>\n      <td>0.575687</td>\n      <td>0.404819</td>\n      <td>0.611252</td>\n      <td>7406.000000</td>\n      <td>7460.000000</td>\n      <td>3227.000000</td>\n      <td>5074.000000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>engine_ratings</td>\n      <td>default_0.5</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>0.636725</td>\n      <td>0.641689</td>\n      <td>0.506985</td>\n      <td>0.580362</td>\n      <td>0.438238</td>\n      <td>0.049151</td>\n      <td>14343.000000</td>\n      <td>523.000000</td>\n      <td>7893.000000</td>\n      <td>408.000000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>engine_ratings</td>\n      <td>inner_balanced_accuracy</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>0.534338</td>\n      <td>0.641689</td>\n      <td>0.559221</td>\n      <td>0.580362</td>\n      <td>0.406002</td>\n      <td>0.647030</td>\n      <td>7008.000000</td>\n      <td>7858.000000</td>\n      <td>2930.000000</td>\n      <td>5371.000000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>fixed_lightgbm</td>\n      <td>default_0.5</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>0.646091</td>\n      <td>0.641689</td>\n      <td>0.526440</td>\n      <td>0.614837</td>\n      <td>0.531327</td>\n      <td>0.104204</td>\n      <td>14103.000000</td>\n      <td>763.000000</td>\n      <td>7436.000000</td>\n      <td>865.000000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>fixed_lightgbm</td>\n      <td>inner_balanced_accuracy</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>0.570164</td>\n      <td>0.641689</td>\n      <td>0.578465</td>\n      <td>0.614837</td>\n      <td>0.429471</td>\n      <td>0.607758</td>\n      <td>8164.000000</td>\n      <td>6702.000000</td>\n      <td>3256.000000</td>\n      <td>5045.000000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>optuna_lightgbm</td>\n      <td>default_0.5</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>0.647602</td>\n      <td>0.641689</td>\n      <td>0.518759</td>\n      <td>0.615589</td>\n      <td>0.573894</td>\n      <td>0.064089</td>\n      <td>14471.000000</td>\n      <td>395.000000</td>\n      <td>7769.000000</td>\n      <td>532.000000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>optuna_lightgbm</td>\n      <td>inner_balanced_accuracy</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>0.561316</td>\n      <td>0.641689</td>\n      <td>0.582503</td>\n      <td>0.615589</td>\n      <td>0.427118</td>\n      <td>0.657270</td>\n      <td>7548.000000</td>\n      <td>7318.000000</td>\n      <td>2845.000000</td>\n      <td>5456.000000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>optuna_xgboost</td>\n      <td>default_0.5</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>0.646609</td>\n      <td>0.641689</td>\n      <td>0.513783</td>\n      <td>0.613403</td>\n      <td>0.589905</td>\n      <td>0.045055</td>\n      <td>14606.000000</td>\n      <td>260.000000</td>\n      <td>7927.000000</td>\n      <td>374.000000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>optuna_xgboost</td>\n      <td>inner_balanced_accuracy</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>0.563129</td>\n      <td>0.641689</td>\n      <td>0.578622</td>\n      <td>0.613403</td>\n      <td>0.426220</td>\n      <td>0.633297</td>\n      <td>7789.000000</td>\n      <td>7077.000000</td>\n      <td>3044.000000</td>\n      <td>5257.000000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>random_forest</td>\n      <td>default_0.5</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>0.642552</td>\n      <td>0.641689</td>\n      <td>0.515994</td>\n      <td>0.605662</td>\n      <td>0.508834</td>\n      <td>0.069389</td>\n      <td>14310.000000</td>\n      <td>556.000000</td>\n      <td>7725.000000</td>\n      <td>576.000000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>random_forest</td>\n      <td>inner_balanced_accuracy</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>0.566280</td>\n      <td>0.641689</td>\n      <td>0.572911</td>\n      <td>0.605662</td>\n      <td>0.425002</td>\n      <td>0.596314</td>\n      <td>8169.000000</td>\n      <td>6697.000000</td>\n      <td>3351.000000</td>\n      <td>4950.000000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>training_majority</td>\n      <td>majority</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>0.641689</td>\n      <td>0.641689</td>\n      <td>0.500000</td>\n      <td>unavailable / undefined</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>14866.000000</td>\n      <td>0.000000</td>\n      <td>8301.000000</td>\n      <td>0.000000</td>\n    </tr>\n  </tbody>\n</table>',
'Positive class: recovered. Default and inner-selected cutoffs are separate strategies.')

show('focused_direct_and_threshold_strategies_four_archives',
'<table border="1" class="dataframe">\n  <thead>\n    <tr style="text-align: right;">\n      <th>archive_set</th>\n      <th>protocol</th>\n      <th>horizon</th>\n      <th>strategy</th>\n      <th>n</th>\n      <th>games</th>\n      <th>accuracy</th>\n      <th>majority_accuracy</th>\n      <th>balanced_accuracy</th>\n      <th>roc_auc</th>\n      <th>recovered_precision</th>\n      <th>recovered_recall</th>\n      <th>preserving_recall</th>\n      <th>tn</th>\n      <th>fp</th>\n      <th>fn</th>\n      <th>tp</th>\n    </tr>\n  </thead>\n  <tbody>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>classifier_inner_accuracy</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>0.662383</td>\n      <td>0.643891</td>\n      <td>0.564042</td>\n      <td>0.661863</td>\n      <td>0.566114</td>\n      <td>0.222320</td>\n      <td>0.905763</td>\n      <td>9304</td>\n      <td>968</td>\n      <td>4418</td>\n      <td>1263</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>classifier_inner_balanced</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>0.597944</td>\n      <td>0.643891</td>\n      <td>0.615491</td>\n      <td>0.661863</td>\n      <td>0.456468</td>\n      <td>0.676465</td>\n      <td>0.554517</td>\n      <td>5696</td>\n      <td>4576</td>\n      <td>1838</td>\n      <td>3843</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>fixed_classifier_0.5</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>0.665079</td>\n      <td>0.643891</td>\n      <td>0.573137</td>\n      <td>0.663828</td>\n      <td>0.566431</td>\n      <td>0.253653</td>\n      <td>0.892621</td>\n      <td>9169</td>\n      <td>1103</td>\n      <td>4240</td>\n      <td>1441</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>regressor_fixed_0.75</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>0.660377</td>\n      <td>0.643891</td>\n      <td>0.557764</td>\n      <td>0.636537</td>\n      <td>0.565002</td>\n      <td>0.201197</td>\n      <td>0.914330</td>\n      <td>9392</td>\n      <td>880</td>\n      <td>4538</td>\n      <td>1143</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>regressor_inner_accuracy</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>0.660942</td>\n      <td>0.643891</td>\n      <td>0.556746</td>\n      <td>0.636537</td>\n      <td>0.570103</td>\n      <td>0.194684</td>\n      <td>0.918808</td>\n      <td>9438</td>\n      <td>834</td>\n      <td>4575</td>\n      <td>1106</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>regressor_inner_balanced</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>0.620385</td>\n      <td>0.643891</td>\n      <td>0.608529</td>\n      <td>0.636537</td>\n      <td>0.472511</td>\n      <td>0.567330</td>\n      <td>0.649727</td>\n      <td>6674</td>\n      <td>3598</td>\n      <td>2458</td>\n      <td>3223</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>selected_classifier_0.5</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>0.607848</td>\n      <td>0.643891</td>\n      <td>0.617360</td>\n      <td>0.661863</td>\n      <td>0.463905</td>\n      <td>0.650414</td>\n      <td>0.584307</td>\n      <td>6002</td>\n      <td>4270</td>\n      <td>1986</td>\n      <td>3695</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>training_majority</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>0.643891</td>\n      <td>0.643891</td>\n      <td>0.500000</td>\n      <td>0.495480</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>1.000000</td>\n      <td>10272</td>\n      <td>0</td>\n      <td>5681</td>\n      <td>0</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>10</td>\n      <td>classifier_inner_accuracy</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>0.641008</td>\n      <td>0.617564</td>\n      <td>0.561635</td>\n      <td>0.638426</td>\n      <td>0.579237</td>\n      <td>0.224062</td>\n      <td>0.899208</td>\n      <td>8859</td>\n      <td>993</td>\n      <td>4734</td>\n      <td>1367</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>10</td>\n      <td>classifier_inner_balanced</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>0.580392</td>\n      <td>0.617564</td>\n      <td>0.596587</td>\n      <td>0.638426</td>\n      <td>0.465970</td>\n      <td>0.665465</td>\n      <td>0.527710</td>\n      <td>5199</td>\n      <td>4653</td>\n      <td>2041</td>\n      <td>4060</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>10</td>\n      <td>fixed_classifier_0.5</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>0.639880</td>\n      <td>0.617564</td>\n      <td>0.565215</td>\n      <td>0.646278</td>\n      <td>0.566767</td>\n      <td>0.247664</td>\n      <td>0.882765</td>\n      <td>8697</td>\n      <td>1155</td>\n      <td>4590</td>\n      <td>1511</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>10</td>\n      <td>regressor_fixed_0.75</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>0.640444</td>\n      <td>0.617564</td>\n      <td>0.559431</td>\n      <td>0.614539</td>\n      <td>0.580860</td>\n      <td>0.214883</td>\n      <td>0.903979</td>\n      <td>8906</td>\n      <td>946</td>\n      <td>4790</td>\n      <td>1311</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>10</td>\n      <td>regressor_inner_accuracy</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>0.638187</td>\n      <td>0.617564</td>\n      <td>0.548867</td>\n      <td>0.614539</td>\n      <td>0.594922</td>\n      <td>0.168989</td>\n      <td>0.928745</td>\n      <td>9150</td>\n      <td>702</td>\n      <td>5070</td>\n      <td>1031</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>10</td>\n      <td>regressor_inner_balanced</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>0.613302</td>\n      <td>0.617564</td>\n      <td>0.587286</td>\n      <td>0.614539</td>\n      <td>0.494222</td>\n      <td>0.476643</td>\n      <td>0.697929</td>\n      <td>6876</td>\n      <td>2976</td>\n      <td>3193</td>\n      <td>2908</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>10</td>\n      <td>selected_classifier_0.5</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>0.610418</td>\n      <td>0.617564</td>\n      <td>0.590225</td>\n      <td>0.638426</td>\n      <td>0.490906</td>\n      <td>0.504344</td>\n      <td>0.676106</td>\n      <td>6661</td>\n      <td>3191</td>\n      <td>3024</td>\n      <td>3077</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>10</td>\n      <td>training_majority</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>0.617564</td>\n      <td>0.617564</td>\n      <td>0.500000</td>\n      <td>0.496547</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>1.000000</td>\n      <td>9852</td>\n      <td>0</td>\n      <td>6101</td>\n      <td>0</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>classifier_inner_accuracy</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>0.662258</td>\n      <td>0.643891</td>\n      <td>0.555880</td>\n      <td>0.642199</td>\n      <td>0.580362</td>\n      <td>0.186235</td>\n      <td>0.925526</td>\n      <td>9507</td>\n      <td>765</td>\n      <td>4623</td>\n      <td>1058</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>classifier_inner_balanced</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>0.575190</td>\n      <td>0.643891</td>\n      <td>0.610174</td>\n      <td>0.642199</td>\n      <td>0.441764</td>\n      <td>0.731737</td>\n      <td>0.488610</td>\n      <td>5019</td>\n      <td>5253</td>\n      <td>1524</td>\n      <td>4157</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>fixed_classifier_0.5</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>0.663762</td>\n      <td>0.643891</td>\n      <td>0.571170</td>\n      <td>0.660373</td>\n      <td>0.562972</td>\n      <td>0.249428</td>\n      <td>0.892913</td>\n      <td>9172</td>\n      <td>1100</td>\n      <td>4264</td>\n      <td>1417</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>regressor_fixed_0.75</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>0.660001</td>\n      <td>0.643891</td>\n      <td>0.557904</td>\n      <td>0.630438</td>\n      <td>0.562652</td>\n      <td>0.203133</td>\n      <td>0.912675</td>\n      <td>9375</td>\n      <td>897</td>\n      <td>4527</td>\n      <td>1154</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>regressor_inner_accuracy</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>0.660252</td>\n      <td>0.643891</td>\n      <td>0.543348</td>\n      <td>0.630438</td>\n      <td>0.600617</td>\n      <td>0.137124</td>\n      <td>0.949572</td>\n      <td>9754</td>\n      <td>518</td>\n      <td>4902</td>\n      <td>779</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>regressor_inner_balanced</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>0.610857</td>\n      <td>0.643891</td>\n      <td>0.599242</td>\n      <td>0.630438</td>\n      <td>0.461684</td>\n      <td>0.558880</td>\n      <td>0.639603</td>\n      <td>6570</td>\n      <td>3702</td>\n      <td>2506</td>\n      <td>3175</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>selected_classifier_0.5</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>0.643641</td>\n      <td>0.643891</td>\n      <td>0.583710</td>\n      <td>0.642199</td>\n      <td>0.499532</td>\n      <td>0.375462</td>\n      <td>0.791959</td>\n      <td>8135</td>\n      <td>2137</td>\n      <td>3548</td>\n      <td>2133</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>training_majority</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>0.643891</td>\n      <td>0.643891</td>\n      <td>0.500000</td>\n      <td>0.495480</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>1.000000</td>\n      <td>10272</td>\n      <td>0</td>\n      <td>5681</td>\n      <td>0</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>10</td>\n      <td>classifier_inner_accuracy</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>0.633799</td>\n      <td>0.617564</td>\n      <td>0.537857</td>\n      <td>0.629167</td>\n      <td>0.597736</td>\n      <td>0.129815</td>\n      <td>0.945899</td>\n      <td>9319</td>\n      <td>533</td>\n      <td>5309</td>\n      <td>792</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>10</td>\n      <td>classifier_inner_balanced</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>0.571429</td>\n      <td>0.617564</td>\n      <td>0.597536</td>\n      <td>0.629167</td>\n      <td>0.460776</td>\n      <td>0.708572</td>\n      <td>0.486500</td>\n      <td>4793</td>\n      <td>5059</td>\n      <td>1778</td>\n      <td>4323</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>10</td>\n      <td>fixed_classifier_0.5</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>0.636307</td>\n      <td>0.617564</td>\n      <td>0.563195</td>\n      <td>0.640917</td>\n      <td>0.553796</td>\n      <td>0.252254</td>\n      <td>0.874137</td>\n      <td>8612</td>\n      <td>1240</td>\n      <td>4562</td>\n      <td>1539</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>10</td>\n      <td>regressor_fixed_0.75</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>0.637059</td>\n      <td>0.617564</td>\n      <td>0.557283</td>\n      <td>0.607163</td>\n      <td>0.566198</td>\n      <td>0.217997</td>\n      <td>0.896569</td>\n      <td>8833</td>\n      <td>1019</td>\n      <td>4771</td>\n      <td>1330</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>10</td>\n      <td>regressor_inner_accuracy</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>0.637686</td>\n      <td>0.617564</td>\n      <td>0.549491</td>\n      <td>0.607163</td>\n      <td>0.588821</td>\n      <td>0.174398</td>\n      <td>0.924584</td>\n      <td>9109</td>\n      <td>743</td>\n      <td>5037</td>\n      <td>1064</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>10</td>\n      <td>regressor_inner_balanced</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>0.615558</td>\n      <td>0.617564</td>\n      <td>0.577069</td>\n      <td>0.607163</td>\n      <td>0.496848</td>\n      <td>0.413375</td>\n      <td>0.740763</td>\n      <td>7298</td>\n      <td>2554</td>\n      <td>3579</td>\n      <td>2522</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>10</td>\n      <td>selected_classifier_0.5</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>0.606532</td>\n      <td>0.617564</td>\n      <td>0.586860</td>\n      <td>0.629167</td>\n      <td>0.486067</td>\n      <td>0.503196</td>\n      <td>0.670524</td>\n      <td>6606</td>\n      <td>3246</td>\n      <td>3031</td>\n      <td>3070</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>10</td>\n      <td>training_majority</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>0.617564</td>\n      <td>0.617564</td>\n      <td>0.500000</td>\n      <td>0.498425</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>1.000000</td>\n      <td>9852</td>\n      <td>0</td>\n      <td>6101</td>\n      <td>0</td>\n    </tr>\n  </tbody>\n</table>',
'Cutoffs are selected on inner folds only.')

show('focused_classifier_paired_contrasts',
'<table border="1" class="dataframe">\n  <thead>\n    <tr style="text-align: right;">\n      <th>archive_set</th>\n      <th>protocol</th>\n      <th>horizon</th>\n      <th>initial_damage_threshold</th>\n      <th>augmented</th>\n      <th>baseline</th>\n      <th>metric</th>\n      <th>gain_pp</th>\n      <th>lower95_pp</th>\n      <th>upper95_pp</th>\n      <th>valid_replicates</th>\n      <th>bootstrap_repeats</th>\n    </tr>\n  </thead>\n  <tbody>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>20</td>\n      <td>classifier_inner_balanced</td>\n      <td>regressor_inner_balanced</td>\n      <td>accuracy</td>\n      <td>-2.244092</td>\n      <td>-2.911801</td>\n      <td>-1.581073</td>\n      <td>2000</td>\n      <td>2000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>20</td>\n      <td>classifier_inner_balanced</td>\n      <td>regressor_inner_balanced</td>\n      <td>balanced_accuracy</td>\n      <td>0.696272</td>\n      <td>0.044022</td>\n      <td>1.353672</td>\n      <td>2000</td>\n      <td>2000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>20</td>\n      <td>classifier_inner_balanced</td>\n      <td>regressor_inner_balanced</td>\n      <td>recovered_precision</td>\n      <td>-1.604385</td>\n      <td>-2.283239</td>\n      <td>-0.933345</td>\n      <td>2000</td>\n      <td>2000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>20</td>\n      <td>classifier_inner_balanced</td>\n      <td>regressor_inner_balanced</td>\n      <td>recovered_recall</td>\n      <td>10.913572</td>\n      <td>9.888184</td>\n      <td>11.957156</td>\n      <td>2000</td>\n      <td>2000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>20</td>\n      <td>classifier_inner_balanced</td>\n      <td>regressor_inner_balanced</td>\n      <td>preserving_recall</td>\n      <td>-9.521028</td>\n      <td>-10.320354</td>\n      <td>-8.716887</td>\n      <td>2000</td>\n      <td>2000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>20</td>\n      <td>classifier_inner_balanced</td>\n      <td>regressor_inner_balanced</td>\n      <td>recovered_f1</td>\n      <td>2.950888</td>\n      <td>2.213028</td>\n      <td>3.706687</td>\n      <td>2000</td>\n      <td>2000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>10</td>\n      <td>20</td>\n      <td>classifier_inner_balanced</td>\n      <td>regressor_inner_balanced</td>\n      <td>accuracy</td>\n      <td>-3.290917</td>\n      <td>-4.017978</td>\n      <td>-2.545046</td>\n      <td>2000</td>\n      <td>2000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>10</td>\n      <td>20</td>\n      <td>classifier_inner_balanced</td>\n      <td>regressor_inner_balanced</td>\n      <td>balanced_accuracy</td>\n      <td>0.930113</td>\n      <td>0.205050</td>\n      <td>1.673856</td>\n      <td>2000</td>\n      <td>2000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>10</td>\n      <td>20</td>\n      <td>classifier_inner_balanced</td>\n      <td>regressor_inner_balanced</td>\n      <td>recovered_precision</td>\n      <td>-2.825123</td>\n      <td>-3.628982</td>\n      <td>-1.958196</td>\n      <td>2000</td>\n      <td>2000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>10</td>\n      <td>20</td>\n      <td>classifier_inner_balanced</td>\n      <td>regressor_inner_balanced</td>\n      <td>recovered_recall</td>\n      <td>18.882150</td>\n      <td>17.606805</td>\n      <td>20.172672</td>\n      <td>2000</td>\n      <td>2000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>10</td>\n      <td>20</td>\n      <td>classifier_inner_balanced</td>\n      <td>regressor_inner_balanced</td>\n      <td>preserving_recall</td>\n      <td>-17.021924</td>\n      <td>-18.120209</td>\n      <td>-15.946332</td>\n      <td>2000</td>\n      <td>2000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>10</td>\n      <td>20</td>\n      <td>classifier_inner_balanced</td>\n      <td>regressor_inner_balanced</td>\n      <td>recovered_f1</td>\n      <td>6.285689</td>\n      <td>5.348040</td>\n      <td>7.219410</td>\n      <td>2000</td>\n      <td>2000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>20</td>\n      <td>classifier_inner_balanced</td>\n      <td>regressor_inner_balanced</td>\n      <td>accuracy</td>\n      <td>-3.566727</td>\n      <td>-4.278241</td>\n      <td>-2.878013</td>\n      <td>2000</td>\n      <td>2000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>20</td>\n      <td>classifier_inner_balanced</td>\n      <td>regressor_inner_balanced</td>\n      <td>balanced_accuracy</td>\n      <td>1.093195</td>\n      <td>0.393181</td>\n      <td>1.781236</td>\n      <td>2000</td>\n      <td>2000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>20</td>\n      <td>classifier_inner_balanced</td>\n      <td>regressor_inner_balanced</td>\n      <td>recovered_precision</td>\n      <td>-1.991979</td>\n      <td>-2.686867</td>\n      <td>-1.294365</td>\n      <td>2000</td>\n      <td>2000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>20</td>\n      <td>classifier_inner_balanced</td>\n      <td>regressor_inner_balanced</td>\n      <td>recovered_recall</td>\n      <td>17.285689</td>\n      <td>16.136353</td>\n      <td>18.409503</td>\n      <td>2000</td>\n      <td>2000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>20</td>\n      <td>classifier_inner_balanced</td>\n      <td>regressor_inner_balanced</td>\n      <td>preserving_recall</td>\n      <td>-15.099299</td>\n      <td>-16.004390</td>\n      <td>-14.221400</td>\n      <td>2000</td>\n      <td>2000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>20</td>\n      <td>classifier_inner_balanced</td>\n      <td>regressor_inner_balanced</td>\n      <td>recovered_f1</td>\n      <td>4.527063</td>\n      <td>3.743884</td>\n      <td>5.318528</td>\n      <td>2000</td>\n      <td>2000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>10</td>\n      <td>20</td>\n      <td>classifier_inner_balanced</td>\n      <td>regressor_inner_balanced</td>\n      <td>accuracy</td>\n      <td>-4.412963</td>\n      <td>-5.234905</td>\n      <td>-3.613035</td>\n      <td>2000</td>\n      <td>2000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>10</td>\n      <td>20</td>\n      <td>classifier_inner_balanced</td>\n      <td>regressor_inner_balanced</td>\n      <td>balanced_accuracy</td>\n      <td>2.046721</td>\n      <td>1.255702</td>\n      <td>2.824952</td>\n      <td>2000</td>\n      <td>2000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>10</td>\n      <td>20</td>\n      <td>classifier_inner_balanced</td>\n      <td>regressor_inner_balanced</td>\n      <td>recovered_precision</td>\n      <td>-3.607196</td>\n      <td>-4.692858</td>\n      <td>-2.590389</td>\n      <td>2000</td>\n      <td>2000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>10</td>\n      <td>20</td>\n      <td>classifier_inner_balanced</td>\n      <td>regressor_inner_balanced</td>\n      <td>recovered_recall</td>\n      <td>29.519751</td>\n      <td>28.127790</td>\n      <td>30.990216</td>\n      <td>2000</td>\n      <td>2000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>10</td>\n      <td>20</td>\n      <td>classifier_inner_balanced</td>\n      <td>regressor_inner_balanced</td>\n      <td>preserving_recall</td>\n      <td>-25.426309</td>\n      <td>-26.491433</td>\n      <td>-24.354330</td>\n      <td>2000</td>\n      <td>2000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>10</td>\n      <td>20</td>\n      <td>classifier_inner_balanced</td>\n      <td>regressor_inner_balanced</td>\n      <td>recovered_f1</td>\n      <td>10.713502</td>\n      <td>9.692129</td>\n      <td>11.807404</td>\n      <td>2000</td>\n      <td>2000</td>\n    </tr>\n  </tbody>\n</table>',
'Both models use inner-selected cutoffs; gains are percentage points.')


**Direct classifiers five archives**

Positive class: recovered. Default and inner-selected cutoffs are separate strategies.

archive_set,protocol,target,model,strategy,n,games,accuracy,majority_baseline_accuracy,balanced_accuracy,roc_auc,recovered_precision,recovered_recall,tn,fp,fn,tp
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,6,decision_tree,default_0.5,23167,6512,0.643933,0.641689,0.506883,0.577693,0.577844,0.023250,14725.000000,141.000000,8108.000000,193.000000
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,6,decision_tree,inner_balanced_accuracy,23167,6512,0.528726,0.641689,0.555221,0.577693,0.402256,0.648717,6864.000000,8002.000000,2916.000000,5385.000000
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,6,engine_only,default_0.5,23167,6512,0.640092,0.641689,0.501469,0.577342,0.423237,0.012288,14727.000000,139.000000,8199.000000,102.000000
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,6,engine_only,inner_balanced_accuracy,23167,6512,0.507316,0.641689,0.553275,0.577342,0.396171,0.715456,5814.000000,9052.000000,2362.000000,5939.000000
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,6,engine_ratings,default_0.5,23167,6512,0.638538,0.641689,0.506456,0.583407,0.450875,0.040357,14458.000000,408.000000,7966.000000,335.000000
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,6,engine_ratings,inner_balanced_accuracy,23167,6512,0.544136,0.641689,0.558264,0.583407,0.408546,0.608120,7558.000000,7308.000000,3253.000000,5048.000000
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,6,fixed_lightgbm,default_0.5,23167,6512,0.646523,0.641689,0.523744,0.617453,0.540288,0.090471,14227.000000,639.000000,7550.000000,751.000000
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,6,fixed_lightgbm,inner_balanced_accuracy,23167,6512,0.580567,0.641689,0.582156,0.617453,0.436639,0.587761,8571.000000,6295.000000,3422.000000,4879.000000
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,6,optuna_lightgbm,default_0.5,23167,6512,0.646005,0.641689,0.524085,0.618699,0.534294,0.093844,14187.000000,679.000000,7522.000000,779.000000
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,6,optuna_lightgbm,inner_balanced_accuracy,23167,6512,0.583718,0.641689,0.581605,0.618699,0.438253,0.574148,8757.000000,6109.000000,3535.000000,4766.000000


**Focused direct and threshold strategies four archives**

Cutoffs are selected on inner folds only.

archive_set,protocol,horizon,strategy,n,games,accuracy,majority_accuracy,balanced_accuracy,roc_auc,recovered_precision,recovered_recall,preserving_recall,tn,fp,fn,tp
"2017-04, 2017-06, 2017-07, 2018-02",game,6,classifier_inner_accuracy,15953,4485,0.662383,0.643891,0.564042,0.661863,0.566114,0.222320,0.905763,9304,968,4418,1263
"2017-04, 2017-06, 2017-07, 2018-02",game,6,classifier_inner_balanced,15953,4485,0.597944,0.643891,0.615491,0.661863,0.456468,0.676465,0.554517,5696,4576,1838,3843
"2017-04, 2017-06, 2017-07, 2018-02",game,6,fixed_classifier_0.5,15953,4485,0.665079,0.643891,0.573137,0.663828,0.566431,0.253653,0.892621,9169,1103,4240,1441
"2017-04, 2017-06, 2017-07, 2018-02",game,6,regressor_fixed_0.75,15953,4485,0.660377,0.643891,0.557764,0.636537,0.565002,0.201197,0.914330,9392,880,4538,1143
"2017-04, 2017-06, 2017-07, 2018-02",game,6,regressor_inner_accuracy,15953,4485,0.660942,0.643891,0.556746,0.636537,0.570103,0.194684,0.918808,9438,834,4575,1106
"2017-04, 2017-06, 2017-07, 2018-02",game,6,regressor_inner_balanced,15953,4485,0.620385,0.643891,0.608529,0.636537,0.472511,0.567330,0.649727,6674,3598,2458,3223
"2017-04, 2017-06, 2017-07, 2018-02",game,6,selected_classifier_0.5,15953,4485,0.607848,0.643891,0.617360,0.661863,0.463905,0.650414,0.584307,6002,4270,1986,3695
"2017-04, 2017-06, 2017-07, 2018-02",game,6,training_majority,15953,4485,0.643891,0.643891,0.500000,0.495480,unavailable / undefined,0.000000,1.000000,10272,0,5681,0
"2017-04, 2017-06, 2017-07, 2018-02",game,10,classifier_inner_accuracy,15953,4485,0.641008,0.617564,0.561635,0.638426,0.579237,0.224062,0.899208,8859,993,4734,1367
"2017-04, 2017-06, 2017-07, 2018-02",game,10,classifier_inner_balanced,15953,4485,0.580392,0.617564,0.596587,0.638426,0.465970,0.665465,0.527710,5199,4653,2041,4060


**Focused classifier paired contrasts**

Both models use inner-selected cutoffs; gains are percentage points.

archive_set,protocol,horizon,initial_damage_threshold,augmented,baseline,metric,gain_pp,lower95_pp,upper95_pp,valid_replicates,bootstrap_repeats
"2017-04, 2017-06, 2017-07, 2018-02",game,6,20,classifier_inner_balanced,regressor_inner_balanced,accuracy,-2.244092,-2.911801,-1.581073,2000,2000
"2017-04, 2017-06, 2017-07, 2018-02",game,6,20,classifier_inner_balanced,regressor_inner_balanced,balanced_accuracy,0.696272,0.044022,1.353672,2000,2000
"2017-04, 2017-06, 2017-07, 2018-02",game,6,20,classifier_inner_balanced,regressor_inner_balanced,recovered_precision,-1.604385,-2.283239,-0.933345,2000,2000
"2017-04, 2017-06, 2017-07, 2018-02",game,6,20,classifier_inner_balanced,regressor_inner_balanced,recovered_recall,10.913572,9.888184,11.957156,2000,2000
"2017-04, 2017-06, 2017-07, 2018-02",game,6,20,classifier_inner_balanced,regressor_inner_balanced,preserving_recall,-9.521028,-10.320354,-8.716887,2000,2000
"2017-04, 2017-06, 2017-07, 2018-02",game,6,20,classifier_inner_balanced,regressor_inner_balanced,recovered_f1,2.950888,2.213028,3.706687,2000,2000
"2017-04, 2017-06, 2017-07, 2018-02",game,10,20,classifier_inner_balanced,regressor_inner_balanced,accuracy,-3.290917,-4.017978,-2.545046,2000,2000
"2017-04, 2017-06, 2017-07, 2018-02",game,10,20,classifier_inner_balanced,regressor_inner_balanced,balanced_accuracy,0.930113,0.205050,1.673856,2000,2000
"2017-04, 2017-06, 2017-07, 2018-02",game,10,20,classifier_inner_balanced,regressor_inner_balanced,recovered_precision,-2.825123,-3.628982,-1.958196,2000,2000
"2017-04, 2017-06, 2017-07, 2018-02",game,10,20,classifier_inner_balanced,regressor_inner_balanced,recovered_recall,18.882150,17.606805,20.172672,2000,2000


### Continuous Maia and combined features

The continuous feature estimates Maia-weighted signed change after a hypothetical reply, from the original mistake-maker's perspective. Its midpoint is the primary input; lower and upper endpoints are sensitivity checks. The combined variant adds normalized clocks and engine reply agreement to the Maia probability. Each contrast uses its own matched reference predictions.


In [13]:
show('continuous_maia_metrics',
'<table border="1" class="dataframe">\n  <thead>\n    <tr style="text-align: right;">\n      <th>archive_set</th>\n      <th>protocol</th>\n      <th>variant</th>\n      <th>target</th>\n      <th>n</th>\n      <th>games</th>\n      <th>mae</th>\n      <th>spearman</th>\n      <th>recovered_n_diagnostic</th>\n      <th>recovered_only_mae_diagnostic</th>\n    </tr>\n  </thead>\n  <tbody>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>original_plus_maia_probability</td>\n      <td>change_in_damage_10ply</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>22.732183</td>\n      <td>0.178333</td>\n      <td>6101</td>\n      <td>38.738184</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>plus_continuous_lower</td>\n      <td>change_in_damage_10ply</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>22.677281</td>\n      <td>0.195519</td>\n      <td>6101</td>\n      <td>38.234902</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>plus_continuous_midpoint</td>\n      <td>change_in_damage_10ply</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>22.660705</td>\n      <td>0.191076</td>\n      <td>6101</td>\n      <td>38.327210</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>plus_continuous_upper</td>\n      <td>change_in_damage_10ply</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>22.675934</td>\n      <td>0.190462</td>\n      <td>6101</td>\n      <td>38.294326</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>training_mean</td>\n      <td>change_in_damage_10ply</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>25.372458</td>\n      <td>unavailable / undefined</td>\n      <td>6101</td>\n      <td>33.281956</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>training_median</td>\n      <td>change_in_damage_10ply</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>23.454078</td>\n      <td>unavailable / undefined</td>\n      <td>6101</td>\n      <td>43.727505</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>zero</td>\n      <td>change_in_damage_10ply</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>23.454479</td>\n      <td>unavailable / undefined</td>\n      <td>6101</td>\n      <td>43.552549</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>original_plus_maia_probability</td>\n      <td>change_in_damage_10ply</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>22.823851</td>\n      <td>0.174504</td>\n      <td>6101</td>\n      <td>38.848907</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>plus_continuous_lower</td>\n      <td>change_in_damage_10ply</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>22.808451</td>\n      <td>0.185605</td>\n      <td>6101</td>\n      <td>38.392599</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>plus_continuous_midpoint</td>\n      <td>change_in_damage_10ply</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>22.818068</td>\n      <td>0.185372</td>\n      <td>6101</td>\n      <td>38.426787</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>plus_continuous_upper</td>\n      <td>change_in_damage_10ply</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>22.803318</td>\n      <td>0.184927</td>\n      <td>6101</td>\n      <td>38.380409</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>training_mean</td>\n      <td>change_in_damage_10ply</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>25.312923</td>\n      <td>unavailable / undefined</td>\n      <td>6101</td>\n      <td>33.474108</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>training_median</td>\n      <td>change_in_damage_10ply</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>23.453530</td>\n      <td>unavailable / undefined</td>\n      <td>6101</td>\n      <td>43.756157</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>zero</td>\n      <td>change_in_damage_10ply</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>23.454479</td>\n      <td>unavailable / undefined</td>\n      <td>6101</td>\n      <td>43.552549</td>\n    </tr>\n  </tbody>\n</table>',
'')

show('continuous_maia_primary_contrast',
'<table border="1" class="dataframe">\n  <thead>\n    <tr style="text-align: right;">\n      <th>archive_set</th>\n      <th>protocol</th>\n      <th>initial_damage_threshold</th>\n      <th>variant</th>\n      <th>comparator</th>\n      <th>n</th>\n      <th>games</th>\n      <th>reference_mae</th>\n      <th>augmented_mae</th>\n      <th>mae_difference_pp</th>\n      <th>difference_lower95</th>\n      <th>difference_upper95</th>\n      <th>improvement_pp</th>\n      <th>improvement_lower95</th>\n      <th>improvement_upper95</th>\n      <th>improvement_percent</th>\n      <th>percent_lower95</th>\n      <th>percent_upper95</th>\n      <th>bootstrap_repeats</th>\n    </tr>\n  </thead>\n  <tbody>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>20</td>\n      <td>plus_continuous_midpoint</td>\n      <td>original_plus_maia_probability</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>22.732183</td>\n      <td>22.660705</td>\n      <td>-0.071478</td>\n      <td>-0.119628</td>\n      <td>-0.022604</td>\n      <td>0.071478</td>\n      <td>0.022604</td>\n      <td>0.119628</td>\n      <td>0.314435</td>\n      <td>0.098616</td>\n      <td>0.524707</td>\n      <td>2000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>20</td>\n      <td>plus_continuous_midpoint</td>\n      <td>original_plus_maia_probability</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>22.823851</td>\n      <td>22.818068</td>\n      <td>-0.005783</td>\n      <td>-0.059596</td>\n      <td>0.051105</td>\n      <td>0.005783</td>\n      <td>-0.051105</td>\n      <td>0.059596</td>\n      <td>0.025336</td>\n      <td>-0.222468</td>\n      <td>0.260868</td>\n      <td>2000</td>\n    </tr>\n  </tbody>\n</table>',
'Positive improvement favors the augmented variant.')

show('combined_maia_clock_agreement_metrics',
'<table border="1" class="dataframe">\n  <thead>\n    <tr style="text-align: right;">\n      <th>archive_set</th>\n      <th>protocol</th>\n      <th>variant</th>\n      <th>target</th>\n      <th>n</th>\n      <th>games</th>\n      <th>mae</th>\n      <th>spearman</th>\n      <th>recovered_n_diagnostic</th>\n      <th>recovered_only_mae_diagnostic</th>\n    </tr>\n  </thead>\n  <tbody>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>combined_lightgbm</td>\n      <td>change_in_damage_10ply</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>22.730060</td>\n      <td>0.175999</td>\n      <td>6101</td>\n      <td>38.666536</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>frozen_lightgbm</td>\n      <td>change_in_damage_10ply</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>22.930386</td>\n      <td>0.133253</td>\n      <td>6101</td>\n      <td>40.483837</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>lightgbm_plus_maia</td>\n      <td>change_in_damage_10ply</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>22.732183</td>\n      <td>0.178333</td>\n      <td>6101</td>\n      <td>38.738184</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>training_mean</td>\n      <td>change_in_damage_10ply</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>25.372458</td>\n      <td>unavailable / undefined</td>\n      <td>6101</td>\n      <td>33.281956</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>training_median</td>\n      <td>change_in_damage_10ply</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>23.454078</td>\n      <td>unavailable / undefined</td>\n      <td>6101</td>\n      <td>43.727505</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>zero</td>\n      <td>change_in_damage_10ply</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>23.454479</td>\n      <td>unavailable / undefined</td>\n      <td>6101</td>\n      <td>43.552549</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>combined_lightgbm</td>\n      <td>change_in_damage_10ply</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>22.808208</td>\n      <td>0.169375</td>\n      <td>6101</td>\n      <td>38.875274</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>frozen_lightgbm</td>\n      <td>change_in_damage_10ply</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>23.056998</td>\n      <td>0.123293</td>\n      <td>6101</td>\n      <td>40.572835</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>lightgbm_plus_maia</td>\n      <td>change_in_damage_10ply</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>22.823851</td>\n      <td>0.174504</td>\n      <td>6101</td>\n      <td>38.848907</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>training_mean</td>\n      <td>change_in_damage_10ply</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>25.312923</td>\n      <td>unavailable / undefined</td>\n      <td>6101</td>\n      <td>33.474108</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>training_median</td>\n      <td>change_in_damage_10ply</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>23.453530</td>\n      <td>unavailable / undefined</td>\n      <td>6101</td>\n      <td>43.756157</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>zero</td>\n      <td>change_in_damage_10ply</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>23.454479</td>\n      <td>unavailable / undefined</td>\n      <td>6101</td>\n      <td>43.552549</td>\n    </tr>\n  </tbody>\n</table>',
'')

show('combined_maia_clock_agreement_primary_contrast',
'<table border="1" class="dataframe">\n  <thead>\n    <tr style="text-align: right;">\n      <th>archive_set</th>\n      <th>protocol</th>\n      <th>target</th>\n      <th>initial_damage_threshold</th>\n      <th>variant</th>\n      <th>comparator</th>\n      <th>n</th>\n      <th>games</th>\n      <th>improvement_pp</th>\n      <th>lower95</th>\n      <th>upper95</th>\n      <th>improvement_percent</th>\n      <th>percent_lower95</th>\n      <th>percent_upper95</th>\n      <th>bootstrap_repeats</th>\n    </tr>\n  </thead>\n  <tbody>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>change_in_damage_10ply</td>\n      <td>20</td>\n      <td>combined_lightgbm</td>\n      <td>lightgbm_plus_maia</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>0.002122</td>\n      <td>-0.060798</td>\n      <td>0.065453</td>\n      <td>0.009335</td>\n      <td>-0.265977</td>\n      <td>0.287321</td>\n      <td>2000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>change_in_damage_10ply</td>\n      <td>20</td>\n      <td>combined_lightgbm</td>\n      <td>lightgbm_plus_maia</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>0.015643</td>\n      <td>-0.034150</td>\n      <td>0.062946</td>\n      <td>0.068537</td>\n      <td>-0.150310</td>\n      <td>0.275294</td>\n      <td>2000</td>\n    </tr>\n  </tbody>\n</table>',
'Positive improvement favors the augmented variant.')

show('maia_probability_endpoint_sensitivity',
'<table border="1" class="dataframe">\n  <thead>\n    <tr style="text-align: right;">\n      <th>archive_set</th>\n      <th>target</th>\n      <th>protocol</th>\n      <th>initial_damage_threshold</th>\n      <th>variant</th>\n      <th>n</th>\n      <th>games</th>\n      <th>improvement_over_fixed_pp</th>\n      <th>improvement_lower95</th>\n      <th>improvement_upper95</th>\n      <th>improvement_percent</th>\n    </tr>\n  </thead>\n  <tbody>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>opponent_reply_regret_pp</td>\n      <td>game</td>\n      <td>20</td>\n      <td>plus_maia</td>\n      <td>15851</td>\n      <td>4472</td>\n      <td>2.134172</td>\n      <td>1.962499</td>\n      <td>2.314839</td>\n      <td>15.347803</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>opponent_reply_regret_pp</td>\n      <td>game</td>\n      <td>20</td>\n      <td>plus_maia_lower</td>\n      <td>15851</td>\n      <td>4472</td>\n      <td>2.125174</td>\n      <td>1.958269</td>\n      <td>2.306028</td>\n      <td>15.283090</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>opponent_reply_regret_pp</td>\n      <td>game</td>\n      <td>20</td>\n      <td>plus_maia_upper</td>\n      <td>15851</td>\n      <td>4472</td>\n      <td>2.125458</td>\n      <td>1.956533</td>\n      <td>2.306611</td>\n      <td>15.285139</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>opponent_reply_regret_pp</td>\n      <td>player</td>\n      <td>20</td>\n      <td>plus_maia</td>\n      <td>15851</td>\n      <td>4472</td>\n      <td>2.046771</td>\n      <td>1.874042</td>\n      <td>2.222993</td>\n      <td>14.758879</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>opponent_reply_regret_pp</td>\n      <td>player</td>\n      <td>20</td>\n      <td>plus_maia_lower</td>\n      <td>15851</td>\n      <td>4472</td>\n      <td>2.058452</td>\n      <td>1.885878</td>\n      <td>2.231979</td>\n      <td>14.843110</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>opponent_reply_regret_pp</td>\n      <td>player</td>\n      <td>20</td>\n      <td>plus_maia_upper</td>\n      <td>15851</td>\n      <td>4472</td>\n      <td>2.060968</td>\n      <td>1.888227</td>\n      <td>2.238541</td>\n      <td>14.861252</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>change_in_damage_10ply</td>\n      <td>game</td>\n      <td>20</td>\n      <td>plus_maia</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>0.198204</td>\n      <td>0.116301</td>\n      <td>0.281079</td>\n      <td>0.864371</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>change_in_damage_10ply</td>\n      <td>game</td>\n      <td>20</td>\n      <td>plus_maia_lower</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>0.208357</td>\n      <td>0.126772</td>\n      <td>0.289977</td>\n      <td>0.908649</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>change_in_damage_10ply</td>\n      <td>game</td>\n      <td>20</td>\n      <td>plus_maia_upper</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>0.229206</td>\n      <td>0.146259</td>\n      <td>0.310162</td>\n      <td>0.999573</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>change_in_damage_10ply</td>\n      <td>player</td>\n      <td>20</td>\n      <td>plus_maia</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>0.233147</td>\n      <td>0.155220</td>\n      <td>0.308837</td>\n      <td>1.011178</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>change_in_damage_10ply</td>\n      <td>player</td>\n      <td>20</td>\n      <td>plus_maia_lower</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>0.240490</td>\n      <td>0.163678</td>\n      <td>0.314116</td>\n      <td>1.043025</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>change_in_damage_10ply</td>\n      <td>player</td>\n      <td>20</td>\n      <td>plus_maia_upper</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>0.232118</td>\n      <td>0.156554</td>\n      <td>0.305632</td>\n      <td>1.006715</td>\n    </tr>\n  </tbody>\n</table>',
'Sensitivity to numerical integration endpoints.')

show('continuous_maia_endpoint_contrasts',
'<table border="1" class="dataframe">\n  <thead>\n    <tr style="text-align: right;">\n      <th>archive_set</th>\n      <th>protocol</th>\n      <th>initial_damage_threshold</th>\n      <th>variant</th>\n      <th>comparator</th>\n      <th>n</th>\n      <th>games</th>\n      <th>reference_mae</th>\n      <th>augmented_mae</th>\n      <th>mae_difference_pp</th>\n      <th>difference_lower95</th>\n      <th>difference_upper95</th>\n      <th>improvement_pp</th>\n      <th>improvement_lower95</th>\n      <th>improvement_upper95</th>\n      <th>improvement_percent</th>\n      <th>percent_lower95</th>\n      <th>percent_upper95</th>\n      <th>bootstrap_repeats</th>\n    </tr>\n  </thead>\n  <tbody>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>20</td>\n      <td>plus_continuous_midpoint</td>\n      <td>original_plus_maia_probability</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>22.732183</td>\n      <td>22.660705</td>\n      <td>-0.071478</td>\n      <td>-0.119628</td>\n      <td>-0.022604</td>\n      <td>0.071478</td>\n      <td>0.022604</td>\n      <td>0.119628</td>\n      <td>0.314435</td>\n      <td>0.098616</td>\n      <td>0.524707</td>\n      <td>2000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>20</td>\n      <td>plus_continuous_lower</td>\n      <td>original_plus_maia_probability</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>22.732183</td>\n      <td>22.677281</td>\n      <td>-0.054902</td>\n      <td>-0.106419</td>\n      <td>-0.003854</td>\n      <td>0.054902</td>\n      <td>0.003854</td>\n      <td>0.106419</td>\n      <td>0.241516</td>\n      <td>0.017162</td>\n      <td>0.464420</td>\n      <td>2000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>20</td>\n      <td>plus_continuous_upper</td>\n      <td>original_plus_maia_probability</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>22.732183</td>\n      <td>22.675934</td>\n      <td>-0.056248</td>\n      <td>-0.107690</td>\n      <td>-0.005677</td>\n      <td>0.056248</td>\n      <td>0.005677</td>\n      <td>0.107690</td>\n      <td>0.247439</td>\n      <td>0.025168</td>\n      <td>0.473542</td>\n      <td>2000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>20</td>\n      <td>plus_continuous_midpoint</td>\n      <td>original_plus_maia_probability</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>22.823851</td>\n      <td>22.818068</td>\n      <td>-0.005783</td>\n      <td>-0.059596</td>\n      <td>0.051105</td>\n      <td>0.005783</td>\n      <td>-0.051105</td>\n      <td>0.059596</td>\n      <td>0.025336</td>\n      <td>-0.222468</td>\n      <td>0.260868</td>\n      <td>2000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>20</td>\n      <td>plus_continuous_lower</td>\n      <td>original_plus_maia_probability</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>22.823851</td>\n      <td>22.808451</td>\n      <td>-0.015400</td>\n      <td>-0.069134</td>\n      <td>0.038714</td>\n      <td>0.015400</td>\n      <td>-0.038714</td>\n      <td>0.069134</td>\n      <td>0.067474</td>\n      <td>-0.170320</td>\n      <td>0.301524</td>\n      <td>2000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>20</td>\n      <td>plus_continuous_upper</td>\n      <td>original_plus_maia_probability</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>22.823851</td>\n      <td>22.803318</td>\n      <td>-0.020533</td>\n      <td>-0.071687</td>\n      <td>0.034319</td>\n      <td>0.020533</td>\n      <td>-0.034319</td>\n      <td>0.071687</td>\n      <td>0.089964</td>\n      <td>-0.150985</td>\n      <td>0.312205</td>\n      <td>2000</td>\n    </tr>\n  </tbody>\n</table>',
'')


**Continuous Maia metrics**

archive_set,protocol,variant,target,n,games,mae,spearman,recovered_n_diagnostic,recovered_only_mae_diagnostic
"2017-04, 2017-06, 2017-07, 2018-02",game,original_plus_maia_probability,change_in_damage_10ply,15953,4485,22.732183,0.178333,6101,38.738184
"2017-04, 2017-06, 2017-07, 2018-02",game,plus_continuous_lower,change_in_damage_10ply,15953,4485,22.677281,0.195519,6101,38.234902
"2017-04, 2017-06, 2017-07, 2018-02",game,plus_continuous_midpoint,change_in_damage_10ply,15953,4485,22.660705,0.191076,6101,38.327210
"2017-04, 2017-06, 2017-07, 2018-02",game,plus_continuous_upper,change_in_damage_10ply,15953,4485,22.675934,0.190462,6101,38.294326
"2017-04, 2017-06, 2017-07, 2018-02",game,training_mean,change_in_damage_10ply,15953,4485,25.372458,unavailable / undefined,6101,33.281956
"2017-04, 2017-06, 2017-07, 2018-02",game,training_median,change_in_damage_10ply,15953,4485,23.454078,unavailable / undefined,6101,43.727505
"2017-04, 2017-06, 2017-07, 2018-02",game,zero,change_in_damage_10ply,15953,4485,23.454479,unavailable / undefined,6101,43.552549
"2017-04, 2017-06, 2017-07, 2018-02",player,original_plus_maia_probability,change_in_damage_10ply,15953,4485,22.823851,0.174504,6101,38.848907
"2017-04, 2017-06, 2017-07, 2018-02",player,plus_continuous_lower,change_in_damage_10ply,15953,4485,22.808451,0.185605,6101,38.392599
"2017-04, 2017-06, 2017-07, 2018-02",player,plus_continuous_midpoint,change_in_damage_10ply,15953,4485,22.818068,0.185372,6101,38.426787


**Continuous Maia primary contrast**

Positive improvement favors the augmented variant.

archive_set,protocol,initial_damage_threshold,variant,comparator,n,games,reference_mae,augmented_mae,mae_difference_pp,difference_lower95,difference_upper95,improvement_pp,improvement_lower95,improvement_upper95,improvement_percent,percent_lower95,percent_upper95,bootstrap_repeats
"2017-04, 2017-06, 2017-07, 2018-02",game,20,plus_continuous_midpoint,original_plus_maia_probability,15953,4485,22.732183,22.660705,-0.071478,-0.119628,-0.022604,0.071478,0.022604,0.119628,0.314435,0.098616,0.524707,2000
"2017-04, 2017-06, 2017-07, 2018-02",player,20,plus_continuous_midpoint,original_plus_maia_probability,15953,4485,22.823851,22.818068,-0.005783,-0.059596,0.051105,0.005783,-0.051105,0.059596,0.025336,-0.222468,0.260868,2000


**Combined Maia clock agreement metrics**

... (5 outputs left out)

Sensitivity to numerical integration endpoints.

archive_set,target,protocol,initial_damage_threshold,variant,n,games,improvement_over_fixed_pp,improvement_lower95,improvement_upper95,improvement_percent
"2017-04, 2017-06, 2017-07, 2018-02",opponent_reply_regret_pp,game,20,plus_maia,15851,4472,2.134172,1.962499,2.314839,15.347803
"2017-04, 2017-06, 2017-07, 2018-02",opponent_reply_regret_pp,game,20,plus_maia_lower,15851,4472,2.125174,1.958269,2.306028,15.283090
"2017-04, 2017-06, 2017-07, 2018-02",opponent_reply_regret_pp,game,20,plus_maia_upper,15851,4472,2.125458,1.956533,2.306611,15.285139
"2017-04, 2017-06, 2017-07, 2018-02",opponent_reply_regret_pp,player,20,plus_maia,15851,4472,2.046771,1.874042,2.222993,14.758879
"2017-04, 2017-06, 2017-07, 2018-02",opponent_reply_regret_pp,player,20,plus_maia_lower,15851,4472,2.058452,1.885878,2.231979,14.843110
"2017-04, 2017-06, 2017-07, 2018-02",opponent_reply_regret_pp,player,20,plus_maia_upper,15851,4472,2.060968,1.888227,2.238541,14.861252
"2017-04, 2017-06, 2017-07, 2018-02",change_in_damage_10ply,game,20,plus_maia,15953,4485,0.198204,0.116301,0.281079,0.864371
"2017-04, 2017-06, 2017-07, 2018-02",change_in_damage_10ply,game,20,plus_maia_lower,15953,4485,0.208357,0.126772,0.289977,0.908649
"2017-04, 2017-06, 2017-07, 2018-02",change_in_damage_10ply,game,20,plus_maia_upper,15953,4485,0.229206,0.146259,0.310162,0.999573
"2017-04, 2017-06, 2017-07, 2018-02",change_in_damage_10ply,player,20,plus_maia,15953,4485,0.233147,0.155220,0.308837,1.011178


**Continuous Maia endpoint contrasts**

archive_set,protocol,initial_damage_threshold,variant,comparator,n,games,reference_mae,augmented_mae,mae_difference_pp,difference_lower95,difference_upper95,improvement_pp,improvement_lower95,improvement_upper95,improvement_percent,percent_lower95,percent_upper95,bootstrap_repeats
"2017-04, 2017-06, 2017-07, 2018-02",game,20,plus_continuous_midpoint,original_plus_maia_probability,15953,4485,22.732183,22.660705,-0.071478,-0.119628,-0.022604,0.071478,0.022604,0.119628,0.314435,0.098616,0.524707,2000
"2017-04, 2017-06, 2017-07, 2018-02",game,20,plus_continuous_lower,original_plus_maia_probability,15953,4485,22.732183,22.677281,-0.054902,-0.106419,-0.003854,0.054902,0.003854,0.106419,0.241516,0.017162,0.464420,2000
"2017-04, 2017-06, 2017-07, 2018-02",game,20,plus_continuous_upper,original_plus_maia_probability,15953,4485,22.732183,22.675934,-0.056248,-0.107690,-0.005677,0.056248,0.005677,0.107690,0.247439,0.025168,0.473542,2000
"2017-04, 2017-06, 2017-07, 2018-02",player,20,plus_continuous_midpoint,original_plus_maia_probability,15953,4485,22.823851,22.818068,-0.005783,-0.059596,0.051105,0.005783,-0.051105,0.059596,0.025336,-0.222468,0.260868,2000
"2017-04, 2017-06, 2017-07, 2018-02",player,20,plus_continuous_lower,original_plus_maia_probability,15953,4485,22.823851,22.808451,-0.015400,-0.069134,0.038714,0.015400,-0.038714,0.069134,0.067474,-0.170320,0.301524,2000
"2017-04, 2017-06, 2017-07, 2018-02",player,20,plus_continuous_upper,original_plus_maia_probability,15953,4485,22.823851,22.803318,-0.020533,-0.071687,0.034319,0.020533,-0.034319,0.071687,0.089964,-0.150985,0.312205,2000


### Optuna, threshold and robustness checks

The five-archive Optuna study used three outer and three inner folds, seed 1701, and 12 sequential TPE trials per algorithm, target and outer fold, without pruning or early stopping. Outer predictions evaluate the inner-selected settings. Equal trial counts do not imply equal runtime or optimal tuning.

Threshold studies keep the actual recovery label fixed at ratio <0.75 and select prediction cutoffs using inner OOF scores. The >25/>30 robustness analyses restrict the same >20 OOF predictions without refitting. Elo, time-control and phase summaries are descriptive.


In [14]:
show('optuna_regression',
'<table border="1" class="dataframe">\n  <thead>\n    <tr style="text-align: right;">\n      <th>archive_set</th>\n      <th>protocol</th>\n      <th>target</th>\n      <th>model</th>\n      <th>n</th>\n      <th>games</th>\n      <th>mae</th>\n      <th>spearman</th>\n      <th>estimate_mae_improvement_pp</th>\n      <th>lower95_mae_improvement_pp</th>\n      <th>upper95_mae_improvement_pp</th>\n    </tr>\n  </thead>\n  <tbody>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>2</td>\n      <td>fixed_lightgbm</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>12.558943</td>\n      <td>0.116483</td>\n      <td>0.017615</td>\n      <td>0.011389</td>\n      <td>0.023895</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>2</td>\n      <td>optuna_lightgbm</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>12.554643</td>\n      <td>0.117968</td>\n      <td>0.021915</td>\n      <td>0.004240</td>\n      <td>0.039136</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>2</td>\n      <td>optuna_xgboost</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>12.544558</td>\n      <td>0.116243</td>\n      <td>0.032000</td>\n      <td>0.018283</td>\n      <td>0.046740</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>4</td>\n      <td>fixed_lightgbm</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>16.843501</td>\n      <td>0.104874</td>\n      <td>0.092363</td>\n      <td>0.040911</td>\n      <td>0.146109</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>4</td>\n      <td>optuna_lightgbm</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>16.811599</td>\n      <td>0.098739</td>\n      <td>0.124265</td>\n      <td>0.060563</td>\n      <td>0.187495</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>4</td>\n      <td>optuna_xgboost</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>16.820052</td>\n      <td>0.098783</td>\n      <td>0.115812</td>\n      <td>0.058706</td>\n      <td>0.171249</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>fixed_lightgbm</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>19.579173</td>\n      <td>0.099988</td>\n      <td>0.255319</td>\n      <td>0.172802</td>\n      <td>0.338477</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>optuna_lightgbm</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>19.565276</td>\n      <td>0.100014</td>\n      <td>0.269216</td>\n      <td>0.184484</td>\n      <td>0.349004</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>6</td>\n      <td>optuna_xgboost</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>19.546584</td>\n      <td>0.101415</td>\n      <td>0.287908</td>\n      <td>0.209031</td>\n      <td>0.364718</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>10</td>\n      <td>fixed_lightgbm</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>23.049053</td>\n      <td>0.123120</td>\n      <td>0.586358</td>\n      <td>0.462247</td>\n      <td>0.707210</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>10</td>\n      <td>optuna_lightgbm</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>23.060939</td>\n      <td>0.120210</td>\n      <td>0.574472</td>\n      <td>0.461893</td>\n      <td>0.685609</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>10</td>\n      <td>optuna_xgboost</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>23.047485</td>\n      <td>0.119099</td>\n      <td>0.587926</td>\n      <td>0.473892</td>\n      <td>0.696195</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>2</td>\n      <td>fixed_lightgbm</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>12.559600</td>\n      <td>0.107507</td>\n      <td>0.016959</td>\n      <td>0.010862</td>\n      <td>0.022967</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>2</td>\n      <td>optuna_lightgbm</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>12.557639</td>\n      <td>0.110772</td>\n      <td>0.018919</td>\n      <td>0.014369</td>\n      <td>0.023428</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>2</td>\n      <td>optuna_xgboost</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>12.558491</td>\n      <td>0.112125</td>\n      <td>0.018067</td>\n      <td>0.006076</td>\n      <td>0.030062</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>4</td>\n      <td>fixed_lightgbm</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>16.878519</td>\n      <td>0.093913</td>\n      <td>0.057345</td>\n      <td>0.013943</td>\n      <td>0.101248</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>4</td>\n      <td>optuna_lightgbm</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>16.880082</td>\n      <td>0.089291</td>\n      <td>0.055783</td>\n      <td>0.013433</td>\n      <td>0.099117</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>4</td>\n      <td>optuna_xgboost</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>16.822227</td>\n      <td>0.096662</td>\n      <td>0.113637</td>\n      <td>0.054803</td>\n      <td>0.175512</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>fixed_lightgbm</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>19.617191</td>\n      <td>0.094367</td>\n      <td>0.217301</td>\n      <td>0.138916</td>\n      <td>0.293363</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>optuna_lightgbm</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>19.637148</td>\n      <td>0.087821</td>\n      <td>0.197344</td>\n      <td>0.115810</td>\n      <td>0.278306</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>6</td>\n      <td>optuna_xgboost</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>19.611260</td>\n      <td>0.088723</td>\n      <td>0.223232</td>\n      <td>0.149032</td>\n      <td>0.292199</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>10</td>\n      <td>fixed_lightgbm</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>23.133595</td>\n      <td>0.111968</td>\n      <td>0.501816</td>\n      <td>0.383311</td>\n      <td>0.620378</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>10</td>\n      <td>optuna_lightgbm</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>23.112520</td>\n      <td>0.107460</td>\n      <td>0.522891</td>\n      <td>0.411050</td>\n      <td>0.636792</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>10</td>\n      <td>optuna_xgboost</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>23.115777</td>\n      <td>0.110181</td>\n      <td>0.519634</td>\n      <td>0.410886</td>\n      <td>0.625239</td>\n    </tr>\n  </tbody>\n</table>',
'Gains compare with no-change; nested selection only for Optuna variants.')

show('optuna_vs_fixed_intervals',
'<table border="1" class="dataframe">\n  <thead>\n    <tr style="text-align: right;">\n      <th>archive_set</th>\n      <th>protocol</th>\n      <th>task</th>\n      <th>target</th>\n      <th>model</th>\n      <th>comparator</th>\n      <th>metric</th>\n      <th>estimate</th>\n      <th>lower95</th>\n      <th>upper95</th>\n      <th>initial_damage_threshold</th>\n      <th>n</th>\n      <th>games</th>\n    </tr>\n  </thead>\n  <tbody>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>regression</td>\n      <td>2</td>\n      <td>optuna_lightgbm</td>\n      <td>fixed_lightgbm</td>\n      <td>mae_improvement_pp</td>\n      <td>0.004300</td>\n      <td>-0.011822</td>\n      <td>0.020755</td>\n      <td>20</td>\n      <td>23167</td>\n      <td>6512</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>regression</td>\n      <td>2</td>\n      <td>optuna_xgboost</td>\n      <td>fixed_lightgbm</td>\n      <td>mae_improvement_pp</td>\n      <td>0.014385</td>\n      <td>0.001510</td>\n      <td>0.027680</td>\n      <td>20</td>\n      <td>23167</td>\n      <td>6512</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>regression</td>\n      <td>4</td>\n      <td>optuna_lightgbm</td>\n      <td>fixed_lightgbm</td>\n      <td>mae_improvement_pp</td>\n      <td>0.031902</td>\n      <td>-0.005864</td>\n      <td>0.067675</td>\n      <td>20</td>\n      <td>23167</td>\n      <td>6512</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>regression</td>\n      <td>4</td>\n      <td>optuna_xgboost</td>\n      <td>fixed_lightgbm</td>\n      <td>mae_improvement_pp</td>\n      <td>0.023449</td>\n      <td>-0.015493</td>\n      <td>0.061635</td>\n      <td>20</td>\n      <td>23167</td>\n      <td>6512</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>regression</td>\n      <td>6</td>\n      <td>optuna_lightgbm</td>\n      <td>fixed_lightgbm</td>\n      <td>mae_improvement_pp</td>\n      <td>0.013898</td>\n      <td>-0.030404</td>\n      <td>0.054432</td>\n      <td>20</td>\n      <td>23167</td>\n      <td>6512</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>regression</td>\n      <td>6</td>\n      <td>optuna_xgboost</td>\n      <td>fixed_lightgbm</td>\n      <td>mae_improvement_pp</td>\n      <td>0.032589</td>\n      <td>-0.005887</td>\n      <td>0.070692</td>\n      <td>20</td>\n      <td>23167</td>\n      <td>6512</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>regression</td>\n      <td>10</td>\n      <td>optuna_lightgbm</td>\n      <td>fixed_lightgbm</td>\n      <td>mae_improvement_pp</td>\n      <td>-0.011886</td>\n      <td>-0.041222</td>\n      <td>0.018081</td>\n      <td>20</td>\n      <td>23167</td>\n      <td>6512</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>regression</td>\n      <td>10</td>\n      <td>optuna_xgboost</td>\n      <td>fixed_lightgbm</td>\n      <td>mae_improvement_pp</td>\n      <td>0.001568</td>\n      <td>-0.039553</td>\n      <td>0.040628</td>\n      <td>20</td>\n      <td>23167</td>\n      <td>6512</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>regression</td>\n      <td>2</td>\n      <td>optuna_lightgbm</td>\n      <td>fixed_lightgbm</td>\n      <td>mae_improvement_pp</td>\n      <td>0.001960</td>\n      <td>-0.000952</td>\n      <td>0.004869</td>\n      <td>20</td>\n      <td>23167</td>\n      <td>6512</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>regression</td>\n      <td>2</td>\n      <td>optuna_xgboost</td>\n      <td>fixed_lightgbm</td>\n      <td>mae_improvement_pp</td>\n      <td>0.001108</td>\n      <td>-0.010386</td>\n      <td>0.012602</td>\n      <td>20</td>\n      <td>23167</td>\n      <td>6512</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>regression</td>\n      <td>4</td>\n      <td>optuna_lightgbm</td>\n      <td>fixed_lightgbm</td>\n      <td>mae_improvement_pp</td>\n      <td>-0.001562</td>\n      <td>-0.023582</td>\n      <td>0.020472</td>\n      <td>20</td>\n      <td>23167</td>\n      <td>6512</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>regression</td>\n      <td>4</td>\n      <td>optuna_xgboost</td>\n      <td>fixed_lightgbm</td>\n      <td>mae_improvement_pp</td>\n      <td>0.056293</td>\n      <td>0.016451</td>\n      <td>0.095195</td>\n      <td>20</td>\n      <td>23167</td>\n      <td>6512</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>regression</td>\n      <td>6</td>\n      <td>optuna_lightgbm</td>\n      <td>fixed_lightgbm</td>\n      <td>mae_improvement_pp</td>\n      <td>-0.019958</td>\n      <td>-0.044299</td>\n      <td>0.004740</td>\n      <td>20</td>\n      <td>23167</td>\n      <td>6512</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>regression</td>\n      <td>6</td>\n      <td>optuna_xgboost</td>\n      <td>fixed_lightgbm</td>\n      <td>mae_improvement_pp</td>\n      <td>0.005930</td>\n      <td>-0.026271</td>\n      <td>0.037449</td>\n      <td>20</td>\n      <td>23167</td>\n      <td>6512</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>regression</td>\n      <td>10</td>\n      <td>optuna_lightgbm</td>\n      <td>fixed_lightgbm</td>\n      <td>mae_improvement_pp</td>\n      <td>0.021075</td>\n      <td>-0.011128</td>\n      <td>0.053993</td>\n      <td>20</td>\n      <td>23167</td>\n      <td>6512</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>regression</td>\n      <td>10</td>\n      <td>optuna_xgboost</td>\n      <td>fixed_lightgbm</td>\n      <td>mae_improvement_pp</td>\n      <td>0.017818</td>\n      <td>-0.027922</td>\n      <td>0.061775</td>\n      <td>20</td>\n      <td>23167</td>\n      <td>6512</td>\n    </tr>\n  </tbody>\n</table>',
'')

show('historical_six_ply_ratio_thresholds_game',
'<table border="1" class="dataframe">\n  <thead>\n    <tr style="text-align: right;">\n      <th>archive_set</th>\n      <th>protocol</th>\n      <th>strategy</th>\n      <th>games</th>\n      <th>n</th>\n      <th>tn</th>\n      <th>fp</th>\n      <th>fn</th>\n      <th>tp</th>\n      <th>accuracy</th>\n      <th>balanced_accuracy</th>\n      <th>recovered_precision</th>\n      <th>recovered_recall</th>\n      <th>persistent_recall</th>\n      <th>predicted_recovered</th>\n      <th>actual_recovered</th>\n      <th>roc_auc</th>\n    </tr>\n  </thead>\n  <tbody>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game; nested inner cutoff selection</td>\n      <td>training_majority</td>\n      <td>6512</td>\n      <td>23167</td>\n      <td>14866</td>\n      <td>0</td>\n      <td>8301</td>\n      <td>0</td>\n      <td>0.641689</td>\n      <td>0.500000</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>1.000000</td>\n      <td>0</td>\n      <td>8301</td>\n      <td>unavailable / undefined</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game; nested inner cutoff selection</td>\n      <td>fixed_0.75</td>\n      <td>6512</td>\n      <td>23167</td>\n      <td>14698</td>\n      <td>168</td>\n      <td>8026</td>\n      <td>275</td>\n      <td>0.646307</td>\n      <td>0.510914</td>\n      <td>0.620767</td>\n      <td>0.033129</td>\n      <td>0.988699</td>\n      <td>443</td>\n      <td>8301</td>\n      <td>0.581350</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game; nested inner cutoff selection</td>\n      <td>tuned_accuracy</td>\n      <td>6512</td>\n      <td>23167</td>\n      <td>14726</td>\n      <td>140</td>\n      <td>8041</td>\n      <td>260</td>\n      <td>0.646868</td>\n      <td>0.510952</td>\n      <td>0.650000</td>\n      <td>0.031322</td>\n      <td>0.990583</td>\n      <td>400</td>\n      <td>8301</td>\n      <td>0.581350</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game; nested inner cutoff selection</td>\n      <td>tuned_balanced_accuracy</td>\n      <td>6512</td>\n      <td>23167</td>\n      <td>7072</td>\n      <td>7794</td>\n      <td>3000</td>\n      <td>5301</td>\n      <td>0.534079</td>\n      <td>0.557157</td>\n      <td>0.404811</td>\n      <td>0.638598</td>\n      <td>0.475716</td>\n      <td>13095</td>\n      <td>8301</td>\n      <td>0.581350</td>\n    </tr>\n  </tbody>\n</table>',
'Six-ply regression-derived labels. Continuous score = negative predicted ratio.')

show('historical_six_ply_classifier_game',
'<table border="1" class="dataframe">\n  <thead>\n    <tr style="text-align: right;">\n      <th>archive_set</th>\n      <th>protocol</th>\n      <th>strategy</th>\n      <th>games</th>\n      <th>n</th>\n      <th>tn</th>\n      <th>fp</th>\n      <th>fn</th>\n      <th>tp</th>\n      <th>accuracy</th>\n      <th>balanced_accuracy</th>\n      <th>recovered_precision</th>\n      <th>recovered_recall</th>\n      <th>recovered_f1</th>\n      <th>specificity</th>\n      <th>false_positive_rate</th>\n      <th>actual_recovered</th>\n      <th>predicted_recovered</th>\n      <th>roc_auc</th>\n      <th>average_precision</th>\n    </tr>\n  </thead>\n  <tbody>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game; nested classifier/threshold selection</td>\n      <td>training_majority</td>\n      <td>6512</td>\n      <td>23167</td>\n      <td>14866</td>\n      <td>0</td>\n      <td>8301</td>\n      <td>0</td>\n      <td>0.641689</td>\n      <td>0.500000</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n      <td>1.000000</td>\n      <td>0.000000</td>\n      <td>8301</td>\n      <td>0</td>\n      <td>unavailable / undefined</td>\n      <td>unavailable / undefined</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game; nested classifier/threshold selection</td>\n      <td>regression_fixed_0.75</td>\n      <td>6512</td>\n      <td>23167</td>\n      <td>14698</td>\n      <td>168</td>\n      <td>8026</td>\n      <td>275</td>\n      <td>0.646307</td>\n      <td>0.510914</td>\n      <td>0.620767</td>\n      <td>0.033129</td>\n      <td>0.062900</td>\n      <td>0.988699</td>\n      <td>0.011301</td>\n      <td>8301</td>\n      <td>443</td>\n      <td>0.581350</td>\n      <td>0.440031</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game; nested classifier/threshold selection</td>\n      <td>regression_tuned_accuracy</td>\n      <td>6512</td>\n      <td>23167</td>\n      <td>14726</td>\n      <td>140</td>\n      <td>8041</td>\n      <td>260</td>\n      <td>0.646868</td>\n      <td>0.510952</td>\n      <td>0.650000</td>\n      <td>0.031322</td>\n      <td>0.059763</td>\n      <td>0.990583</td>\n      <td>0.009417</td>\n      <td>8301</td>\n      <td>400</td>\n      <td>0.581350</td>\n      <td>0.440031</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game; nested classifier/threshold selection</td>\n      <td>regression_tuned_balanced_accuracy</td>\n      <td>6512</td>\n      <td>23167</td>\n      <td>7072</td>\n      <td>7794</td>\n      <td>3000</td>\n      <td>5301</td>\n      <td>0.534079</td>\n      <td>0.557157</td>\n      <td>0.404811</td>\n      <td>0.638598</td>\n      <td>0.495513</td>\n      <td>0.475716</td>\n      <td>0.524284</td>\n      <td>8301</td>\n      <td>13095</td>\n      <td>0.581350</td>\n      <td>0.440031</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game; nested classifier/threshold selection</td>\n      <td>classifier_0.5</td>\n      <td>6512</td>\n      <td>23167</td>\n      <td>14389</td>\n      <td>477</td>\n      <td>7721</td>\n      <td>580</td>\n      <td>0.646135</td>\n      <td>0.518892</td>\n      <td>0.548723</td>\n      <td>0.069871</td>\n      <td>0.123958</td>\n      <td>0.967913</td>\n      <td>0.032087</td>\n      <td>8301</td>\n      <td>1057</td>\n      <td>0.615719</td>\n      <td>0.459812</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game; nested classifier/threshold selection</td>\n      <td>classifier_tuned_accuracy</td>\n      <td>6512</td>\n      <td>23167</td>\n      <td>14787</td>\n      <td>79</td>\n      <td>8128</td>\n      <td>173</td>\n      <td>0.645746</td>\n      <td>0.507763</td>\n      <td>0.686508</td>\n      <td>0.020841</td>\n      <td>0.040454</td>\n      <td>0.994686</td>\n      <td>0.005314</td>\n      <td>8301</td>\n      <td>252</td>\n      <td>0.615719</td>\n      <td>0.459812</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game; nested classifier/threshold selection</td>\n      <td>classifier_tuned_balanced_accuracy</td>\n      <td>6512</td>\n      <td>23167</td>\n      <td>7748</td>\n      <td>7118</td>\n      <td>2969</td>\n      <td>5332</td>\n      <td>0.564596</td>\n      <td>0.581761</td>\n      <td>0.428273</td>\n      <td>0.642332</td>\n      <td>0.513903</td>\n      <td>0.521189</td>\n      <td>0.478811</td>\n      <td>8301</td>\n      <td>12450</td>\n      <td>0.615719</td>\n      <td>0.459812</td>\n    </tr>\n  </tbody>\n</table>',
'Earlier development protocol; different study from the matched final comparison.')

show('historical_six_ply_ratio_thresholds_player',
'<table border="1" class="dataframe">\n  <thead>\n    <tr style="text-align: right;">\n      <th>archive_set</th>\n      <th>protocol</th>\n      <th>strategy</th>\n      <th>games</th>\n      <th>n</th>\n      <th>tn</th>\n      <th>fp</th>\n      <th>fn</th>\n      <th>tp</th>\n      <th>accuracy</th>\n      <th>balanced_accuracy</th>\n      <th>recovered_precision</th>\n      <th>recovered_recall</th>\n      <th>persistent_recall</th>\n      <th>predicted_recovered</th>\n      <th>actual_recovered</th>\n      <th>roc_auc</th>\n    </tr>\n  </thead>\n  <tbody>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player; nested inner cutoff selection</td>\n      <td>training_majority</td>\n      <td>6512</td>\n      <td>23167</td>\n      <td>14866</td>\n      <td>0</td>\n      <td>8301</td>\n      <td>0</td>\n      <td>0.641689</td>\n      <td>0.500000</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>1.000000</td>\n      <td>0</td>\n      <td>8301</td>\n      <td>unavailable / undefined</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player; nested inner cutoff selection</td>\n      <td>fixed_0.75</td>\n      <td>6512</td>\n      <td>23167</td>\n      <td>14669</td>\n      <td>197</td>\n      <td>8019</td>\n      <td>282</td>\n      <td>0.645358</td>\n      <td>0.510360</td>\n      <td>0.588727</td>\n      <td>0.033972</td>\n      <td>0.986748</td>\n      <td>479</td>\n      <td>8301</td>\n      <td>0.574481</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player; nested inner cutoff selection</td>\n      <td>tuned_accuracy</td>\n      <td>6512</td>\n      <td>23167</td>\n      <td>14676</td>\n      <td>190</td>\n      <td>8023</td>\n      <td>278</td>\n      <td>0.645487</td>\n      <td>0.510355</td>\n      <td>0.594017</td>\n      <td>0.033490</td>\n      <td>0.987219</td>\n      <td>468</td>\n      <td>8301</td>\n      <td>0.574481</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player; nested inner cutoff selection</td>\n      <td>tuned_balanced_accuracy</td>\n      <td>6512</td>\n      <td>23167</td>\n      <td>8433</td>\n      <td>6433</td>\n      <td>3895</td>\n      <td>4406</td>\n      <td>0.554193</td>\n      <td>0.549024</td>\n      <td>0.406495</td>\n      <td>0.530779</td>\n      <td>0.567268</td>\n      <td>10839</td>\n      <td>8301</td>\n      <td>0.574481</td>\n    </tr>\n  </tbody>\n</table>',
'Six-ply regression-derived labels. Continuous score = negative predicted ratio.')

show('historical_six_ply_classifier_player',
'<table border="1" class="dataframe">\n  <thead>\n    <tr style="text-align: right;">\n      <th>archive_set</th>\n      <th>protocol</th>\n      <th>strategy</th>\n      <th>games</th>\n      <th>n</th>\n      <th>tn</th>\n      <th>fp</th>\n      <th>fn</th>\n      <th>tp</th>\n      <th>accuracy</th>\n      <th>balanced_accuracy</th>\n      <th>recovered_precision</th>\n      <th>recovered_recall</th>\n      <th>recovered_f1</th>\n      <th>specificity</th>\n      <th>false_positive_rate</th>\n      <th>actual_recovered</th>\n      <th>predicted_recovered</th>\n      <th>roc_auc</th>\n      <th>average_precision</th>\n    </tr>\n  </thead>\n  <tbody>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player; nested classifier/threshold selection</td>\n      <td>training_majority</td>\n      <td>6512</td>\n      <td>23167</td>\n      <td>14866</td>\n      <td>0</td>\n      <td>8301</td>\n      <td>0</td>\n      <td>0.641689</td>\n      <td>0.500000</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n      <td>1.000000</td>\n      <td>0.000000</td>\n      <td>8301</td>\n      <td>0</td>\n      <td>unavailable / undefined</td>\n      <td>unavailable / undefined</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player; nested classifier/threshold selection</td>\n      <td>regression_fixed_0.75</td>\n      <td>6512</td>\n      <td>23167</td>\n      <td>14669</td>\n      <td>197</td>\n      <td>8019</td>\n      <td>282</td>\n      <td>0.645358</td>\n      <td>0.510360</td>\n      <td>0.588727</td>\n      <td>0.033972</td>\n      <td>0.064237</td>\n      <td>0.986748</td>\n      <td>0.013252</td>\n      <td>8301</td>\n      <td>479</td>\n      <td>0.574481</td>\n      <td>0.434552</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player; nested classifier/threshold selection</td>\n      <td>regression_tuned_accuracy</td>\n      <td>6512</td>\n      <td>23167</td>\n      <td>14676</td>\n      <td>190</td>\n      <td>8023</td>\n      <td>278</td>\n      <td>0.645487</td>\n      <td>0.510355</td>\n      <td>0.594017</td>\n      <td>0.033490</td>\n      <td>0.063405</td>\n      <td>0.987219</td>\n      <td>0.012781</td>\n      <td>8301</td>\n      <td>468</td>\n      <td>0.574481</td>\n      <td>0.434552</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player; nested classifier/threshold selection</td>\n      <td>regression_tuned_balanced_accuracy</td>\n      <td>6512</td>\n      <td>23167</td>\n      <td>8433</td>\n      <td>6433</td>\n      <td>3895</td>\n      <td>4406</td>\n      <td>0.554193</td>\n      <td>0.549024</td>\n      <td>0.406495</td>\n      <td>0.530779</td>\n      <td>0.460397</td>\n      <td>0.567268</td>\n      <td>0.432732</td>\n      <td>8301</td>\n      <td>10839</td>\n      <td>0.574481</td>\n      <td>0.434552</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player; nested classifier/threshold selection</td>\n      <td>classifier_0.5</td>\n      <td>6512</td>\n      <td>23167</td>\n      <td>14109</td>\n      <td>757</td>\n      <td>7463</td>\n      <td>838</td>\n      <td>0.645185</td>\n      <td>0.525015</td>\n      <td>0.525392</td>\n      <td>0.100952</td>\n      <td>0.169361</td>\n      <td>0.949078</td>\n      <td>0.050922</td>\n      <td>8301</td>\n      <td>1595</td>\n      <td>0.613388</td>\n      <td>0.457854</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player; nested classifier/threshold selection</td>\n      <td>classifier_tuned_accuracy</td>\n      <td>6512</td>\n      <td>23167</td>\n      <td>14428</td>\n      <td>438</td>\n      <td>7746</td>\n      <td>555</td>\n      <td>0.646739</td>\n      <td>0.518698</td>\n      <td>0.558912</td>\n      <td>0.066859</td>\n      <td>0.119432</td>\n      <td>0.970537</td>\n      <td>0.029463</td>\n      <td>8301</td>\n      <td>993</td>\n      <td>0.613388</td>\n      <td>0.457854</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player; nested classifier/threshold selection</td>\n      <td>classifier_tuned_balanced_accuracy</td>\n      <td>6512</td>\n      <td>23167</td>\n      <td>7797</td>\n      <td>7069</td>\n      <td>3036</td>\n      <td>5265</td>\n      <td>0.563819</td>\n      <td>0.579373</td>\n      <td>0.426869</td>\n      <td>0.634261</td>\n      <td>0.510298</td>\n      <td>0.524485</td>\n      <td>0.475515</td>\n      <td>8301</td>\n      <td>12334</td>\n      <td>0.613388</td>\n      <td>0.457854</td>\n    </tr>\n  </tbody>\n</table>',
'Earlier development protocol; different study from the matched final comparison.')

show('primary_regression_robustness_intervals',
'<table border="1" class="dataframe">\n  <thead>\n    <tr style="text-align: right;">\n      <th>archive_set</th>\n      <th>protocol</th>\n      <th>task</th>\n      <th>target</th>\n      <th>model</th>\n      <th>comparator</th>\n      <th>metric</th>\n      <th>estimate</th>\n      <th>lower95</th>\n      <th>upper95</th>\n      <th>initial_damage_threshold</th>\n      <th>n</th>\n      <th>games</th>\n    </tr>\n  </thead>\n  <tbody>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>regression</td>\n      <td>2</td>\n      <td>fixed_lightgbm</td>\n      <td>no_change</td>\n      <td>mae_improvement_pp</td>\n      <td>0.014781</td>\n      <td>0.007538</td>\n      <td>0.022332</td>\n      <td>25</td>\n      <td>17235</td>\n      <td>5972</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>regression</td>\n      <td>2</td>\n      <td>fixed_lightgbm</td>\n      <td>no_change</td>\n      <td>mae_improvement_pp</td>\n      <td>0.010758</td>\n      <td>0.002184</td>\n      <td>0.019253</td>\n      <td>30</td>\n      <td>12887</td>\n      <td>5364</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>regression</td>\n      <td>4</td>\n      <td>fixed_lightgbm</td>\n      <td>no_change</td>\n      <td>mae_improvement_pp</td>\n      <td>0.111122</td>\n      <td>0.044800</td>\n      <td>0.177206</td>\n      <td>25</td>\n      <td>17235</td>\n      <td>5972</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>regression</td>\n      <td>4</td>\n      <td>fixed_lightgbm</td>\n      <td>no_change</td>\n      <td>mae_improvement_pp</td>\n      <td>0.123774</td>\n      <td>0.042409</td>\n      <td>0.199742</td>\n      <td>30</td>\n      <td>12887</td>\n      <td>5364</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>regression</td>\n      <td>6</td>\n      <td>fixed_lightgbm</td>\n      <td>no_change</td>\n      <td>mae_improvement_pp</td>\n      <td>0.291819</td>\n      <td>0.183999</td>\n      <td>0.392633</td>\n      <td>25</td>\n      <td>17235</td>\n      <td>5972</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>regression</td>\n      <td>6</td>\n      <td>fixed_lightgbm</td>\n      <td>no_change</td>\n      <td>mae_improvement_pp</td>\n      <td>0.331890</td>\n      <td>0.207303</td>\n      <td>0.461288</td>\n      <td>30</td>\n      <td>12887</td>\n      <td>5364</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>regression</td>\n      <td>10</td>\n      <td>fixed_lightgbm</td>\n      <td>no_change</td>\n      <td>mae_improvement_pp</td>\n      <td>0.654506</td>\n      <td>0.503850</td>\n      <td>0.801621</td>\n      <td>25</td>\n      <td>17235</td>\n      <td>5972</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>regression</td>\n      <td>10</td>\n      <td>fixed_lightgbm</td>\n      <td>no_change</td>\n      <td>mae_improvement_pp</td>\n      <td>0.745551</td>\n      <td>0.562922</td>\n      <td>0.923600</td>\n      <td>30</td>\n      <td>12887</td>\n      <td>5364</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>regression</td>\n      <td>2</td>\n      <td>fixed_lightgbm</td>\n      <td>no_change</td>\n      <td>mae_improvement_pp</td>\n      <td>0.014683</td>\n      <td>0.007605</td>\n      <td>0.021528</td>\n      <td>25</td>\n      <td>17235</td>\n      <td>5972</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>regression</td>\n      <td>2</td>\n      <td>fixed_lightgbm</td>\n      <td>no_change</td>\n      <td>mae_improvement_pp</td>\n      <td>0.011013</td>\n      <td>0.003448</td>\n      <td>0.018623</td>\n      <td>30</td>\n      <td>12887</td>\n      <td>5364</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>regression</td>\n      <td>4</td>\n      <td>fixed_lightgbm</td>\n      <td>no_change</td>\n      <td>mae_improvement_pp</td>\n      <td>0.072789</td>\n      <td>0.019693</td>\n      <td>0.126270</td>\n      <td>25</td>\n      <td>17235</td>\n      <td>5972</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>regression</td>\n      <td>4</td>\n      <td>fixed_lightgbm</td>\n      <td>no_change</td>\n      <td>mae_improvement_pp</td>\n      <td>0.070022</td>\n      <td>0.003190</td>\n      <td>0.131712</td>\n      <td>30</td>\n      <td>12887</td>\n      <td>5364</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>regression</td>\n      <td>6</td>\n      <td>fixed_lightgbm</td>\n      <td>no_change</td>\n      <td>mae_improvement_pp</td>\n      <td>0.253776</td>\n      <td>0.155202</td>\n      <td>0.346863</td>\n      <td>25</td>\n      <td>17235</td>\n      <td>5972</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>regression</td>\n      <td>6</td>\n      <td>fixed_lightgbm</td>\n      <td>no_change</td>\n      <td>mae_improvement_pp</td>\n      <td>0.274147</td>\n      <td>0.159845</td>\n      <td>0.393544</td>\n      <td>30</td>\n      <td>12887</td>\n      <td>5364</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>regression</td>\n      <td>10</td>\n      <td>fixed_lightgbm</td>\n      <td>no_change</td>\n      <td>mae_improvement_pp</td>\n      <td>0.550181</td>\n      <td>0.398612</td>\n      <td>0.701457</td>\n      <td>25</td>\n      <td>17235</td>\n      <td>5972</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>regression</td>\n      <td>10</td>\n      <td>fixed_lightgbm</td>\n      <td>no_change</td>\n      <td>mae_improvement_pp</td>\n      <td>0.614596</td>\n      <td>0.433382</td>\n      <td>0.795106</td>\n      <td>30</td>\n      <td>12887</td>\n      <td>5364</td>\n    </tr>\n  </tbody>\n</table>',
'')

show('ten_ply_maia_robustness',
'<table border="1" class="dataframe">\n  <thead>\n    <tr style="text-align: right;">\n      <th>archive_set</th>\n      <th>target</th>\n      <th>protocol</th>\n      <th>initial_damage_threshold</th>\n      <th>variant</th>\n      <th>n</th>\n      <th>games</th>\n      <th>improvement_over_fixed_pp</th>\n      <th>improvement_lower95</th>\n      <th>improvement_upper95</th>\n      <th>improvement_percent</th>\n    </tr>\n  </thead>\n  <tbody>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>opponent_reply_regret_pp</td>\n      <td>game</td>\n      <td>20</td>\n      <td>plus_maia</td>\n      <td>15851</td>\n      <td>4472</td>\n      <td>2.134172</td>\n      <td>1.962499</td>\n      <td>2.314839</td>\n      <td>15.347803</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>opponent_reply_regret_pp</td>\n      <td>game</td>\n      <td>20</td>\n      <td>plus_maia_lower</td>\n      <td>15851</td>\n      <td>4472</td>\n      <td>2.125174</td>\n      <td>1.958269</td>\n      <td>2.306028</td>\n      <td>15.283090</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>opponent_reply_regret_pp</td>\n      <td>game</td>\n      <td>20</td>\n      <td>plus_maia_upper</td>\n      <td>15851</td>\n      <td>4472</td>\n      <td>2.125458</td>\n      <td>1.956533</td>\n      <td>2.306611</td>\n      <td>15.285139</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>opponent_reply_regret_pp</td>\n      <td>game</td>\n      <td>25</td>\n      <td>plus_maia</td>\n      <td>11774</td>\n      <td>4092</td>\n      <td>2.333614</td>\n      <td>2.116986</td>\n      <td>2.555035</td>\n      <td>15.449651</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>opponent_reply_regret_pp</td>\n      <td>game</td>\n      <td>25</td>\n      <td>plus_maia_lower</td>\n      <td>11774</td>\n      <td>4092</td>\n      <td>2.327746</td>\n      <td>2.110204</td>\n      <td>2.552588</td>\n      <td>15.410805</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>opponent_reply_regret_pp</td>\n      <td>game</td>\n      <td>25</td>\n      <td>plus_maia_upper</td>\n      <td>11774</td>\n      <td>4092</td>\n      <td>2.320711</td>\n      <td>2.102832</td>\n      <td>2.540647</td>\n      <td>15.364231</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>opponent_reply_regret_pp</td>\n      <td>game</td>\n      <td>30</td>\n      <td>plus_maia</td>\n      <td>8769</td>\n      <td>3678</td>\n      <td>2.427059</td>\n      <td>2.148193</td>\n      <td>2.713231</td>\n      <td>14.946657</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>opponent_reply_regret_pp</td>\n      <td>game</td>\n      <td>30</td>\n      <td>plus_maia_lower</td>\n      <td>8769</td>\n      <td>3678</td>\n      <td>2.419264</td>\n      <td>2.146451</td>\n      <td>2.705432</td>\n      <td>14.898653</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>opponent_reply_regret_pp</td>\n      <td>game</td>\n      <td>30</td>\n      <td>plus_maia_upper</td>\n      <td>8769</td>\n      <td>3678</td>\n      <td>2.406386</td>\n      <td>2.138604</td>\n      <td>2.692694</td>\n      <td>14.819346</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>opponent_reply_regret_pp</td>\n      <td>player</td>\n      <td>20</td>\n      <td>plus_maia</td>\n      <td>15851</td>\n      <td>4472</td>\n      <td>2.046771</td>\n      <td>1.874042</td>\n      <td>2.222993</td>\n      <td>14.758879</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>opponent_reply_regret_pp</td>\n      <td>player</td>\n      <td>20</td>\n      <td>plus_maia_lower</td>\n      <td>15851</td>\n      <td>4472</td>\n      <td>2.058452</td>\n      <td>1.885878</td>\n      <td>2.231979</td>\n      <td>14.843110</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>opponent_reply_regret_pp</td>\n      <td>player</td>\n      <td>20</td>\n      <td>plus_maia_upper</td>\n      <td>15851</td>\n      <td>4472</td>\n      <td>2.060968</td>\n      <td>1.888227</td>\n      <td>2.238541</td>\n      <td>14.861252</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>opponent_reply_regret_pp</td>\n      <td>player</td>\n      <td>25</td>\n      <td>plus_maia</td>\n      <td>11774</td>\n      <td>4092</td>\n      <td>2.234299</td>\n      <td>2.020761</td>\n      <td>2.449482</td>\n      <td>14.841304</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>opponent_reply_regret_pp</td>\n      <td>player</td>\n      <td>25</td>\n      <td>plus_maia_lower</td>\n      <td>11774</td>\n      <td>4092</td>\n      <td>2.241854</td>\n      <td>2.028663</td>\n      <td>2.457517</td>\n      <td>14.891489</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>opponent_reply_regret_pp</td>\n      <td>player</td>\n      <td>25</td>\n      <td>plus_maia_upper</td>\n      <td>11774</td>\n      <td>4092</td>\n      <td>2.253483</td>\n      <td>2.038236</td>\n      <td>2.464491</td>\n      <td>14.968735</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>opponent_reply_regret_pp</td>\n      <td>player</td>\n      <td>30</td>\n      <td>plus_maia</td>\n      <td>8769</td>\n      <td>3678</td>\n      <td>2.329127</td>\n      <td>2.052894</td>\n      <td>2.614639</td>\n      <td>14.382583</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>opponent_reply_regret_pp</td>\n      <td>player</td>\n      <td>30</td>\n      <td>plus_maia_lower</td>\n      <td>8769</td>\n      <td>3678</td>\n      <td>2.329088</td>\n      <td>2.060880</td>\n      <td>2.615979</td>\n      <td>14.382343</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>opponent_reply_regret_pp</td>\n      <td>player</td>\n      <td>30</td>\n      <td>plus_maia_upper</td>\n      <td>8769</td>\n      <td>3678</td>\n      <td>2.335259</td>\n      <td>2.062433</td>\n      <td>2.623987</td>\n      <td>14.420449</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>change_in_damage_10ply</td>\n      <td>game</td>\n      <td>20</td>\n      <td>plus_maia</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>0.198204</td>\n      <td>0.116301</td>\n      <td>0.281079</td>\n      <td>0.864371</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>change_in_damage_10ply</td>\n      <td>game</td>\n      <td>20</td>\n      <td>plus_maia_lower</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>0.208357</td>\n      <td>0.126772</td>\n      <td>0.289977</td>\n      <td>0.908649</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>change_in_damage_10ply</td>\n      <td>game</td>\n      <td>20</td>\n      <td>plus_maia_upper</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>0.229206</td>\n      <td>0.146259</td>\n      <td>0.310162</td>\n      <td>0.999573</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>change_in_damage_10ply</td>\n      <td>game</td>\n      <td>25</td>\n      <td>plus_maia</td>\n      <td>11867</td>\n      <td>4105</td>\n      <td>0.242810</td>\n      <td>0.146495</td>\n      <td>0.342678</td>\n      <td>1.049011</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>change_in_damage_10ply</td>\n      <td>game</td>\n      <td>25</td>\n      <td>plus_maia_lower</td>\n      <td>11867</td>\n      <td>4105</td>\n      <td>0.252146</td>\n      <td>0.152933</td>\n      <td>0.351975</td>\n      <td>1.089345</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>change_in_damage_10ply</td>\n      <td>game</td>\n      <td>25</td>\n      <td>plus_maia_upper</td>\n      <td>11867</td>\n      <td>4105</td>\n      <td>0.284650</td>\n      <td>0.184087</td>\n      <td>0.380725</td>\n      <td>1.229776</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>change_in_damage_10ply</td>\n      <td>game</td>\n      <td>30</td>\n      <td>plus_maia</td>\n      <td>8852</td>\n      <td>3695</td>\n      <td>0.247210</td>\n      <td>0.127173</td>\n      <td>0.367996</td>\n      <td>1.062916</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>change_in_damage_10ply</td>\n      <td>game</td>\n      <td>30</td>\n      <td>plus_maia_lower</td>\n      <td>8852</td>\n      <td>3695</td>\n      <td>0.253233</td>\n      <td>0.133851</td>\n      <td>0.375232</td>\n      <td>1.088814</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>change_in_damage_10ply</td>\n      <td>game</td>\n      <td>30</td>\n      <td>plus_maia_upper</td>\n      <td>8852</td>\n      <td>3695</td>\n      <td>0.281998</td>\n      <td>0.157299</td>\n      <td>0.406796</td>\n      <td>1.212494</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>change_in_damage_10ply</td>\n      <td>player</td>\n      <td>20</td>\n      <td>plus_maia</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>0.233147</td>\n      <td>0.155220</td>\n      <td>0.308837</td>\n      <td>1.011178</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>change_in_damage_10ply</td>\n      <td>player</td>\n      <td>20</td>\n      <td>plus_maia_lower</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>0.240490</td>\n      <td>0.163678</td>\n      <td>0.314116</td>\n      <td>1.043025</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>change_in_damage_10ply</td>\n      <td>player</td>\n      <td>20</td>\n      <td>plus_maia_upper</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>0.232118</td>\n      <td>0.156554</td>\n      <td>0.305632</td>\n      <td>1.006715</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>change_in_damage_10ply</td>\n      <td>player</td>\n      <td>25</td>\n      <td>plus_maia</td>\n      <td>11867</td>\n      <td>4105</td>\n      <td>0.273619</td>\n      <td>0.181736</td>\n      <td>0.369456</td>\n      <td>1.174942</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>change_in_damage_10ply</td>\n      <td>player</td>\n      <td>25</td>\n      <td>plus_maia_lower</td>\n      <td>11867</td>\n      <td>4105</td>\n      <td>0.282441</td>\n      <td>0.191399</td>\n      <td>0.378652</td>\n      <td>1.212823</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>change_in_damage_10ply</td>\n      <td>player</td>\n      <td>25</td>\n      <td>plus_maia_upper</td>\n      <td>11867</td>\n      <td>4105</td>\n      <td>0.269153</td>\n      <td>0.179469</td>\n      <td>0.362412</td>\n      <td>1.155762</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>change_in_damage_10ply</td>\n      <td>player</td>\n      <td>30</td>\n      <td>plus_maia</td>\n      <td>8852</td>\n      <td>3695</td>\n      <td>0.280572</td>\n      <td>0.171118</td>\n      <td>0.387439</td>\n      <td>1.196929</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>change_in_damage_10ply</td>\n      <td>player</td>\n      <td>30</td>\n      <td>plus_maia_lower</td>\n      <td>8852</td>\n      <td>3695</td>\n      <td>0.288108</td>\n      <td>0.181928</td>\n      <td>0.398554</td>\n      <td>1.229076</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>change_in_damage_10ply</td>\n      <td>player</td>\n      <td>30</td>\n      <td>plus_maia_upper</td>\n      <td>8852</td>\n      <td>3695</td>\n      <td>0.263767</td>\n      <td>0.157318</td>\n      <td>0.375346</td>\n      <td>1.125238</td>\n    </tr>\n  </tbody>\n</table>',
'')

show('within_10pp_diagnostic',
'<table border="1" class="dataframe">\n  <thead>\n    <tr style="text-align: right;">\n      <th>archive_set</th>\n      <th>protocol</th>\n      <th>variant</th>\n      <th>n</th>\n      <th>games</th>\n      <th>within_10pp_count</th>\n      <th>outside_10pp_count</th>\n      <th>within_10pp_percent</th>\n    </tr>\n  </thead>\n  <tbody>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>original_plus_maia_probability</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>6483</td>\n      <td>9470</td>\n      <td>40.638124</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>plus_continuous_lower</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>6419</td>\n      <td>9534</td>\n      <td>40.236946</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>plus_continuous_midpoint</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>6420</td>\n      <td>9533</td>\n      <td>40.243214</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>plus_continuous_upper</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>6383</td>\n      <td>9570</td>\n      <td>40.011283</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>training_mean</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>2648</td>\n      <td>13305</td>\n      <td>16.598759</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>training_median</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>6916</td>\n      <td>9037</td>\n      <td>43.352348</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>game</td>\n      <td>zero</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>6890</td>\n      <td>9063</td>\n      <td>43.189369</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>original_plus_maia_probability</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>6422</td>\n      <td>9531</td>\n      <td>40.255751</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>plus_continuous_lower</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>6344</td>\n      <td>9609</td>\n      <td>39.766815</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>plus_continuous_midpoint</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>6345</td>\n      <td>9608</td>\n      <td>39.773083</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>plus_continuous_upper</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>6356</td>\n      <td>9597</td>\n      <td>39.842036</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>training_mean</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>2741</td>\n      <td>13212</td>\n      <td>17.181721</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>training_median</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>6916</td>\n      <td>9037</td>\n      <td>43.352348</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>player</td>\n      <td>zero</td>\n      <td>15953</td>\n      <td>4485</td>\n      <td>6890</td>\n      <td>9063</td>\n      <td>43.189369</td>\n    </tr>\n  </tbody>\n</table>',
'Absolute-error tolerance rate; separate from MAE and classification accuracy.')


**Optuna regression**

Gains compare with no-change; nested selection only for Optuna variants.

archive_set,protocol,target,model,n,games,mae,spearman,estimate_mae_improvement_pp,lower95_mae_improvement_pp,upper95_mae_improvement_pp
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,2,fixed_lightgbm,23167,6512,12.558943,0.116483,0.017615,0.011389,0.023895
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,2,optuna_lightgbm,23167,6512,12.554643,0.117968,0.021915,0.004240,0.039136
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,2,optuna_xgboost,23167,6512,12.544558,0.116243,0.032000,0.018283,0.046740
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,4,fixed_lightgbm,23167,6512,16.843501,0.104874,0.092363,0.040911,0.146109
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,4,optuna_lightgbm,23167,6512,16.811599,0.098739,0.124265,0.060563,0.187495
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,4,optuna_xgboost,23167,6512,16.820052,0.098783,0.115812,0.058706,0.171249
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,6,fixed_lightgbm,23167,6512,19.579173,0.099988,0.255319,0.172802,0.338477
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,6,optuna_lightgbm,23167,6512,19.565276,0.100014,0.269216,0.184484,0.349004
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,6,optuna_xgboost,23167,6512,19.546584,0.101415,0.287908,0.209031,0.364718
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,10,fixed_lightgbm,23167,6512,23.049053,0.123120,0.586358,0.462247,0.707210


**Optuna vs fixed intervals**

archive_set,protocol,task,target,model,comparator,metric,estimate,lower95,upper95,initial_damage_threshold,n,games
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,regression,2,optuna_lightgbm,fixed_lightgbm,mae_improvement_pp,0.004300,-0.011822,0.020755,20,23167,6512
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,regression,2,optuna_xgboost,fixed_lightgbm,mae_improvement_pp,0.014385,0.001510,0.027680,20,23167,6512
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,regression,4,optuna_lightgbm,fixed_lightgbm,mae_improvement_pp,0.031902,-0.005864,0.067675,20,23167,6512
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,regression,4,optuna_xgboost,fixed_lightgbm,mae_improvement_pp,0.023449,-0.015493,0.061635,20,23167,6512
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,regression,6,optuna_lightgbm,fixed_lightgbm,mae_improvement_pp,0.013898,-0.030404,0.054432,20,23167,6512
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,regression,6,optuna_xgboost,fixed_lightgbm,mae_improvement_pp,0.032589,-0.005887,0.070692,20,23167,6512
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,regression,10,optuna_lightgbm,fixed_lightgbm,mae_improvement_pp,-0.011886,-0.041222,0.018081,20,23167,6512
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",game,regression,10,optuna_xgboost,fixed_lightgbm,mae_improvement_pp,0.001568,-0.039553,0.040628,20,23167,6512
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",player,regression,2,optuna_lightgbm,fixed_lightgbm,mae_improvement_pp,0.001960,-0.000952,0.004869,20,23167,6512
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",player,regression,2,optuna_xgboost,fixed_lightgbm,mae_improvement_pp,0.001108,-0.010386,0.012602,20,23167,6512


**Historical six ply ratio thresholds game**

... (14 outputs left out)

archive_set,target,protocol,initial_damage_threshold,variant,n,games,improvement_over_fixed_pp,improvement_lower95,improvement_upper95,improvement_percent
"2017-04, 2017-06, 2017-07, 2018-02",opponent_reply_regret_pp,game,20,plus_maia,15851,4472,2.134172,1.962499,2.314839,15.347803
"2017-04, 2017-06, 2017-07, 2018-02",opponent_reply_regret_pp,game,20,plus_maia_lower,15851,4472,2.125174,1.958269,2.306028,15.283090
"2017-04, 2017-06, 2017-07, 2018-02",opponent_reply_regret_pp,game,20,plus_maia_upper,15851,4472,2.125458,1.956533,2.306611,15.285139
"2017-04, 2017-06, 2017-07, 2018-02",opponent_reply_regret_pp,game,25,plus_maia,11774,4092,2.333614,2.116986,2.555035,15.449651
"2017-04, 2017-06, 2017-07, 2018-02",opponent_reply_regret_pp,game,25,plus_maia_lower,11774,4092,2.327746,2.110204,2.552588,15.410805
"2017-04, 2017-06, 2017-07, 2018-02",opponent_reply_regret_pp,game,25,plus_maia_upper,11774,4092,2.320711,2.102832,2.540647,15.364231
"2017-04, 2017-06, 2017-07, 2018-02",opponent_reply_regret_pp,game,30,plus_maia,8769,3678,2.427059,2.148193,2.713231,14.946657
"2017-04, 2017-06, 2017-07, 2018-02",opponent_reply_regret_pp,game,30,plus_maia_lower,8769,3678,2.419264,2.146451,2.705432,14.898653
"2017-04, 2017-06, 2017-07, 2018-02",opponent_reply_regret_pp,game,30,plus_maia_upper,8769,3678,2.406386,2.138604,2.692694,14.819346
"2017-04, 2017-06, 2017-07, 2018-02",opponent_reply_regret_pp,player,20,plus_maia,15851,4472,2.046771,1.874042,2.222993,14.758879


**Within 10pp diagnostic**

Absolute-error tolerance rate; separate from MAE and classification accuracy.

archive_set,protocol,variant,n,games,within_10pp_count,outside_10pp_count,within_10pp_percent
"2017-04, 2017-06, 2017-07, 2018-02",game,original_plus_maia_probability,15953,4485,6483,9470,40.638124
"2017-04, 2017-06, 2017-07, 2018-02",game,plus_continuous_lower,15953,4485,6419,9534,40.236946
"2017-04, 2017-06, 2017-07, 2018-02",game,plus_continuous_midpoint,15953,4485,6420,9533,40.243214
"2017-04, 2017-06, 2017-07, 2018-02",game,plus_continuous_upper,15953,4485,6383,9570,40.011283
"2017-04, 2017-06, 2017-07, 2018-02",game,training_mean,15953,4485,2648,13305,16.598759
"2017-04, 2017-06, 2017-07, 2018-02",game,training_median,15953,4485,6916,9037,43.352348
"2017-04, 2017-06, 2017-07, 2018-02",game,zero,15953,4485,6890,9063,43.189369
"2017-04, 2017-06, 2017-07, 2018-02",player,original_plus_maia_probability,15953,4485,6422,9531,40.255751
"2017-04, 2017-06, 2017-07, 2018-02",player,plus_continuous_lower,15953,4485,6344,9609,39.766815
"2017-04, 2017-06, 2017-07, 2018-02",player,plus_continuous_midpoint,15953,4485,6345,9608,39.773083


### Earlier evaluations and policy integration checks

The policy pilot sampled 100 mistakes per archive. Its 491 valid replies across five months are not population-weighted. The four-month subset predating documented pretraining contains 392 valid replies from 364 games; eight of its 400 sampled mistakes have missing replies. The uniform legal-move comparison uses that same labeled subset. Budget simulations assess numerical integration error, not human-behavior or engine uncertainty.

The earlier signed reply targets are `W_actual_reply - W_best_reply` (regret) and `W_actual_reply - W_after_mistake` (missed opportunity). Positive values leave the original mistake-maker better off; negative values are retained. These are outcomes, not predictors.


In [15]:
show('pilot_policy_all_five_exploratory',
'<table border="1" class="dataframe">\n  <thead>\n    <tr style="text-align: right;">\n      <th>archive_set</th>\n      <th>scope</th>\n      <th>protocol</th>\n      <th>method</th>\n      <th>games</th>\n      <th>n</th>\n      <th>event_prevalence</th>\n      <th>mean_probability</th>\n      <th>brier</th>\n      <th>log_loss</th>\n      <th>roc_auc</th>\n      <th>average_precision</th>\n    </tr>\n  </thead>\n  <tbody>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>all_five_exploratory</td>\n      <td>game</td>\n      <td>calibrated_policy_mass</td>\n      <td>460</td>\n      <td>491</td>\n      <td>0.578411</td>\n      <td>0.580714</td>\n      <td>0.170037</td>\n      <td>0.510348</td>\n      <td>0.817786</td>\n      <td>0.854959</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>all_five_exploratory</td>\n      <td>game</td>\n      <td>favorite_acceptable_only</td>\n      <td>460</td>\n      <td>491</td>\n      <td>0.578411</td>\n      <td>0.458582</td>\n      <td>0.235718</td>\n      <td>0.806235</td>\n      <td>0.739845</td>\n      <td>0.809026</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>all_five_exploratory</td>\n      <td>game</td>\n      <td>raw_policy_mass</td>\n      <td>460</td>\n      <td>491</td>\n      <td>0.578411</td>\n      <td>0.577887</td>\n      <td>0.171374</td>\n      <td>0.521472</td>\n      <td>0.822498</td>\n      <td>0.858022</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>all_five_exploratory</td>\n      <td>game</td>\n      <td>training_prevalence</td>\n      <td>460</td>\n      <td>491</td>\n      <td>0.578411</td>\n      <td>0.557910</td>\n      <td>0.244475</td>\n      <td>0.682067</td>\n      <td>0.454132</td>\n      <td>0.556955</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>all_five_exploratory</td>\n      <td>game</td>\n      <td>uniform_legal</td>\n      <td>460</td>\n      <td>491</td>\n      <td>0.578411</td>\n      <td>0.087519</td>\n      <td>0.475940</td>\n      <td>1.639945</td>\n      <td>0.583325</td>\n      <td>0.689296</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>all_five_exploratory</td>\n      <td>player</td>\n      <td>calibrated_policy_mass</td>\n      <td>460</td>\n      <td>491</td>\n      <td>0.578411</td>\n      <td>0.580678</td>\n      <td>0.169955</td>\n      <td>0.510588</td>\n      <td>0.819198</td>\n      <td>0.856339</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>all_five_exploratory</td>\n      <td>player</td>\n      <td>favorite_acceptable_only</td>\n      <td>460</td>\n      <td>491</td>\n      <td>0.578411</td>\n      <td>0.458582</td>\n      <td>0.235718</td>\n      <td>0.806235</td>\n      <td>0.739845</td>\n      <td>0.809026</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>all_five_exploratory</td>\n      <td>player</td>\n      <td>raw_policy_mass</td>\n      <td>460</td>\n      <td>491</td>\n      <td>0.578411</td>\n      <td>0.577887</td>\n      <td>0.171374</td>\n      <td>0.521472</td>\n      <td>0.822498</td>\n      <td>0.858022</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>all_five_exploratory</td>\n      <td>player</td>\n      <td>training_prevalence</td>\n      <td>460</td>\n      <td>491</td>\n      <td>0.578411</td>\n      <td>0.561376</td>\n      <td>0.244457</td>\n      <td>0.682029</td>\n      <td>0.454132</td>\n      <td>0.556955</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>all_five_exploratory</td>\n      <td>player</td>\n      <td>uniform_legal</td>\n      <td>460</td>\n      <td>491</td>\n      <td>0.578411</td>\n      <td>0.087519</td>\n      <td>0.475940</td>\n      <td>1.639945</td>\n      <td>0.583325</td>\n      <td>0.689296</td>\n    </tr>\n  </tbody>\n</table>',
'Raw and calibrated probabilities are separate methods. February 2019 may overlap external pretraining.')

show('pilot_policy_predates_documented_pretraining',
'<table border="1" class="dataframe">\n  <thead>\n    <tr style="text-align: right;">\n      <th>archive_set</th>\n      <th>scope</th>\n      <th>protocol</th>\n      <th>method</th>\n      <th>games</th>\n      <th>n</th>\n      <th>event_prevalence</th>\n      <th>mean_probability</th>\n      <th>brier</th>\n      <th>log_loss</th>\n      <th>roc_auc</th>\n      <th>average_precision</th>\n    </tr>\n  </thead>\n  <tbody>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>predates_documented_pretraining</td>\n      <td>game</td>\n      <td>calibrated_policy_mass</td>\n      <td>364</td>\n      <td>392</td>\n      <td>0.579082</td>\n      <td>0.578955</td>\n      <td>0.162380</td>\n      <td>0.492913</td>\n      <td>0.834708</td>\n      <td>0.867408</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>predates_documented_pretraining</td>\n      <td>game</td>\n      <td>favorite_acceptable_only</td>\n      <td>364</td>\n      <td>392</td>\n      <td>0.579082</td>\n      <td>0.463382</td>\n      <td>0.226500</td>\n      <td>0.730574</td>\n      <td>0.757175</td>\n      <td>0.823835</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>predates_documented_pretraining</td>\n      <td>game</td>\n      <td>raw_policy_mass</td>\n      <td>364</td>\n      <td>392</td>\n      <td>0.579082</td>\n      <td>0.581494</td>\n      <td>0.162214</td>\n      <td>0.501027</td>\n      <td>0.838446</td>\n      <td>0.869184</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>predates_documented_pretraining</td>\n      <td>game</td>\n      <td>training_prevalence</td>\n      <td>364</td>\n      <td>392</td>\n      <td>0.579082</td>\n      <td>0.558512</td>\n      <td>0.244315</td>\n      <td>0.681743</td>\n      <td>0.489908</td>\n      <td>0.570123</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>predates_documented_pretraining</td>\n      <td>game</td>\n      <td>uniform_legal</td>\n      <td>364</td>\n      <td>392</td>\n      <td>0.579082</td>\n      <td>0.091639</td>\n      <td>0.470740</td>\n      <td>1.615026</td>\n      <td>0.606982</td>\n      <td>0.706064</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>predates_documented_pretraining</td>\n      <td>player</td>\n      <td>calibrated_policy_mass</td>\n      <td>364</td>\n      <td>392</td>\n      <td>0.579082</td>\n      <td>0.582853</td>\n      <td>0.162025</td>\n      <td>0.493124</td>\n      <td>0.835082</td>\n      <td>0.867615</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>predates_documented_pretraining</td>\n      <td>player</td>\n      <td>favorite_acceptable_only</td>\n      <td>364</td>\n      <td>392</td>\n      <td>0.579082</td>\n      <td>0.463382</td>\n      <td>0.226500</td>\n      <td>0.730574</td>\n      <td>0.757175</td>\n      <td>0.823835</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>predates_documented_pretraining</td>\n      <td>player</td>\n      <td>raw_policy_mass</td>\n      <td>364</td>\n      <td>392</td>\n      <td>0.579082</td>\n      <td>0.581494</td>\n      <td>0.162214</td>\n      <td>0.501027</td>\n      <td>0.838446</td>\n      <td>0.869184</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>predates_documented_pretraining</td>\n      <td>player</td>\n      <td>training_prevalence</td>\n      <td>364</td>\n      <td>392</td>\n      <td>0.579082</td>\n      <td>0.563119</td>\n      <td>0.244240</td>\n      <td>0.681593</td>\n      <td>0.461954</td>\n      <td>0.560543</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>predates_documented_pretraining</td>\n      <td>player</td>\n      <td>uniform_legal</td>\n      <td>364</td>\n      <td>392</td>\n      <td>0.579082</td>\n      <td>0.091639</td>\n      <td>0.470740</td>\n      <td>1.615026</td>\n      <td>0.606982</td>\n      <td>0.706064</td>\n    </tr>\n  </tbody>\n</table>',
'Four archives predating documented pretraining; raw and calibrated methods are separate.')

show('pilot_uniform_legal_reply_comparison',
'<table border="1" class="dataframe">\n  <thead>\n    <tr style="text-align: right;">\n      <th>archive_set</th>\n      <th>protocol</th>\n      <th>n</th>\n      <th>games</th>\n      <th>uniform_legal_brier</th>\n      <th>maia_brier</th>\n      <th>maia_brier_improvement</th>\n      <th>lower95</th>\n      <th>upper95</th>\n      <th>bootstrap_repeats</th>\n    </tr>\n  </thead>\n  <tbody>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>Pilot fixed policy; paired game bootstrap</td>\n      <td>392</td>\n      <td>364</td>\n      <td>0.470740</td>\n      <td>0.162214</td>\n      <td>0.308527</td>\n      <td>0.260233</td>\n      <td>0.363870</td>\n      <td>2000</td>\n    </tr>\n  </tbody>\n</table>',
'')

show('policy_integration_budget_simulation',
'<table border="1" class="dataframe">\n  <thead>\n    <tr style="text-align: right;">\n      <th>archive_set</th>\n      <th>protocol</th>\n      <th>scope</th>\n      <th>method</th>\n      <th>coverage</th>\n      <th>top_k</th>\n      <th>n</th>\n      <th>full_nonterminal_calls</th>\n      <th>selected_nonterminal_calls</th>\n      <th>calls_saved</th>\n      <th>calls_saved_percent</th>\n      <th>full_calls_after_legacy_cache</th>\n      <th>selected_calls_after_legacy_cache</th>\n      <th>legacy_aware_calls_saved</th>\n      <th>mean_evaluated_replies</th>\n      <th>mean_bound_width</th>\n      <th>median_bound_width</th>\n      <th>p95_bound_width</th>\n      <th>max_bound_width</th>\n      <th>mean_absolute_error</th>\n      <th>max_absolute_error</th>\n    </tr>\n  </thead>\n  <tbody>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>Offline simulation; all_400 or labeled_392 scope</td>\n      <td>all_400</td>\n      <td>top_1</td>\n      <td>unavailable / undefined</td>\n      <td>1.000000</td>\n      <td>400</td>\n      <td>12886</td>\n      <td>396</td>\n      <td>12490</td>\n      <td>96.926897</td>\n      <td>12286</td>\n      <td>111</td>\n      <td>12175</td>\n      <td>1.000000</td>\n      <td>0.382478</td>\n      <td>0.391841</td>\n      <td>0.745028</td>\n      <td>0.835280</td>\n      <td>0.127312</td>\n      <td>0.401755</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>Offline simulation; all_400 or labeled_392 scope</td>\n      <td>all_400</td>\n      <td>top_3</td>\n      <td>unavailable / undefined</td>\n      <td>3.000000</td>\n      <td>400</td>\n      <td>12886</td>\n      <td>1185</td>\n      <td>11701</td>\n      <td>90.803973</td>\n      <td>12286</td>\n      <td>728</td>\n      <td>11558</td>\n      <td>2.972500</td>\n      <td>0.146829</td>\n      <td>0.105791</td>\n      <td>0.419507</td>\n      <td>0.580629</td>\n      <td>0.056748</td>\n      <td>0.250220</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>Offline simulation; all_400 or labeled_392 scope</td>\n      <td>all_400</td>\n      <td>top_5</td>\n      <td>unavailable / undefined</td>\n      <td>5.000000</td>\n      <td>400</td>\n      <td>12886</td>\n      <td>1950</td>\n      <td>10936</td>\n      <td>84.867298</td>\n      <td>12286</td>\n      <td>1435</td>\n      <td>10851</td>\n      <td>4.885000</td>\n      <td>0.073912</td>\n      <td>0.043531</td>\n      <td>0.258630</td>\n      <td>0.433187</td>\n      <td>0.030988</td>\n      <td>0.190020</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>Offline simulation; all_400 or labeled_392 scope</td>\n      <td>all_400</td>\n      <td>top_8</td>\n      <td>unavailable / undefined</td>\n      <td>8.000000</td>\n      <td>400</td>\n      <td>12886</td>\n      <td>3065</td>\n      <td>9821</td>\n      <td>76.214496</td>\n      <td>12286</td>\n      <td>2511</td>\n      <td>9775</td>\n      <td>7.672500</td>\n      <td>0.034107</td>\n      <td>0.016092</td>\n      <td>0.139124</td>\n      <td>0.275302</td>\n      <td>0.014626</td>\n      <td>0.104172</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>Offline simulation; all_400 or labeled_392 scope</td>\n      <td>all_400</td>\n      <td>top_10</td>\n      <td>unavailable / undefined</td>\n      <td>10.000000</td>\n      <td>400</td>\n      <td>12886</td>\n      <td>3799</td>\n      <td>9087</td>\n      <td>70.518392</td>\n      <td>12286</td>\n      <td>3233</td>\n      <td>9053</td>\n      <td>9.507500</td>\n      <td>0.021986</td>\n      <td>0.008826</td>\n      <td>0.097465</td>\n      <td>0.202093</td>\n      <td>0.009588</td>\n      <td>0.084328</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>Offline simulation; all_400 or labeled_392 scope</td>\n      <td>all_400</td>\n      <td>top_15</td>\n      <td>unavailable / undefined</td>\n      <td>15.000000</td>\n      <td>400</td>\n      <td>12886</td>\n      <td>5589</td>\n      <td>7297</td>\n      <td>56.627348</td>\n      <td>12286</td>\n      <td>5005</td>\n      <td>7281</td>\n      <td>13.982500</td>\n      <td>0.008548</td>\n      <td>0.002702</td>\n      <td>0.043359</td>\n      <td>0.109118</td>\n      <td>0.003991</td>\n      <td>0.054559</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>Offline simulation; all_400 or labeled_392 scope</td>\n      <td>all_400</td>\n      <td>mass_0900</td>\n      <td>0.900000</td>\n      <td>unavailable / undefined</td>\n      <td>400</td>\n      <td>12886</td>\n      <td>1712</td>\n      <td>11174</td>\n      <td>86.714264</td>\n      <td>12286</td>\n      <td>1203</td>\n      <td>11083</td>\n      <td>4.290000</td>\n      <td>0.068704</td>\n      <td>0.077464</td>\n      <td>0.097836</td>\n      <td>0.099883</td>\n      <td>0.029272</td>\n      <td>0.049919</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>Offline simulation; all_400 or labeled_392 scope</td>\n      <td>all_400</td>\n      <td>mass_0950</td>\n      <td>0.950000</td>\n      <td>unavailable / undefined</td>\n      <td>400</td>\n      <td>12886</td>\n      <td>2391</td>\n      <td>10495</td>\n      <td>81.444979</td>\n      <td>12286</td>\n      <td>1842</td>\n      <td>10444</td>\n      <td>5.987500</td>\n      <td>0.035860</td>\n      <td>0.040774</td>\n      <td>0.049056</td>\n      <td>0.049977</td>\n      <td>0.016135</td>\n      <td>0.024988</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>Offline simulation; all_400 or labeled_392 scope</td>\n      <td>all_400</td>\n      <td>mass_0975</td>\n      <td>0.975000</td>\n      <td>unavailable / undefined</td>\n      <td>400</td>\n      <td>12886</td>\n      <td>3203</td>\n      <td>9683</td>\n      <td>75.143567</td>\n      <td>12286</td>\n      <td>2635</td>\n      <td>9651</td>\n      <td>8.017500</td>\n      <td>0.018268</td>\n      <td>0.021160</td>\n      <td>0.024637</td>\n      <td>0.024991</td>\n      <td>0.008539</td>\n      <td>0.012496</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>Offline simulation; all_400 or labeled_392 scope</td>\n      <td>all_400</td>\n      <td>mass_0990</td>\n      <td>0.990000</td>\n      <td>unavailable / undefined</td>\n      <td>400</td>\n      <td>12886</td>\n      <td>4406</td>\n      <td>8480</td>\n      <td>65.807853</td>\n      <td>12286</td>\n      <td>3824</td>\n      <td>8462</td>\n      <td>11.025000</td>\n      <td>0.007555</td>\n      <td>0.008542</td>\n      <td>0.009850</td>\n      <td>0.009993</td>\n      <td>0.003599</td>\n      <td>0.004997</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>Offline simulation; all_400 or labeled_392 scope</td>\n      <td>labeled_392</td>\n      <td>top_1</td>\n      <td>unavailable / undefined</td>\n      <td>1.000000</td>\n      <td>392</td>\n      <td>12627</td>\n      <td>388</td>\n      <td>12239</td>\n      <td>96.927219</td>\n      <td>12027</td>\n      <td>103</td>\n      <td>11924</td>\n      <td>1.000000</td>\n      <td>0.382642</td>\n      <td>0.390407</td>\n      <td>0.746073</td>\n      <td>0.835280</td>\n      <td>0.127357</td>\n      <td>0.401755</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>Offline simulation; all_400 or labeled_392 scope</td>\n      <td>labeled_392</td>\n      <td>top_3</td>\n      <td>unavailable / undefined</td>\n      <td>3.000000</td>\n      <td>392</td>\n      <td>12627</td>\n      <td>1163</td>\n      <td>11464</td>\n      <td>90.789578</td>\n      <td>12027</td>\n      <td>706</td>\n      <td>11321</td>\n      <td>2.977041</td>\n      <td>0.146563</td>\n      <td>0.105359</td>\n      <td>0.420053</td>\n      <td>0.580629</td>\n      <td>0.056930</td>\n      <td>0.250220</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>Offline simulation; all_400 or labeled_392 scope</td>\n      <td>labeled_392</td>\n      <td>top_5</td>\n      <td>unavailable / undefined</td>\n      <td>5.000000</td>\n      <td>392</td>\n      <td>12627</td>\n      <td>1914</td>\n      <td>10713</td>\n      <td>84.842005</td>\n      <td>12027</td>\n      <td>1399</td>\n      <td>10628</td>\n      <td>4.892857</td>\n      <td>0.073652</td>\n      <td>0.043472</td>\n      <td>0.258945</td>\n      <td>0.433187</td>\n      <td>0.031031</td>\n      <td>0.190020</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>Offline simulation; all_400 or labeled_392 scope</td>\n      <td>labeled_392</td>\n      <td>top_8</td>\n      <td>unavailable / undefined</td>\n      <td>8.000000</td>\n      <td>392</td>\n      <td>12627</td>\n      <td>3008</td>\n      <td>9619</td>\n      <td>76.178031</td>\n      <td>12027</td>\n      <td>2454</td>\n      <td>9573</td>\n      <td>7.683673</td>\n      <td>0.034111</td>\n      <td>0.015439</td>\n      <td>0.140213</td>\n      <td>0.275302</td>\n      <td>0.014704</td>\n      <td>0.104172</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>Offline simulation; all_400 or labeled_392 scope</td>\n      <td>labeled_392</td>\n      <td>top_10</td>\n      <td>unavailable / undefined</td>\n      <td>10.000000</td>\n      <td>392</td>\n      <td>12627</td>\n      <td>3728</td>\n      <td>8899</td>\n      <td>70.475964</td>\n      <td>12027</td>\n      <td>3162</td>\n      <td>8865</td>\n      <td>9.520408</td>\n      <td>0.021975</td>\n      <td>0.008737</td>\n      <td>0.098032</td>\n      <td>0.202093</td>\n      <td>0.009631</td>\n      <td>0.084328</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>Offline simulation; all_400 or labeled_392 scope</td>\n      <td>labeled_392</td>\n      <td>top_15</td>\n      <td>unavailable / undefined</td>\n      <td>15.000000</td>\n      <td>392</td>\n      <td>12627</td>\n      <td>5483</td>\n      <td>7144</td>\n      <td>56.577176</td>\n      <td>12027</td>\n      <td>4899</td>\n      <td>7128</td>\n      <td>13.997449</td>\n      <td>0.008541</td>\n      <td>0.002658</td>\n      <td>0.043669</td>\n      <td>0.109118</td>\n      <td>0.003983</td>\n      <td>0.054559</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>Offline simulation; all_400 or labeled_392 scope</td>\n      <td>labeled_392</td>\n      <td>mass_0900</td>\n      <td>0.900000</td>\n      <td>unavailable / undefined</td>\n      <td>392</td>\n      <td>12627</td>\n      <td>1677</td>\n      <td>10950</td>\n      <td>86.718936</td>\n      <td>12027</td>\n      <td>1168</td>\n      <td>10859</td>\n      <td>4.288265</td>\n      <td>0.068833</td>\n      <td>0.077697</td>\n      <td>0.097889</td>\n      <td>0.099883</td>\n      <td>0.029384</td>\n      <td>0.049919</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>Offline simulation; all_400 or labeled_392 scope</td>\n      <td>labeled_392</td>\n      <td>mass_0950</td>\n      <td>0.950000</td>\n      <td>unavailable / undefined</td>\n      <td>392</td>\n      <td>12627</td>\n      <td>2341</td>\n      <td>10286</td>\n      <td>81.460363</td>\n      <td>12027</td>\n      <td>1792</td>\n      <td>10235</td>\n      <td>5.982143</td>\n      <td>0.035896</td>\n      <td>0.040774</td>\n      <td>0.049121</td>\n      <td>0.049977</td>\n      <td>0.016203</td>\n      <td>0.024988</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>Offline simulation; all_400 or labeled_392 scope</td>\n      <td>labeled_392</td>\n      <td>mass_0975</td>\n      <td>0.975000</td>\n      <td>unavailable / undefined</td>\n      <td>392</td>\n      <td>12627</td>\n      <td>3134</td>\n      <td>9493</td>\n      <td>75.180169</td>\n      <td>12027</td>\n      <td>2566</td>\n      <td>9461</td>\n      <td>8.005102</td>\n      <td>0.018249</td>\n      <td>0.021160</td>\n      <td>0.024646</td>\n      <td>0.024991</td>\n      <td>0.008564</td>\n      <td>0.012496</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>Offline simulation; all_400 or labeled_392 scope</td>\n      <td>labeled_392</td>\n      <td>mass_0990</td>\n      <td>0.990000</td>\n      <td>unavailable / undefined</td>\n      <td>392</td>\n      <td>12627</td>\n      <td>4307</td>\n      <td>8320</td>\n      <td>65.890552</td>\n      <td>12027</td>\n      <td>3725</td>\n      <td>8302</td>\n      <td>10.997449</td>\n      <td>0.007545</td>\n      <td>0.008522</td>\n      <td>0.009853</td>\n      <td>0.009993</td>\n      <td>0.003607</td>\n      <td>0.004997</td>\n    </tr>\n  </tbody>\n</table>',
'')

show('five_archive_continuous_reply_regression',
'<table border="1" class="dataframe">\n  <thead>\n    <tr style="text-align: right;">\n      <th>archive_set</th>\n      <th>protocol</th>\n      <th>target</th>\n      <th>model</th>\n      <th>n</th>\n      <th>games</th>\n      <th>mae</th>\n      <th>spearman</th>\n      <th>estimate_mae_improvement_pp</th>\n      <th>lower95_mae_improvement_pp</th>\n      <th>upper95_mae_improvement_pp</th>\n    </tr>\n  </thead>\n  <tbody>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>missed_opportunity_damage_pp</td>\n      <td>decision_tree</td>\n      <td>23023</td>\n      <td>6495</td>\n      <td>15.751249</td>\n      <td>0.098930</td>\n      <td>-0.822007</td>\n      <td>-1.019293</td>\n      <td>-0.610811</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>missed_opportunity_damage_pp</td>\n      <td>engine_only</td>\n      <td>23023</td>\n      <td>6495</td>\n      <td>14.375102</td>\n      <td>0.055819</td>\n      <td>0.554140</td>\n      <td>0.495344</td>\n      <td>0.615334</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>missed_opportunity_damage_pp</td>\n      <td>engine_ratings</td>\n      <td>23023</td>\n      <td>6495</td>\n      <td>14.362634</td>\n      <td>0.060911</td>\n      <td>0.566607</td>\n      <td>0.505894</td>\n      <td>0.627638</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>missed_opportunity_damage_pp</td>\n      <td>fixed_lightgbm</td>\n      <td>23023</td>\n      <td>6495</td>\n      <td>13.926238</td>\n      <td>0.154206</td>\n      <td>1.003004</td>\n      <td>0.892120</td>\n      <td>1.114976</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>missed_opportunity_damage_pp</td>\n      <td>no_change</td>\n      <td>23023</td>\n      <td>6495</td>\n      <td>14.929242</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>missed_opportunity_damage_pp</td>\n      <td>optuna_lightgbm</td>\n      <td>23023</td>\n      <td>6495</td>\n      <td>13.899741</td>\n      <td>0.160415</td>\n      <td>1.029501</td>\n      <td>0.919279</td>\n      <td>1.138703</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>missed_opportunity_damage_pp</td>\n      <td>optuna_xgboost</td>\n      <td>23023</td>\n      <td>6495</td>\n      <td>13.964605</td>\n      <td>0.138325</td>\n      <td>0.964637</td>\n      <td>0.862671</td>\n      <td>1.068161</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>missed_opportunity_damage_pp</td>\n      <td>random_forest</td>\n      <td>23023</td>\n      <td>6495</td>\n      <td>15.576427</td>\n      <td>0.149411</td>\n      <td>-0.647185</td>\n      <td>-0.852752</td>\n      <td>-0.438432</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>missed_opportunity_damage_pp</td>\n      <td>training_mean</td>\n      <td>23023</td>\n      <td>6495</td>\n      <td>16.364569</td>\n      <td>unavailable / undefined</td>\n      <td>-1.435327</td>\n      <td>-1.622282</td>\n      <td>-1.241909</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>missed_opportunity_damage_pp</td>\n      <td>training_median</td>\n      <td>23023</td>\n      <td>6495</td>\n      <td>14.536219</td>\n      <td>unavailable / undefined</td>\n      <td>0.393023</td>\n      <td>0.360637</td>\n      <td>0.424883</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>opponent_reply_regret_pp</td>\n      <td>decision_tree</td>\n      <td>23023</td>\n      <td>6495</td>\n      <td>15.769205</td>\n      <td>0.097225</td>\n      <td>-1.238576</td>\n      <td>-1.439972</td>\n      <td>-1.023101</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>opponent_reply_regret_pp</td>\n      <td>engine_only</td>\n      <td>23023</td>\n      <td>6495</td>\n      <td>14.323229</td>\n      <td>0.051554</td>\n      <td>0.207400</td>\n      <td>0.145029</td>\n      <td>0.273917</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>opponent_reply_regret_pp</td>\n      <td>engine_ratings</td>\n      <td>23023</td>\n      <td>6495</td>\n      <td>14.317332</td>\n      <td>0.061708</td>\n      <td>0.213297</td>\n      <td>0.148072</td>\n      <td>0.274689</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>opponent_reply_regret_pp</td>\n      <td>fixed_lightgbm</td>\n      <td>23023</td>\n      <td>6495</td>\n      <td>13.888746</td>\n      <td>0.141282</td>\n      <td>0.641883</td>\n      <td>0.507374</td>\n      <td>0.775906</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>opponent_reply_regret_pp</td>\n      <td>no_change</td>\n      <td>23023</td>\n      <td>6495</td>\n      <td>14.530629</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>opponent_reply_regret_pp</td>\n      <td>optuna_lightgbm</td>\n      <td>23023</td>\n      <td>6495</td>\n      <td>13.900121</td>\n      <td>0.142671</td>\n      <td>0.630508</td>\n      <td>0.489286</td>\n      <td>0.766472</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>opponent_reply_regret_pp</td>\n      <td>optuna_xgboost</td>\n      <td>23023</td>\n      <td>6495</td>\n      <td>13.945168</td>\n      <td>0.122166</td>\n      <td>0.585461</td>\n      <td>0.485109</td>\n      <td>0.688045</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>opponent_reply_regret_pp</td>\n      <td>random_forest</td>\n      <td>23023</td>\n      <td>6495</td>\n      <td>15.592782</td>\n      <td>0.149468</td>\n      <td>-1.062153</td>\n      <td>-1.274891</td>\n      <td>-0.847840</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>opponent_reply_regret_pp</td>\n      <td>training_mean</td>\n      <td>23023</td>\n      <td>6495</td>\n      <td>16.384171</td>\n      <td>unavailable / undefined</td>\n      <td>-1.853542</td>\n      <td>-2.046456</td>\n      <td>-1.654710</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game</td>\n      <td>opponent_reply_regret_pp</td>\n      <td>training_median</td>\n      <td>23023</td>\n      <td>6495</td>\n      <td>14.488289</td>\n      <td>unavailable / undefined</td>\n      <td>0.042340</td>\n      <td>0.018988</td>\n      <td>0.065798</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>missed_opportunity_damage_pp</td>\n      <td>decision_tree</td>\n      <td>23023</td>\n      <td>6495</td>\n      <td>15.713979</td>\n      <td>0.106828</td>\n      <td>-0.784737</td>\n      <td>-0.990254</td>\n      <td>-0.578920</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>missed_opportunity_damage_pp</td>\n      <td>engine_only</td>\n      <td>23023</td>\n      <td>6495</td>\n      <td>14.384353</td>\n      <td>0.051209</td>\n      <td>0.544888</td>\n      <td>0.487787</td>\n      <td>0.607508</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>missed_opportunity_damage_pp</td>\n      <td>engine_ratings</td>\n      <td>23023</td>\n      <td>6495</td>\n      <td>14.371079</td>\n      <td>0.059956</td>\n      <td>0.558163</td>\n      <td>0.497293</td>\n      <td>0.620941</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>missed_opportunity_damage_pp</td>\n      <td>fixed_lightgbm</td>\n      <td>23023</td>\n      <td>6495</td>\n      <td>13.939959</td>\n      <td>0.149105</td>\n      <td>0.989283</td>\n      <td>0.885048</td>\n      <td>1.098148</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>missed_opportunity_damage_pp</td>\n      <td>no_change</td>\n      <td>23023</td>\n      <td>6495</td>\n      <td>14.929242</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>missed_opportunity_damage_pp</td>\n      <td>optuna_lightgbm</td>\n      <td>23023</td>\n      <td>6495</td>\n      <td>13.962432</td>\n      <td>0.127264</td>\n      <td>0.966810</td>\n      <td>0.873179</td>\n      <td>1.058093</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>missed_opportunity_damage_pp</td>\n      <td>optuna_xgboost</td>\n      <td>23023</td>\n      <td>6495</td>\n      <td>14.000420</td>\n      <td>0.132540</td>\n      <td>0.928822</td>\n      <td>0.835023</td>\n      <td>1.025893</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>missed_opportunity_damage_pp</td>\n      <td>random_forest</td>\n      <td>23023</td>\n      <td>6495</td>\n      <td>15.558056</td>\n      <td>0.149648</td>\n      <td>-0.628814</td>\n      <td>-0.831274</td>\n      <td>-0.421778</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>missed_opportunity_damage_pp</td>\n      <td>training_mean</td>\n      <td>23023</td>\n      <td>6495</td>\n      <td>16.294691</td>\n      <td>unavailable / undefined</td>\n      <td>-1.365450</td>\n      <td>-1.550268</td>\n      <td>-1.175873</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>missed_opportunity_damage_pp</td>\n      <td>training_median</td>\n      <td>23023</td>\n      <td>6495</td>\n      <td>14.536105</td>\n      <td>unavailable / undefined</td>\n      <td>0.393137</td>\n      <td>0.361939</td>\n      <td>0.424362</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>opponent_reply_regret_pp</td>\n      <td>decision_tree</td>\n      <td>23023</td>\n      <td>6495</td>\n      <td>15.732734</td>\n      <td>0.105545</td>\n      <td>-1.202105</td>\n      <td>-1.407953</td>\n      <td>-0.987904</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>opponent_reply_regret_pp</td>\n      <td>engine_only</td>\n      <td>23023</td>\n      <td>6495</td>\n      <td>14.327158</td>\n      <td>0.051445</td>\n      <td>0.203471</td>\n      <td>0.144657</td>\n      <td>0.264171</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>opponent_reply_regret_pp</td>\n      <td>engine_ratings</td>\n      <td>23023</td>\n      <td>6495</td>\n      <td>14.307400</td>\n      <td>0.061412</td>\n      <td>0.223229</td>\n      <td>0.162706</td>\n      <td>0.284133</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>opponent_reply_regret_pp</td>\n      <td>fixed_lightgbm</td>\n      <td>23023</td>\n      <td>6495</td>\n      <td>13.926060</td>\n      <td>0.131429</td>\n      <td>0.604569</td>\n      <td>0.479359</td>\n      <td>0.731831</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>opponent_reply_regret_pp</td>\n      <td>no_change</td>\n      <td>23023</td>\n      <td>6495</td>\n      <td>14.530629</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>opponent_reply_regret_pp</td>\n      <td>optuna_lightgbm</td>\n      <td>23023</td>\n      <td>6495</td>\n      <td>13.865441</td>\n      <td>0.142658</td>\n      <td>0.665188</td>\n      <td>0.541885</td>\n      <td>0.783747</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>opponent_reply_regret_pp</td>\n      <td>optuna_xgboost</td>\n      <td>23023</td>\n      <td>6495</td>\n      <td>13.934275</td>\n      <td>0.118297</td>\n      <td>0.596354</td>\n      <td>0.503814</td>\n      <td>0.695083</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>opponent_reply_regret_pp</td>\n      <td>random_forest</td>\n      <td>23023</td>\n      <td>6495</td>\n      <td>15.585350</td>\n      <td>0.148552</td>\n      <td>-1.054721</td>\n      <td>-1.266985</td>\n      <td>-0.843531</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>opponent_reply_regret_pp</td>\n      <td>training_mean</td>\n      <td>23023</td>\n      <td>6495</td>\n      <td>16.316185</td>\n      <td>unavailable / undefined</td>\n      <td>-1.785556</td>\n      <td>-1.974814</td>\n      <td>-1.589981</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player</td>\n      <td>opponent_reply_regret_pp</td>\n      <td>training_median</td>\n      <td>23023</td>\n      <td>6495</td>\n      <td>14.488042</td>\n      <td>unavailable / undefined</td>\n      <td>0.042587</td>\n      <td>0.021318</td>\n      <td>0.063929</td>\n    </tr>\n  </tbody>\n</table>',
'23,023 valid replies / 6,495 games; shared main-study outer game map.')

show('earlier_reply_models_game',
'<table border="1" class="dataframe">\n  <thead>\n    <tr style="text-align: right;">\n      <th>archive_set</th>\n      <th>protocol</th>\n      <th>target</th>\n      <th>algorithm</th>\n      <th>games</th>\n      <th>n</th>\n      <th>mae</th>\n      <th>spearman</th>\n      <th>mae_gain_vs_no_change</th>\n      <th>mae_gain_vs_no_change_percent</th>\n    </tr>\n  </thead>\n  <tbody>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game; earlier reply-only eligible game universe</td>\n      <td>missed_opportunity_damage_pp</td>\n      <td>LightGBM</td>\n      <td>6495</td>\n      <td>23023</td>\n      <td>13.858693</td>\n      <td>0.162781</td>\n      <td>1.070549</td>\n      <td>7.170820</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game; earlier reply-only eligible game universe</td>\n      <td>missed_opportunity_damage_pp</td>\n      <td>XGBoost</td>\n      <td>6495</td>\n      <td>23023</td>\n      <td>13.936418</td>\n      <td>0.142247</td>\n      <td>0.992824</td>\n      <td>6.650199</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game; earlier reply-only eligible game universe</td>\n      <td>missed_opportunity_damage_pp</td>\n      <td>no_change</td>\n      <td>6495</td>\n      <td>23023</td>\n      <td>14.929242</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game; earlier reply-only eligible game universe</td>\n      <td>missed_opportunity_damage_pp</td>\n      <td>training_median</td>\n      <td>6495</td>\n      <td>23023</td>\n      <td>14.535339</td>\n      <td>-0.012388</td>\n      <td>0.393903</td>\n      <td>2.638468</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game; earlier reply-only eligible game universe</td>\n      <td>opponent_reply_regret_pp</td>\n      <td>LightGBM</td>\n      <td>6495</td>\n      <td>23023</td>\n      <td>13.766936</td>\n      <td>0.151394</td>\n      <td>0.763693</td>\n      <td>5.255747</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game; earlier reply-only eligible game universe</td>\n      <td>opponent_reply_regret_pp</td>\n      <td>XGBoost</td>\n      <td>6495</td>\n      <td>23023</td>\n      <td>13.878897</td>\n      <td>0.131015</td>\n      <td>0.651732</td>\n      <td>4.485226</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game; earlier reply-only eligible game universe</td>\n      <td>opponent_reply_regret_pp</td>\n      <td>no_change</td>\n      <td>6495</td>\n      <td>23023</td>\n      <td>14.530629</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>game; earlier reply-only eligible game universe</td>\n      <td>opponent_reply_regret_pp</td>\n      <td>training_median</td>\n      <td>6495</td>\n      <td>23023</td>\n      <td>14.486185</td>\n      <td>-0.012439</td>\n      <td>0.044444</td>\n      <td>0.305863</td>\n    </tr>\n  </tbody>\n</table>',
'Uses an earlier eligible-game universe.')

show('earlier_reply_models_player',
'<table border="1" class="dataframe">\n  <thead>\n    <tr style="text-align: right;">\n      <th>archive_set</th>\n      <th>protocol</th>\n      <th>target</th>\n      <th>algorithm</th>\n      <th>games</th>\n      <th>n</th>\n      <th>mae</th>\n      <th>spearman</th>\n      <th>mae_gain_vs_no_change</th>\n      <th>mae_gain_vs_no_change_percent</th>\n    </tr>\n  </thead>\n  <tbody>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player; earlier reply-only eligible game universe</td>\n      <td>missed_opportunity_damage_pp</td>\n      <td>LightGBM</td>\n      <td>6495</td>\n      <td>23023</td>\n      <td>13.901618</td>\n      <td>0.154255</td>\n      <td>1.027624</td>\n      <td>6.883299</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player; earlier reply-only eligible game universe</td>\n      <td>missed_opportunity_damage_pp</td>\n      <td>XGBoost</td>\n      <td>6495</td>\n      <td>23023</td>\n      <td>13.965869</td>\n      <td>0.138908</td>\n      <td>0.963373</td>\n      <td>6.452928</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player; earlier reply-only eligible game universe</td>\n      <td>missed_opportunity_damage_pp</td>\n      <td>no_change</td>\n      <td>6495</td>\n      <td>23023</td>\n      <td>14.929242</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player; earlier reply-only eligible game universe</td>\n      <td>missed_opportunity_damage_pp</td>\n      <td>training_median</td>\n      <td>6495</td>\n      <td>23023</td>\n      <td>14.536317</td>\n      <td>-0.012388</td>\n      <td>0.392925</td>\n      <td>2.631916</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player; earlier reply-only eligible game universe</td>\n      <td>opponent_reply_regret_pp</td>\n      <td>LightGBM</td>\n      <td>6495</td>\n      <td>23023</td>\n      <td>13.851476</td>\n      <td>0.120667</td>\n      <td>0.679153</td>\n      <td>4.673941</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player; earlier reply-only eligible game universe</td>\n      <td>opponent_reply_regret_pp</td>\n      <td>XGBoost</td>\n      <td>6495</td>\n      <td>23023</td>\n      <td>13.926193</td>\n      <td>0.132334</td>\n      <td>0.604436</td>\n      <td>4.159739</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player; earlier reply-only eligible game universe</td>\n      <td>opponent_reply_regret_pp</td>\n      <td>no_change</td>\n      <td>6495</td>\n      <td>23023</td>\n      <td>14.530629</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>player; earlier reply-only eligible game universe</td>\n      <td>opponent_reply_regret_pp</td>\n      <td>training_median</td>\n      <td>6495</td>\n      <td>23023</td>\n      <td>14.488763</td>\n      <td>-0.012439</td>\n      <td>0.041866</td>\n      <td>0.288120</td>\n    </tr>\n  </tbody>\n</table>',
'Uses an earlier eligible-game universe.')

show('earlier_xgboost_lightgbm_v1_regression',
'<table border="1" class="dataframe">\n  <thead>\n    <tr style="text-align: right;">\n      <th>archive_set</th>\n      <th>protocol</th>\n      <th>horizon</th>\n      <th>algorithm</th>\n      <th>n</th>\n      <th>games</th>\n      <th>mae</th>\n      <th>spearman</th>\n      <th>mae_gain_vs_no_change</th>\n      <th>mae_gain_vs_no_change_percent</th>\n    </tr>\n  </thead>\n  <tbody>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Game-grouped outer CV; nested inner selection</td>\n      <td>2</td>\n      <td>LightGBM</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>12.556772</td>\n      <td>0.119690</td>\n      <td>0.019786</td>\n      <td>0.157328</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Game-grouped outer CV; nested inner selection</td>\n      <td>2</td>\n      <td>XGBoost</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>12.544548</td>\n      <td>0.119889</td>\n      <td>0.032010</td>\n      <td>0.254524</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Game-grouped outer CV; nested inner selection</td>\n      <td>2</td>\n      <td>no_change</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>12.576558</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Game-grouped outer CV; nested inner selection</td>\n      <td>2</td>\n      <td>training_mean</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>14.650068</td>\n      <td>-0.002965</td>\n      <td>-2.073509</td>\n      <td>-16.487095</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Game-grouped outer CV; nested inner selection</td>\n      <td>4</td>\n      <td>LightGBM</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>16.847292</td>\n      <td>0.094286</td>\n      <td>0.088572</td>\n      <td>0.522986</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Game-grouped outer CV; nested inner selection</td>\n      <td>4</td>\n      <td>XGBoost</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>16.807706</td>\n      <td>0.106290</td>\n      <td>0.128159</td>\n      <td>0.756730</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Game-grouped outer CV; nested inner selection</td>\n      <td>4</td>\n      <td>no_change</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>16.935864</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Game-grouped outer CV; nested inner selection</td>\n      <td>4</td>\n      <td>training_mean</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>18.868808</td>\n      <td>-0.008566</td>\n      <td>-1.932944</td>\n      <td>-11.413316</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Game-grouped outer CV; nested inner selection</td>\n      <td>6</td>\n      <td>LightGBM</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>19.596744</td>\n      <td>0.098165</td>\n      <td>0.237748</td>\n      <td>1.198659</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Game-grouped outer CV; nested inner selection</td>\n      <td>6</td>\n      <td>XGBoost</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>19.559867</td>\n      <td>0.100449</td>\n      <td>0.274625</td>\n      <td>1.384584</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Game-grouped outer CV; nested inner selection</td>\n      <td>6</td>\n      <td>no_change</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>19.834492</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Game-grouped outer CV; nested inner selection</td>\n      <td>6</td>\n      <td>training_mean</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>21.768507</td>\n      <td>-0.009466</td>\n      <td>-1.934015</td>\n      <td>-9.750766</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Game-grouped outer CV; nested inner selection</td>\n      <td>10</td>\n      <td>LightGBM</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>23.054373</td>\n      <td>0.116905</td>\n      <td>0.581039</td>\n      <td>2.458339</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Game-grouped outer CV; nested inner selection</td>\n      <td>10</td>\n      <td>XGBoost</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>23.059055</td>\n      <td>0.120976</td>\n      <td>0.576356</td>\n      <td>2.438528</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Game-grouped outer CV; nested inner selection</td>\n      <td>10</td>\n      <td>no_change</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>23.635411</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Game-grouped outer CV; nested inner selection</td>\n      <td>10</td>\n      <td>training_mean</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>25.600246</td>\n      <td>-0.005981</td>\n      <td>-1.964835</td>\n      <td>-8.313097</td>\n    </tr>\n  </tbody>\n</table>',
'Earlier development study; overlapping data do not provide independent replication.')

show('earlier_xgboost_lightgbm_v1_classification',
'<table border="1" class="dataframe">\n  <thead>\n    <tr style="text-align: right;">\n      <th>archive_set</th>\n      <th>protocol</th>\n      <th>algorithm</th>\n      <th>strategy</th>\n      <th>games</th>\n      <th>n</th>\n      <th>tn</th>\n      <th>fp</th>\n      <th>fn</th>\n      <th>tp</th>\n      <th>accuracy</th>\n      <th>balanced_accuracy</th>\n      <th>recovered_precision</th>\n      <th>recovered_recall</th>\n      <th>recovered_f1</th>\n      <th>specificity</th>\n      <th>false_positive_rate</th>\n      <th>actual_recovered</th>\n      <th>predicted_recovered</th>\n      <th>roc_auc</th>\n      <th>average_precision</th>\n    </tr>\n  </thead>\n  <tbody>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Game-grouped outer CV; nested inner selection</td>\n      <td>LightGBM</td>\n      <td>fixed_0.5</td>\n      <td>6512</td>\n      <td>23167</td>\n      <td>14389</td>\n      <td>477</td>\n      <td>7721</td>\n      <td>580</td>\n      <td>0.646135</td>\n      <td>0.518892</td>\n      <td>0.548723</td>\n      <td>0.069871</td>\n      <td>0.123958</td>\n      <td>0.967913</td>\n      <td>0.032087</td>\n      <td>8301</td>\n      <td>1057</td>\n      <td>0.615719</td>\n      <td>0.459812</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Game-grouped outer CV; nested inner selection</td>\n      <td>LightGBM</td>\n      <td>tuned_balanced_accuracy</td>\n      <td>6512</td>\n      <td>23167</td>\n      <td>7748</td>\n      <td>7118</td>\n      <td>2969</td>\n      <td>5332</td>\n      <td>0.564596</td>\n      <td>0.581761</td>\n      <td>0.428273</td>\n      <td>0.642332</td>\n      <td>0.513903</td>\n      <td>0.521189</td>\n      <td>0.478811</td>\n      <td>8301</td>\n      <td>12450</td>\n      <td>0.615719</td>\n      <td>0.459812</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Game-grouped outer CV; nested inner selection</td>\n      <td>XGBoost</td>\n      <td>fixed_0.5</td>\n      <td>6512</td>\n      <td>23167</td>\n      <td>14580</td>\n      <td>286</td>\n      <td>7889</td>\n      <td>412</td>\n      <td>0.647127</td>\n      <td>0.515197</td>\n      <td>0.590258</td>\n      <td>0.049633</td>\n      <td>0.091566</td>\n      <td>0.980761</td>\n      <td>0.019239</td>\n      <td>8301</td>\n      <td>698</td>\n      <td>0.615610</td>\n      <td>0.462219</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Game-grouped outer CV; nested inner selection</td>\n      <td>XGBoost</td>\n      <td>tuned_balanced_accuracy</td>\n      <td>6512</td>\n      <td>23167</td>\n      <td>7525</td>\n      <td>7341</td>\n      <td>2847</td>\n      <td>5454</td>\n      <td>0.560237</td>\n      <td>0.581609</td>\n      <td>0.426260</td>\n      <td>0.657029</td>\n      <td>0.517065</td>\n      <td>0.506189</td>\n      <td>0.493811</td>\n      <td>8301</td>\n      <td>12795</td>\n      <td>0.615610</td>\n      <td>0.462219</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Game-grouped outer CV; nested inner selection</td>\n      <td>training_majority</td>\n      <td>training_majority</td>\n      <td>6512</td>\n      <td>23167</td>\n      <td>14866</td>\n      <td>0</td>\n      <td>8301</td>\n      <td>0</td>\n      <td>0.641689</td>\n      <td>0.500000</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n      <td>1.000000</td>\n      <td>0.000000</td>\n      <td>8301</td>\n      <td>0</td>\n      <td>unavailable / undefined</td>\n      <td>unavailable / undefined</td>\n    </tr>\n  </tbody>\n</table>',
'Earlier development study; overlapping data do not provide independent replication.')

show('earlier_xgboost_lightgbm_v2_regression',
'<table border="1" class="dataframe">\n  <thead>\n    <tr style="text-align: right;">\n      <th>archive_set</th>\n      <th>protocol</th>\n      <th>horizon</th>\n      <th>algorithm</th>\n      <th>n</th>\n      <th>games</th>\n      <th>mae</th>\n      <th>spearman</th>\n      <th>mae_gain_vs_no_change</th>\n      <th>mae_gain_vs_no_change_percent</th>\n    </tr>\n  </thead>\n  <tbody>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Game-grouped outer CV; nested inner selection</td>\n      <td>2</td>\n      <td>LightGBM</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>12.556772</td>\n      <td>0.119690</td>\n      <td>0.019786</td>\n      <td>0.157328</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Game-grouped outer CV; nested inner selection</td>\n      <td>2</td>\n      <td>XGBoost</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>12.544548</td>\n      <td>0.119889</td>\n      <td>0.032010</td>\n      <td>0.254524</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Game-grouped outer CV; nested inner selection</td>\n      <td>2</td>\n      <td>no_change</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>12.576558</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Game-grouped outer CV; nested inner selection</td>\n      <td>2</td>\n      <td>training_mean</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>14.650068</td>\n      <td>-0.002965</td>\n      <td>-2.073509</td>\n      <td>-16.487095</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Game-grouped outer CV; nested inner selection</td>\n      <td>4</td>\n      <td>LightGBM</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>16.847292</td>\n      <td>0.094286</td>\n      <td>0.088572</td>\n      <td>0.522986</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Game-grouped outer CV; nested inner selection</td>\n      <td>4</td>\n      <td>XGBoost</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>16.807706</td>\n      <td>0.106290</td>\n      <td>0.128159</td>\n      <td>0.756730</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Game-grouped outer CV; nested inner selection</td>\n      <td>4</td>\n      <td>no_change</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>16.935864</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Game-grouped outer CV; nested inner selection</td>\n      <td>4</td>\n      <td>training_mean</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>18.868808</td>\n      <td>-0.008566</td>\n      <td>-1.932944</td>\n      <td>-11.413316</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Game-grouped outer CV; nested inner selection</td>\n      <td>6</td>\n      <td>LightGBM</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>19.596744</td>\n      <td>0.098165</td>\n      <td>0.237748</td>\n      <td>1.198659</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Game-grouped outer CV; nested inner selection</td>\n      <td>6</td>\n      <td>XGBoost</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>19.559867</td>\n      <td>0.100449</td>\n      <td>0.274625</td>\n      <td>1.384584</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Game-grouped outer CV; nested inner selection</td>\n      <td>6</td>\n      <td>no_change</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>19.834492</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Game-grouped outer CV; nested inner selection</td>\n      <td>6</td>\n      <td>training_mean</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>21.768507</td>\n      <td>-0.009466</td>\n      <td>-1.934015</td>\n      <td>-9.750766</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Game-grouped outer CV; nested inner selection</td>\n      <td>10</td>\n      <td>LightGBM</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>23.054373</td>\n      <td>0.116905</td>\n      <td>0.581039</td>\n      <td>2.458339</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Game-grouped outer CV; nested inner selection</td>\n      <td>10</td>\n      <td>XGBoost</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>23.059055</td>\n      <td>0.120976</td>\n      <td>0.576356</td>\n      <td>2.438528</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Game-grouped outer CV; nested inner selection</td>\n      <td>10</td>\n      <td>no_change</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>23.635411</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Game-grouped outer CV; nested inner selection</td>\n      <td>10</td>\n      <td>training_mean</td>\n      <td>23167</td>\n      <td>6512</td>\n      <td>25.600246</td>\n      <td>-0.005981</td>\n      <td>-1.964835</td>\n      <td>-8.313097</td>\n    </tr>\n  </tbody>\n</table>',
'Earlier development study; overlapping data do not provide independent replication.')

show('earlier_xgboost_lightgbm_v2_classification',
'<table border="1" class="dataframe">\n  <thead>\n    <tr style="text-align: right;">\n      <th>archive_set</th>\n      <th>protocol</th>\n      <th>algorithm</th>\n      <th>strategy</th>\n      <th>games</th>\n      <th>n</th>\n      <th>tn</th>\n      <th>fp</th>\n      <th>fn</th>\n      <th>tp</th>\n      <th>accuracy</th>\n      <th>balanced_accuracy</th>\n      <th>recovered_precision</th>\n      <th>recovered_recall</th>\n      <th>recovered_f1</th>\n      <th>specificity</th>\n      <th>false_positive_rate</th>\n      <th>actual_recovered</th>\n      <th>predicted_recovered</th>\n      <th>roc_auc</th>\n      <th>average_precision</th>\n    </tr>\n  </thead>\n  <tbody>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Game-grouped outer CV; nested inner selection</td>\n      <td>LightGBM</td>\n      <td>fixed_0.5</td>\n      <td>6512</td>\n      <td>23167</td>\n      <td>14389</td>\n      <td>477</td>\n      <td>7721</td>\n      <td>580</td>\n      <td>0.646135</td>\n      <td>0.518892</td>\n      <td>0.548723</td>\n      <td>0.069871</td>\n      <td>0.123958</td>\n      <td>0.967913</td>\n      <td>0.032087</td>\n      <td>8301</td>\n      <td>1057</td>\n      <td>0.615719</td>\n      <td>0.459812</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Game-grouped outer CV; nested inner selection</td>\n      <td>LightGBM</td>\n      <td>tuned_balanced_accuracy</td>\n      <td>6512</td>\n      <td>23167</td>\n      <td>7748</td>\n      <td>7118</td>\n      <td>2969</td>\n      <td>5332</td>\n      <td>0.564596</td>\n      <td>0.581761</td>\n      <td>0.428273</td>\n      <td>0.642332</td>\n      <td>0.513903</td>\n      <td>0.521189</td>\n      <td>0.478811</td>\n      <td>8301</td>\n      <td>12450</td>\n      <td>0.615719</td>\n      <td>0.459812</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Game-grouped outer CV; nested inner selection</td>\n      <td>XGBoost</td>\n      <td>fixed_0.5</td>\n      <td>6512</td>\n      <td>23167</td>\n      <td>14580</td>\n      <td>286</td>\n      <td>7889</td>\n      <td>412</td>\n      <td>0.647127</td>\n      <td>0.515197</td>\n      <td>0.590258</td>\n      <td>0.049633</td>\n      <td>0.091566</td>\n      <td>0.980761</td>\n      <td>0.019239</td>\n      <td>8301</td>\n      <td>698</td>\n      <td>0.615610</td>\n      <td>0.462219</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Game-grouped outer CV; nested inner selection</td>\n      <td>XGBoost</td>\n      <td>tuned_balanced_accuracy</td>\n      <td>6512</td>\n      <td>23167</td>\n      <td>7525</td>\n      <td>7341</td>\n      <td>2847</td>\n      <td>5454</td>\n      <td>0.560237</td>\n      <td>0.581609</td>\n      <td>0.426260</td>\n      <td>0.657029</td>\n      <td>0.517065</td>\n      <td>0.506189</td>\n      <td>0.493811</td>\n      <td>8301</td>\n      <td>12795</td>\n      <td>0.615610</td>\n      <td>0.462219</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Game-grouped outer CV; nested inner selection</td>\n      <td>training_majority</td>\n      <td>training_majority</td>\n      <td>6512</td>\n      <td>23167</td>\n      <td>14866</td>\n      <td>0</td>\n      <td>8301</td>\n      <td>0</td>\n      <td>0.641689</td>\n      <td>0.500000</td>\n      <td>unavailable / undefined</td>\n      <td>0.000000</td>\n      <td>0.000000</td>\n      <td>1.000000</td>\n      <td>0.000000</td>\n      <td>8301</td>\n      <td>0</td>\n      <td>unavailable / undefined</td>\n      <td>unavailable / undefined</td>\n    </tr>\n  </tbody>\n</table>',
'Earlier development study; overlapping data do not provide independent replication.')

show('learning_curves_20261110T222901Z_d3c7e778_final_iteration',
'<table border="1" class="dataframe">\n  <thead>\n    <tr style="text-align: right;">\n      <th>archive_set</th>\n      <th>protocol</th>\n      <th>horizon</th>\n      <th>feature_set</th>\n      <th>iteration</th>\n      <th>split</th>\n      <th>folds</th>\n      <th>observation_occurrences</th>\n      <th>mae</th>\n      <th>mae_min_fold</th>\n      <th>mae_max_fold</th>\n      <th>accuracy_exploratory</th>\n      <th>accuracy_exploratory_min_fold</th>\n      <th>accuracy_exploratory_max_fold</th>\n      <th>no_change_mae</th>\n      <th>no_change_mae_min_fold</th>\n      <th>no_change_mae_max_fold</th>\n      <th>majority_accuracy</th>\n      <th>majority_accuracy_min_fold</th>\n      <th>majority_accuracy_max_fold</th>\n    </tr>\n  </thead>\n  <tbody>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Saved learning-curve split; see split column</td>\n      <td>6</td>\n      <td>full_multipv</td>\n      <td>240</td>\n      <td>training</td>\n      <td>3</td>\n      <td>46334</td>\n      <td>19.143593</td>\n      <td>19.101213</td>\n      <td>19.205253</td>\n      <td>0.649782</td>\n      <td>0.646979</td>\n      <td>0.651494</td>\n      <td>19.834492</td>\n      <td>19.763157</td>\n      <td>19.916033</td>\n      <td>0.641689</td>\n      <td>0.638588</td>\n      <td>0.643411</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Saved learning-curve split; see split column</td>\n      <td>6</td>\n      <td>full_multipv</td>\n      <td>240</td>\n      <td>validation</td>\n      <td>3</td>\n      <td>23167</td>\n      <td>19.585325</td>\n      <td>19.502981</td>\n      <td>19.704338</td>\n      <td>0.645919</td>\n      <td>0.643435</td>\n      <td>0.650554</td>\n      <td>19.834492</td>\n      <td>19.669900</td>\n      <td>19.975275</td>\n      <td>0.641689</td>\n      <td>0.638229</td>\n      <td>0.647948</td>\n    </tr>\n  </tbody>\n</table>',
'Training errors are in-sample; validation curves are development diagnostics.')

show('learning_curves_20261110T223938Z_6ed4aaf1_final_iteration',
'<table border="1" class="dataframe">\n  <thead>\n    <tr style="text-align: right;">\n      <th>archive_set</th>\n      <th>protocol</th>\n      <th>horizon</th>\n      <th>feature_set</th>\n      <th>iteration</th>\n      <th>split</th>\n      <th>folds</th>\n      <th>observation_occurrences</th>\n      <th>mae</th>\n      <th>mae_min_fold</th>\n      <th>mae_max_fold</th>\n      <th>accuracy_exploratory</th>\n      <th>accuracy_exploratory_min_fold</th>\n      <th>accuracy_exploratory_max_fold</th>\n      <th>no_change_mae</th>\n      <th>no_change_mae_min_fold</th>\n      <th>no_change_mae_max_fold</th>\n      <th>majority_accuracy</th>\n      <th>majority_accuracy_min_fold</th>\n      <th>majority_accuracy_max_fold</th>\n    </tr>\n  </thead>\n  <tbody>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Saved learning-curve split; see split column</td>\n      <td>6</td>\n      <td>full_multipv</td>\n      <td>240</td>\n      <td>training</td>\n      <td>3</td>\n      <td>46334</td>\n      <td>19.143593</td>\n      <td>19.101213</td>\n      <td>19.205253</td>\n      <td>0.649782</td>\n      <td>0.646979</td>\n      <td>0.651494</td>\n      <td>19.834492</td>\n      <td>19.763157</td>\n      <td>19.916033</td>\n      <td>0.641689</td>\n      <td>0.638588</td>\n      <td>0.643411</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Saved learning-curve split; see split column</td>\n      <td>6</td>\n      <td>full_multipv</td>\n      <td>240</td>\n      <td>validation</td>\n      <td>3</td>\n      <td>23167</td>\n      <td>19.585325</td>\n      <td>19.502981</td>\n      <td>19.704338</td>\n      <td>0.645919</td>\n      <td>0.643435</td>\n      <td>0.650554</td>\n      <td>19.834492</td>\n      <td>19.669900</td>\n      <td>19.975275</td>\n      <td>0.641689</td>\n      <td>0.638229</td>\n      <td>0.647948</td>\n    </tr>\n  </tbody>\n</table>',
'Training errors are in-sample; validation curves are development diagnostics.')

show('learning_curves_20261110T224023Z_31ae5f8e_final_iteration',
'<table border="1" class="dataframe">\n  <thead>\n    <tr style="text-align: right;">\n      <th>archive_set</th>\n      <th>protocol</th>\n      <th>horizon</th>\n      <th>feature_set</th>\n      <th>iteration</th>\n      <th>split</th>\n      <th>folds</th>\n      <th>observation_occurrences</th>\n      <th>mae</th>\n      <th>mae_min_fold</th>\n      <th>mae_max_fold</th>\n      <th>accuracy_exploratory</th>\n      <th>accuracy_exploratory_min_fold</th>\n      <th>accuracy_exploratory_max_fold</th>\n      <th>no_change_mae</th>\n      <th>no_change_mae_min_fold</th>\n      <th>no_change_mae_max_fold</th>\n      <th>majority_accuracy</th>\n      <th>majority_accuracy_min_fold</th>\n      <th>majority_accuracy_max_fold</th>\n    </tr>\n  </thead>\n  <tbody>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Saved learning-curve split; see split column</td>\n      <td>6</td>\n      <td>full_multipv</td>\n      <td>240</td>\n      <td>training</td>\n      <td>3</td>\n      <td>46334</td>\n      <td>19.143593</td>\n      <td>19.101213</td>\n      <td>19.205253</td>\n      <td>0.649782</td>\n      <td>0.646979</td>\n      <td>0.651494</td>\n      <td>19.834492</td>\n      <td>19.763157</td>\n      <td>19.916033</td>\n      <td>0.641689</td>\n      <td>0.638588</td>\n      <td>0.643411</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Saved learning-curve split; see split column</td>\n      <td>6</td>\n      <td>full_multipv</td>\n      <td>240</td>\n      <td>validation</td>\n      <td>3</td>\n      <td>23167</td>\n      <td>19.585325</td>\n      <td>19.502981</td>\n      <td>19.704338</td>\n      <td>0.645919</td>\n      <td>0.643435</td>\n      <td>0.650554</td>\n      <td>19.834492</td>\n      <td>19.669900</td>\n      <td>19.975275</td>\n      <td>0.641689</td>\n      <td>0.638229</td>\n      <td>0.647948</td>\n    </tr>\n  </tbody>\n</table>',
'Training errors are in-sample; validation curves are development diagnostics.')


**Pilot policy all five exploratory**

Raw and calibrated probabilities are separate methods. February 2019 may overlap external pretraining.

archive_set,scope,protocol,method,games,n,event_prevalence,mean_probability,brier,log_loss,roc_auc,average_precision
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",all_five_exploratory,game,calibrated_policy_mass,460,491,0.578411,0.580714,0.170037,0.510348,0.817786,0.854959
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",all_five_exploratory,game,favorite_acceptable_only,460,491,0.578411,0.458582,0.235718,0.806235,0.739845,0.809026
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",all_five_exploratory,game,raw_policy_mass,460,491,0.578411,0.577887,0.171374,0.521472,0.822498,0.858022
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",all_five_exploratory,game,training_prevalence,460,491,0.578411,0.557910,0.244475,0.682067,0.454132,0.556955
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",all_five_exploratory,game,uniform_legal,460,491,0.578411,0.087519,0.475940,1.639945,0.583325,0.689296
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",all_five_exploratory,player,calibrated_policy_mass,460,491,0.578411,0.580678,0.169955,0.510588,0.819198,0.856339
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",all_five_exploratory,player,favorite_acceptable_only,460,491,0.578411,0.458582,0.235718,0.806235,0.739845,0.809026
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",all_five_exploratory,player,raw_policy_mass,460,491,0.578411,0.577887,0.171374,0.521472,0.822498,0.858022
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",all_five_exploratory,player,training_prevalence,460,491,0.578411,0.561376,0.244457,0.682029,0.454132,0.556955
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",all_five_exploratory,player,uniform_legal,460,491,0.578411,0.087519,0.475940,1.639945,0.583325,0.689296


**Pilot policy predates documented pretraining**

Four archives predating documented pretraining; raw and calibrated methods are separate.

archive_set,scope,protocol,method,games,n,event_prevalence,mean_probability,brier,log_loss,roc_auc,average_precision
"2017-04, 2017-06, 2017-07, 2018-02",predates_documented_pretraining,game,calibrated_policy_mass,364,392,0.579082,0.578955,0.162380,0.492913,0.834708,0.867408
"2017-04, 2017-06, 2017-07, 2018-02",predates_documented_pretraining,game,favorite_acceptable_only,364,392,0.579082,0.463382,0.226500,0.730574,0.757175,0.823835
"2017-04, 2017-06, 2017-07, 2018-02",predates_documented_pretraining,game,raw_policy_mass,364,392,0.579082,0.581494,0.162214,0.501027,0.838446,0.869184
"2017-04, 2017-06, 2017-07, 2018-02",predates_documented_pretraining,game,training_prevalence,364,392,0.579082,0.558512,0.244315,0.681743,0.489908,0.570123
"2017-04, 2017-06, 2017-07, 2018-02",predates_documented_pretraining,game,uniform_legal,364,392,0.579082,0.091639,0.470740,1.615026,0.606982,0.706064
"2017-04, 2017-06, 2017-07, 2018-02",predates_documented_pretraining,player,calibrated_policy_mass,364,392,0.579082,0.582853,0.162025,0.493124,0.835082,0.867615
"2017-04, 2017-06, 2017-07, 2018-02",predates_documented_pretraining,player,favorite_acceptable_only,364,392,0.579082,0.463382,0.226500,0.730574,0.757175,0.823835
"2017-04, 2017-06, 2017-07, 2018-02",predates_documented_pretraining,player,raw_policy_mass,364,392,0.579082,0.581494,0.162214,0.501027,0.838446,0.869184
"2017-04, 2017-06, 2017-07, 2018-02",predates_documented_pretraining,player,training_prevalence,364,392,0.579082,0.563119,0.244240,0.681593,0.461954,0.560543
"2017-04, 2017-06, 2017-07, 2018-02",predates_documented_pretraining,player,uniform_legal,364,392,0.579082,0.091639,0.470740,1.615026,0.606982,0.706064


... (30 outputs left out)

archive_set,protocol,horizon,feature_set,iteration,split,folds,observation_occurrences,mae,mae_min_fold,mae_max_fold,accuracy_exploratory,accuracy_exploratory_min_fold,accuracy_exploratory_max_fold,no_change_mae,no_change_mae_min_fold,no_change_mae_max_fold,majority_accuracy,majority_accuracy_min_fold,majority_accuracy_max_fold
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",Saved learning-curve split; see split column,6,full_multipv,240,training,3,46334,19.143593,19.101213,19.205253,0.649782,0.646979,0.651494,19.834492,19.763157,19.916033,0.641689,0.638588,0.643411
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",Saved learning-curve split; see split column,6,full_multipv,240,validation,3,23167,19.585325,19.502981,19.704338,0.645919,0.643435,0.650554,19.834492,19.669900,19.975275,0.641689,0.638229,0.647948


**Learning curves 20261110T224023Z 31ae5f8e final iteration**

Training errors are in-sample; validation curves are development diagnostics.

archive_set,protocol,horizon,feature_set,iteration,split,folds,observation_occurrences,mae,mae_min_fold,mae_max_fold,accuracy_exploratory,accuracy_exploratory_min_fold,accuracy_exploratory_max_fold,no_change_mae,no_change_mae_min_fold,no_change_mae_max_fold,majority_accuracy,majority_accuracy_min_fold,majority_accuracy_max_fold
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",Saved learning-curve split; see split column,6,full_multipv,240,training,3,46334,19.143593,19.101213,19.205253,0.649782,0.646979,0.651494,19.834492,19.763157,19.916033,0.641689,0.638588,0.643411
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",Saved learning-curve split; see split column,6,full_multipv,240,validation,3,23167,19.585325,19.502981,19.704338,0.645919,0.643435,0.650554,19.834492,19.669900,19.975275,0.641689,0.638229,0.647948


## 5. Sources and reproducibility

Tables identify the archive set and validation protocol. Studies that reuse rows, folds or models are not independent replications. Feature comparisons and subgroup differences support predictive associations rather than causal conclusions.

### Model settings and historical selections

The following tables record predictor lists, fixed parameters and historical development selections. The fixed reference and historical selected configurations are distinct.


In [16]:
show('fixed_model_predictors',
'<table border="1" class="dataframe">\n  <thead>\n    <tr style="text-align: right;">\n      <th>archive_set</th>\n      <th>protocol</th>\n      <th>model</th>\n      <th>predictors</th>\n    </tr>\n  </thead>\n  <tbody>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Fixed allowlists from source protocol</td>\n      <td>fixed_lightgbm</td>\n      <td>["eval_before_cp", "initial_damage", "player_elo", "opponent_elo", "move_number", "player_clock_before_move", "opponent_clock_before_move", "initial_time_seconds", "increment_seconds", "material_balance_cp", "number_of_pieces", "hanging_pieces", "opponent_hanging_pieces", "king_attackers", "opponent_king_attackers", "reasonable_candidate_moves", "eval_gap_best_second_cp", "in_check", "gives_check", "is_capture", "is_promotion", "game_phase", "multipv_good_reply_count_10pp", "multipv_best_second_gap_pp"]</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Fixed allowlists from source protocol</td>\n      <td>optuna_lightgbm</td>\n      <td>["eval_before_cp", "initial_damage", "player_elo", "opponent_elo", "move_number", "player_clock_before_move", "opponent_clock_before_move", "initial_time_seconds", "increment_seconds", "material_balance_cp", "number_of_pieces", "hanging_pieces", "opponent_hanging_pieces", "king_attackers", "opponent_king_attackers", "reasonable_candidate_moves", "eval_gap_best_second_cp", "in_check", "gives_check", "is_capture", "is_promotion", "game_phase", "multipv_good_reply_count_10pp", "multipv_best_second_gap_pp"]</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Fixed allowlists from source protocol</td>\n      <td>optuna_xgboost</td>\n      <td>["eval_before_cp", "initial_damage", "player_elo", "opponent_elo", "move_number", "player_clock_before_move", "opponent_clock_before_move", "initial_time_seconds", "increment_seconds", "material_balance_cp", "number_of_pieces", "hanging_pieces", "opponent_hanging_pieces", "king_attackers", "opponent_king_attackers", "reasonable_candidate_moves", "eval_gap_best_second_cp", "in_check", "gives_check", "is_capture", "is_promotion", "game_phase", "multipv_good_reply_count_10pp", "multipv_best_second_gap_pp"]</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Fixed allowlists from source protocol</td>\n      <td>decision_tree</td>\n      <td>["eval_before_cp", "initial_damage", "player_elo", "opponent_elo", "move_number", "player_clock_before_move", "opponent_clock_before_move", "initial_time_seconds", "increment_seconds", "material_balance_cp", "number_of_pieces", "hanging_pieces", "opponent_hanging_pieces", "king_attackers", "opponent_king_attackers", "reasonable_candidate_moves", "eval_gap_best_second_cp", "in_check", "gives_check", "is_capture", "is_promotion", "game_phase", "multipv_good_reply_count_10pp", "multipv_best_second_gap_pp"]</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Fixed allowlists from source protocol</td>\n      <td>random_forest</td>\n      <td>["eval_before_cp", "initial_damage", "player_elo", "opponent_elo", "move_number", "player_clock_before_move", "opponent_clock_before_move", "initial_time_seconds", "increment_seconds", "material_balance_cp", "number_of_pieces", "hanging_pieces", "opponent_hanging_pieces", "king_attackers", "opponent_king_attackers", "reasonable_candidate_moves", "eval_gap_best_second_cp", "in_check", "gives_check", "is_capture", "is_promotion", "game_phase", "multipv_good_reply_count_10pp", "multipv_best_second_gap_pp"]</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Fixed allowlists from source protocol</td>\n      <td>engine_only</td>\n      <td>["eval_before_cp", "initial_damage"]</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Fixed allowlists from source protocol</td>\n      <td>engine_ratings</td>\n      <td>["eval_before_cp", "initial_damage", "player_elo", "opponent_elo"]</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Fixed allowlists from source protocol</td>\n      <td>no_change</td>\n      <td>[]</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Fixed allowlists from source protocol</td>\n      <td>training_mean</td>\n      <td>[]</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Fixed allowlists from source protocol</td>\n      <td>training_median</td>\n      <td>[]</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Fixed allowlists from source protocol</td>\n      <td>training_majority</td>\n      <td>[]</td>\n    </tr>\n  </tbody>\n</table>',
'')

show('fixed_model_settings',
'<table border="1" class="dataframe">\n  <thead>\n    <tr style="text-align: right;">\n      <th>archive_set</th>\n      <th>protocol</th>\n      <th>model</th>\n      <th>settings</th>\n    </tr>\n  </thead>\n  <tbody>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Saved fixed settings; seed 1701</td>\n      <td>fixed_lightgbm</td>\n      <td>{"objective": "regression_l1", "n_estimators": 240, "learning_rate": 0.04, "num_leaves": 11, "min_child_samples": 90, "reg_lambda": 7, "verbosity": -1}</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Saved fixed settings; seed 1701</td>\n      <td>decision_tree</td>\n      <td>{"max_depth": 5, "min_samples_leaf": 90}</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Saved fixed settings; seed 1701</td>\n      <td>random_forest</td>\n      <td>{"n_estimators": 150, "min_samples_leaf": 40, "max_features": 0.8}</td>\n    </tr>\n  </tbody>\n</table>',
'')

show('historical_selected_configurations',
'<table border="1" class="dataframe">\n  <thead>\n    <tr style="text-align: right;">\n      <th>archive_set</th>\n      <th>protocol</th>\n      <th>horizon</th>\n      <th>feature_set</th>\n      <th>selection_mean_fold_mae</th>\n      <th>parameters</th>\n    </tr>\n  </thead>\n  <tbody>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Historical 3-fold game CV used for selection; optimistic, not matched outer evaluation</td>\n      <td>2</td>\n      <td>full_multipv</td>\n      <td>12.546288</td>\n      <td>{"objective": "regression_l1", "n_estimators": 120, "learning_rate": 0.04, "num_leaves": 7, "min_child_samples": 40, "reg_lambda": 1, "verbosity": -1}</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Historical 3-fold game CV used for selection; optimistic, not matched outer evaluation</td>\n      <td>4</td>\n      <td>full_multipv</td>\n      <td>16.786366</td>\n      <td>{"objective": "regression_l1", "n_estimators": 120, "learning_rate": 0.04, "num_leaves": 7, "min_child_samples": 40, "reg_lambda": 1, "verbosity": -1}</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Historical 3-fold game CV used for selection; optimistic, not matched outer evaluation</td>\n      <td>6</td>\n      <td>full_multipv</td>\n      <td>19.555519</td>\n      <td>{"objective": "regression_l1", "n_estimators": 120, "learning_rate": 0.04, "num_leaves": 7, "min_child_samples": 40, "reg_lambda": 1, "verbosity": -1}</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Historical 3-fold game CV used for selection; optimistic, not matched outer evaluation</td>\n      <td>10</td>\n      <td>context_multipv</td>\n      <td>23.052400</td>\n      <td>{"objective": "regression_l1", "n_estimators": 400, "learning_rate": 0.02, "num_leaves": 15, "min_child_samples": 40, "reg_lambda": 1, "verbosity": -1}</td>\n    </tr>\n  </tbody>\n</table>',
'')


**Fixed model predictors**

archive_set,protocol,model,predictors
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",Fixed allowlists from source protocol,fixed_lightgbm,"[""eval_before_cp"", ""initial_damage"", ""player_elo"", ""opponent_elo"", ""move_number"", ""player_clock_before_move"", ""opponent_clock_before_move"", ""initial_time_seconds"", ""increment_seconds"", ""material_balance_cp"", ""number_of_pieces"", ""hanging_pieces"", ""opponent_hanging_pieces"", ""king_attackers"", ""opponent_king_attackers"", ""reasonable_candidate_moves"", ""eval_gap_best_second_cp"", ""in_check"", ""gives_check"", ""is_capture"", ""is_promotion"", ""game_phase"", ""multipv_good_reply_count_10pp"", ""multipv_best_second_gap_pp""]"
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",Fixed allowlists from source protocol,optuna_lightgbm,"[""eval_before_cp"", ""initial_damage"", ""player_elo"", ""opponent_elo"", ""move_number"", ""player_clock_before_move"", ""opponent_clock_before_move"", ""initial_time_seconds"", ""increment_seconds"", ""material_balance_cp"", ""number_of_pieces"", ""hanging_pieces"", ""opponent_hanging_pieces"", ""king_attackers"", ""opponent_king_attackers"", ""reasonable_candidate_moves"", ""eval_gap_best_second_cp"", ""in_check"", ""gives_check"", ""is_capture"", ""is_promotion"", ""game_phase"", ""multipv_good_reply_count_10pp"", ""multipv_best_second_gap_pp""]"
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",Fixed allowlists from source protocol,optuna_xgboost,"[""eval_before_cp"", ""initial_damage"", ""player_elo"", ""opponent_elo"", ""move_number"", ""player_clock_before_move"", ""opponent_clock_before_move"", ""initial_time_seconds"", ""increment_seconds"", ""material_balance_cp"", ""number_of_pieces"", ""hanging_pieces"", ""opponent_hanging_pieces"", ""king_attackers"", ""opponent_king_attackers"", ""reasonable_candidate_moves"", ""eval_gap_best_second_cp"", ""in_check"", ""gives_check"", ""is_capture"", ""is_promotion"", ""game_phase"", ""multipv_good_reply_count_10pp"", ""multipv_best_second_gap_pp""]"
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",Fixed allowlists from source protocol,decision_tree,"[""eval_before_cp"", ""initial_damage"", ""player_elo"", ""opponent_elo"", ""move_number"", ""player_clock_before_move"", ""opponent_clock_before_move"", ""initial_time_seconds"", ""increment_seconds"", ""material_balance_cp"", ""number_of_pieces"", ""hanging_pieces"", ""opponent_hanging_pieces"", ""king_attackers"", ""opponent_king_attackers"", ""reasonable_candidate_moves"", ""eval_gap_best_second_cp"", ""in_check"", ""gives_check"", ""is_capture"", ""is_promotion"", ""game_phase"", ""multipv_good_reply_count_10pp"", ""multipv_best_second_gap_pp""]"
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",Fixed allowlists from source protocol,random_forest,"[""eval_before_cp"", ""initial_damage"", ""player_elo"", ""opponent_elo"", ""move_number"", ""player_clock_before_move"", ""opponent_clock_before_move"", ""initial_time_seconds"", ""increment_seconds"", ""material_balance_cp"", ""number_of_pieces"", ""hanging_pieces"", ""opponent_hanging_pieces"", ""king_attackers"", ""opponent_king_attackers"", ""reasonable_candidate_moves"", ""eval_gap_best_second_cp"", ""in_check"", ""gives_check"", ""is_capture"", ""is_promotion"", ""game_phase"", ""multipv_good_reply_count_10pp"", ""multipv_best_second_gap_pp""]"
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",Fixed allowlists from source protocol,engine_only,"[""eval_before_cp"", ""initial_damage""]"
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",Fixed allowlists from source protocol,engine_ratings,"[""eval_before_cp"", ""initial_damage"", ""player_elo"", ""opponent_elo""]"
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",Fixed allowlists from source protocol,no_change,[]
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",Fixed allowlists from source protocol,training_mean,[]
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",Fixed allowlists from source protocol,training_median,[]


**Fixed model settings**

archive_set,protocol,model,settings
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",Saved fixed settings; seed 1701,fixed_lightgbm,"{""objective"": ""regression_l1"", ""n_estimators"": 240, ""learning_rate"": 0.04, ""num_leaves"": 11, ""min_child_samples"": 90, ""reg_lambda"": 7, ""verbosity"": -1}"
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",Saved fixed settings; seed 1701,decision_tree,"{""max_depth"": 5, ""min_samples_leaf"": 90}"
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",Saved fixed settings; seed 1701,random_forest,"{""n_estimators"": 150, ""min_samples_leaf"": 40, ""max_features"": 0.8}"


**Historical selected configurations**

archive_set,protocol,horizon,feature_set,selection_mean_fold_mae,parameters
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02","Historical 3-fold game CV used for selection; optimistic, not matched outer evaluation",2,full_multipv,12.546288,"{""objective"": ""regression_l1"", ""n_estimators"": 120, ""learning_rate"": 0.04, ""num_leaves"": 7, ""min_child_samples"": 40, ""reg_lambda"": 1, ""verbosity"": -1}"
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02","Historical 3-fold game CV used for selection; optimistic, not matched outer evaluation",4,full_multipv,16.786366,"{""objective"": ""regression_l1"", ""n_estimators"": 120, ""learning_rate"": 0.04, ""num_leaves"": 7, ""min_child_samples"": 40, ""reg_lambda"": 1, ""verbosity"": -1}"
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02","Historical 3-fold game CV used for selection; optimistic, not matched outer evaluation",6,full_multipv,19.555519,"{""objective"": ""regression_l1"", ""n_estimators"": 120, ""learning_rate"": 0.04, ""num_leaves"": 7, ""min_child_samples"": 40, ""reg_lambda"": 1, ""verbosity"": -1}"
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02","Historical 3-fold game CV used for selection; optimistic, not matched outer evaluation",10,context_multipv,23.052400,"{""objective"": ""regression_l1"", ""n_estimators"": 400, ""learning_rate"": 0.02, ""num_leaves"": 15, ""min_child_samples"": 40, ""reg_lambda"": 1, ""verbosity"": -1}"


### Source reports and software versions

The source reports provide full methods and saved-result references. The notebook displays an embedded results snapshot and does not recompute metrics.


In [17]:
show('source_software_versions',
'<table border="1" class="dataframe">\n  <thead>\n    <tr style="text-align: right;">\n      <th>archive_set</th>\n      <th>protocol</th>\n      <th>run</th>\n      <th>manifest</th>\n      <th>software</th>\n    </tr>\n  </thead>\n  <tbody>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Provenance only; individual archive sets in source manifests</td>\n      <td>final_optuna_development_20261117_v1</td>\n      <td>protocol.json</td>\n      <td>{"chess": "1.11.2", "lightgbm": "4.6.0", "matplotlib": "3.9.4", "numpy": "2.0.2", "optuna": "4.5.0", "pandas": "2.3.3", "scikit-learn": "1.6.1", "scipy": "1.13.1", "xgboost": "2.1.4"}</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>Provenance only; individual archive sets in source manifests</td>\n      <td>main_maia_horizon_classification_development_20261118_v1</td>\n      <td>protocol.json</td>\n      <td>{"lightgbm": "4.6.0", "numpy": "2.0.2", "pandas": "2.3.3", "scikit-learn": "1.6.1", "scipy": "1.13.1"}</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>Provenance only; individual archive sets in source manifests</td>\n      <td>simple_reply_baseline_development_20261118_v1</td>\n      <td>protocol.json</td>\n      <td>{"lightgbm": "4.6.0", "numpy": "2.0.2", "pandas": "2.3.3", "scikit-learn": "1.6.1", "scipy": "1.13.1"}</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>Provenance only; individual archive sets in source manifests</td>\n      <td>maia_bounded_four_archive_20261117_v1</td>\n      <td>protocol.json</td>\n      <td>{"chess": "1.11.2", "maia2": "0.11.0", "matplotlib": "3.9.4", "numpy": "2.5.5", "pandas": "2.3.3", "scikit-learn": "1.6.1", "scipy": "1.18.1", "torch": "2.8.0"}</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>Provenance only; individual archive sets in source manifests</td>\n      <td>maia_bounded_four_archive_20261117_v1</td>\n      <td>regression_protocol.json</td>\n      <td>{"lightgbm": "4.6.0", "numpy": "2.0.2", "pandas": "2.3.3", "scikit-learn": "1.6.1", "scipy": "1.13.1"}</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Provenance only; individual archive sets in source manifests</td>\n      <td>pre_reply_features_development_20261117_v1</td>\n      <td>protocol.json</td>\n      <td>{"chess": "1.11.2", "lightgbm": "4.6.0", "numpy": "2.0.2", "pandas": "2.3.3", "scikit-learn": "1.6.1", "scipy": "1.13.1"}</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>Provenance only; individual archive sets in source manifests</td>\n      <td>focused_recovery_classifiers_development_20261118_v1</td>\n      <td>protocol.json</td>\n      <td>{"lightgbm": "4.6.0", "numpy": "2.0.2", "pandas": "2.3.3", "scikit-learn": "1.6.1", "scipy": "1.13.1"}</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>Provenance only; individual archive sets in source manifests</td>\n      <td>combined_maia_features_development_20261117_v1</td>\n      <td>protocol.json</td>\n      <td>{"lightgbm": "4.6.0", "numpy": "2.0.2", "pandas": "2.3.3", "scikit-learn": "1.6.1", "scipy": "1.13.1"}</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>Provenance only; individual archive sets in source manifests</td>\n      <td>continuous_maia_reply_development_20261117_v1</td>\n      <td>protocol.json</td>\n      <td>{"lightgbm": "4.6.0", "numpy": "2.0.2", "pandas": "2.3.3", "scikit-learn": "1.6.1", "scipy": "1.13.1"}</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Provenance only; individual archive sets in source manifests</td>\n      <td>human_reply_policy_development_20261117_v1</td>\n      <td>protocol.json</td>\n      <td>{"chess": "1.11.2", "maia2": "0.11.0", "matplotlib": "3.9.4", "numpy": "2.5.5", "pandas": "2.3.3", "scikit-learn": "1.6.1", "scipy": "1.18.1", "torch": "2.8.0"}</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02</td>\n      <td>Provenance only; individual archive sets in source manifests</td>\n      <td>maia_reply_recall_gap_development_20261118_v1</td>\n      <td>protocol.json</td>\n      <td>{"numpy": "2.0.2", "pandas": "2.3.3"}</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Provenance only; individual archive sets in source manifests</td>\n      <td>immediate_opponent_reply_development_20261117_v1</td>\n      <td>manifest.json</td>\n      <td>{"numpy": "2.0.2", "pandas": "2.3.3", "scipy": "1.13.1"}</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Provenance only; individual archive sets in source manifests</td>\n      <td>opponent_reply_targets_development_v1</td>\n      <td>manifest.json</td>\n      <td>{"pandas": "2.3.3", "python-chess": "1.999"}</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Provenance only; individual archive sets in source manifests</td>\n      <td>opponent_reply_models_game_v1</td>\n      <td>manifest.json</td>\n      <td>{"lightgbm": "4.6.0", "numpy": "2.0.2", "pandas": "2.3.3", "scikit-learn": "1.6.1", "scipy": "1.13.1", "xgboost": "2.1.4"}</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Provenance only; individual archive sets in source manifests</td>\n      <td>opponent_reply_models_player_v1</td>\n      <td>manifest.json</td>\n      <td>{"lightgbm": "4.6.0", "numpy": "2.0.2", "pandas": "2.3.3", "scikit-learn": "1.6.1", "scipy": "1.13.1", "xgboost": "2.1.4"}</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Provenance only; individual archive sets in source manifests</td>\n      <td>recovery_threshold_nested_game_v1</td>\n      <td>manifest.json</td>\n      <td>{"lightgbm": "4.6.0", "matplotlib": "3.9.4", "numpy": "2.0.2", "pandas": "2.3.3", "scikit-learn": "1.6.1", "scipy": "1.13.1"}</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Provenance only; individual archive sets in source manifests</td>\n      <td>recovery_threshold_nested_player_v1</td>\n      <td>manifest.json</td>\n      <td>{"lightgbm": "4.6.0", "matplotlib": "3.9.4", "numpy": "2.0.2", "pandas": "2.3.3", "scikit-learn": "1.6.1", "scipy": "1.13.1"}</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Provenance only; individual archive sets in source manifests</td>\n      <td>recovery_classifier_nested_game_v1</td>\n      <td>manifest.json</td>\n      <td>{"lightgbm": "4.6.0", "matplotlib": "3.9.4", "numpy": "2.0.2", "pandas": "2.3.3", "scikit-learn": "1.6.1", "scipy": "1.13.1"}</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Provenance only; individual archive sets in source manifests</td>\n      <td>recovery_classifier_nested_player_v1</td>\n      <td>manifest.json</td>\n      <td>{"lightgbm": "4.6.0", "matplotlib": "3.9.4", "numpy": "2.0.2", "pandas": "2.3.3", "scikit-learn": "1.6.1", "scipy": "1.13.1"}</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Provenance only; individual archive sets in source manifests</td>\n      <td>xgboost_lightgbm_nested_game_v1</td>\n      <td>manifest.json</td>\n      <td>{"lightgbm": "4.6.0", "matplotlib": "3.9.4", "numpy": "2.0.2", "pandas": "2.3.3", "scikit-learn": "1.6.1", "scipy": "1.13.1", "xgboost": "2.1.4"}</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Provenance only; individual archive sets in source manifests</td>\n      <td>xgboost_lightgbm_nested_game_v2</td>\n      <td>manifest.json</td>\n      <td>{"lightgbm": "4.6.0", "matplotlib": "3.9.4", "numpy": "2.0.2", "pandas": "2.3.3", "scikit-learn": "1.6.1", "scipy": "1.13.1", "xgboost": "2.1.4"}</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Provenance only; individual archive sets in source manifests</td>\n      <td>learning_curves_20261110T222901Z_d3c7e778</td>\n      <td>manifest.json</td>\n      <td>{"lightgbm": "4.6.0", "matplotlib": "3.9.4", "numpy": "2.0.2", "pandas": "2.3.3", "scikit-learn": "1.6.1", "scipy": "1.13.1"}</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Provenance only; individual archive sets in source manifests</td>\n      <td>learning_curves_20261110T223938Z_6ed4aaf1</td>\n      <td>manifest.json</td>\n      <td>{"lightgbm": "4.6.0", "matplotlib": "3.9.4", "numpy": "2.0.2", "pandas": "2.3.3", "scikit-learn": "1.6.1", "scipy": "1.13.1"}</td>\n    </tr>\n    <tr>\n      <td>2017-04, 2017-06, 2017-07, 2018-02, 2019-02</td>\n      <td>Provenance only; individual archive sets in source manifests</td>\n      <td>learning_curves_20261110T224023Z_31ae5f8e</td>\n      <td>manifest.json</td>\n      <td>{"lightgbm": "4.6.0", "matplotlib": "3.9.4", "numpy": "2.0.2", "pandas": "2.3.3", "scikit-learn": "1.6.1", "scipy": "1.13.1"}</td>\n    </tr>\n  </tbody>\n</table>',
'')

display(Markdown('<details>\n<summary>Source reports and saved runs</summary>\n\n- [final_optuna_development_20261117_v1](notebook_runs/final_optuna_development_20261117_v1/REPORT.md)\n- [main_maia_horizon_classification_development_20261118_v1](notebook_runs/main_maia_horizon_classification_development_20261118_v1/REPORT.md)\n- [simple_reply_baseline_development_20261118_v1](notebook_runs/simple_reply_baseline_development_20261118_v1/REPORT.md)\n- [maia_bounded_four_archive_20261117_v1](notebook_runs/maia_bounded_four_archive_20261117_v1/REPORT.md)\n- [pre_reply_features_development_20261117_v1](notebook_runs/pre_reply_features_development_20261117_v1/REPORT.md)\n- [focused_recovery_classifiers_development_20261118_v1](notebook_runs/focused_recovery_classifiers_development_20261118_v1/REPORT.md)\n- [combined_maia_features_development_20261117_v1](notebook_runs/combined_maia_features_development_20261117_v1/REPORT.md)\n- [continuous_maia_reply_development_20261117_v1](notebook_runs/continuous_maia_reply_development_20261117_v1/REPORT.md)\n- [human_reply_policy_development_20261117_v1](notebook_runs/human_reply_policy_development_20261117_v1/REPORT.md)\n- [maia_budget_simulation_20261117_v1](notebook_runs/maia_budget_simulation_20261117_v1/REPORT.md)\n- [maia_reply_classification_development_20261117_v1](notebook_runs/maia_reply_classification_development_20261117_v1/REPORT.md)\n- [maia_reply_subgroups_development_20261118_v1](notebook_runs/maia_reply_subgroups_development_20261118_v1/REPORT.md)\n- [maia_reply_recall_gap_development_20261118_v1](notebook_runs/maia_reply_recall_gap_development_20261118_v1/REPORT.md)\n- [maia_reply_probability_plots_development_20261118_v1](notebook_runs/maia_reply_probability_plots_development_20261118_v1/REPORT.md)\n- [predictions_within10pp_development_20261117_v1](notebook_runs/predictions_within10pp_development_20261117_v1/REPORT.md)\n- [immediate_opponent_reply_development_20261117_v1](notebook_runs/immediate_opponent_reply_development_20261117_v1/REPORT.md)\n- [immediate_reply_tables_development_20261117_v1](notebook_runs/immediate_reply_tables_development_20261117_v1/TABLES.md)\n- [opponent_reply_targets_development_v1](notebook_runs/opponent_reply_targets_development_v1/)\n- [opponent_reply_analysis_development_v1](notebook_runs/opponent_reply_analysis_development_v1/)\n- [opponent_reply_models_game_v1](notebook_runs/opponent_reply_models_game_v1/)\n- [opponent_reply_models_player_v1](notebook_runs/opponent_reply_models_player_v1/)\n- [recovery_threshold_nested_game_v1](notebook_runs/recovery_threshold_nested_game_v1/)\n- [recovery_threshold_nested_player_v1](notebook_runs/recovery_threshold_nested_player_v1/)\n- [recovery_classifier_nested_game_v1](notebook_runs/recovery_classifier_nested_game_v1/)\n- [recovery_classifier_nested_player_v1](notebook_runs/recovery_classifier_nested_player_v1/)\n- [xgboost_lightgbm_nested_game_v1](notebook_runs/xgboost_lightgbm_nested_game_v1/)\n- [xgboost_lightgbm_nested_game_v2](notebook_runs/xgboost_lightgbm_nested_game_v2/)\n- [learning_curves_20261110T222901Z_d3c7e778](notebook_runs/learning_curves_20261110T222901Z_d3c7e778/)\n- [learning_curves_20261110T223938Z_6ed4aaf1](notebook_runs/learning_curves_20261110T223938Z_6ed4aaf1/)\n- [learning_curves_20261110T224023Z_31ae5f8e](notebook_runs/learning_curves_20261110T224023Z_31ae5f8e/)\n\n</details>'))


**Source software versions**

archive_set,protocol,run,manifest,software
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",Provenance only; individual archive sets in source manifests,final_optuna_development_20261117_v1,protocol.json,"{""chess"": ""1.11.2"", ""lightgbm"": ""4.6.0"", ""matplotlib"": ""3.9.4"", ""numpy"": ""2.0.2"", ""optuna"": ""4.5.0"", ""pandas"": ""2.3.3"", ""scikit-learn"": ""1.6.1"", ""scipy"": ""1.13.1"", ""xgboost"": ""2.1.4""}"
"2017-04, 2017-06, 2017-07, 2018-02",Provenance only; individual archive sets in source manifests,main_maia_horizon_classification_development_20261118_v1,protocol.json,"{""lightgbm"": ""4.6.0"", ""numpy"": ""2.0.2"", ""pandas"": ""2.3.3"", ""scikit-learn"": ""1.6.1"", ""scipy"": ""1.13.1""}"
"2017-04, 2017-06, 2017-07, 2018-02",Provenance only; individual archive sets in source manifests,simple_reply_baseline_development_20261118_v1,protocol.json,"{""lightgbm"": ""4.6.0"", ""numpy"": ""2.0.2"", ""pandas"": ""2.3.3"", ""scikit-learn"": ""1.6.1"", ""scipy"": ""1.13.1""}"
"2017-04, 2017-06, 2017-07, 2018-02",Provenance only; individual archive sets in source manifests,maia_bounded_four_archive_20261117_v1,protocol.json,"{""chess"": ""1.11.2"", ""maia2"": ""0.11.0"", ""matplotlib"": ""3.9.4"", ""numpy"": ""2.5.5"", ""pandas"": ""2.3.3"", ""scikit-learn"": ""1.6.1"", ""scipy"": ""1.18.1"", ""torch"": ""2.8.0""}"
"2017-04, 2017-06, 2017-07, 2018-02",Provenance only; individual archive sets in source manifests,maia_bounded_four_archive_20261117_v1,regression_protocol.json,"{""lightgbm"": ""4.6.0"", ""numpy"": ""2.0.2"", ""pandas"": ""2.3.3"", ""scikit-learn"": ""1.6.1"", ""scipy"": ""1.13.1""}"
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",Provenance only; individual archive sets in source manifests,pre_reply_features_development_20261117_v1,protocol.json,"{""chess"": ""1.11.2"", ""lightgbm"": ""4.6.0"", ""numpy"": ""2.0.2"", ""pandas"": ""2.3.3"", ""scikit-learn"": ""1.6.1"", ""scipy"": ""1.13.1""}"
"2017-04, 2017-06, 2017-07, 2018-02",Provenance only; individual archive sets in source manifests,focused_recovery_classifiers_development_20261118_v1,protocol.json,"{""lightgbm"": ""4.6.0"", ""numpy"": ""2.0.2"", ""pandas"": ""2.3.3"", ""scikit-learn"": ""1.6.1"", ""scipy"": ""1.13.1""}"
"2017-04, 2017-06, 2017-07, 2018-02",Provenance only; individual archive sets in source manifests,combined_maia_features_development_20261117_v1,protocol.json,"{""lightgbm"": ""4.6.0"", ""numpy"": ""2.0.2"", ""pandas"": ""2.3.3"", ""scikit-learn"": ""1.6.1"", ""scipy"": ""1.13.1""}"
"2017-04, 2017-06, 2017-07, 2018-02",Provenance only; individual archive sets in source manifests,continuous_maia_reply_development_20261117_v1,protocol.json,"{""lightgbm"": ""4.6.0"", ""numpy"": ""2.0.2"", ""pandas"": ""2.3.3"", ""scikit-learn"": ""1.6.1"", ""scipy"": ""1.13.1""}"
"2017-04, 2017-06, 2017-07, 2018-02, 2019-02",Provenance only; individual archive sets in source manifests,human_reply_policy_development_20261117_v1,protocol.json,"{""chess"": ""1.11.2"", ""maia2"": ""0.11.0"", ""matplotlib"": ""3.9.4"", ""numpy"": ""2.5.5"", ""pandas"": ""2.3.3"", ""scikit-learn"": ""1.6.1"", ""scipy"": ""1.18.1"", ""torch"": ""2.8.0""}"


<details>
<summary>Source reports and saved runs</summary>

- [final_optuna_development_20261117_v1](notebook_runs/final_optuna_development_20261117_v1/REPORT.md)
- [main_maia_horizon_classification_development_20261118_v1](notebook_runs/main_maia_horizon_classification_development_20261118_v1/REPORT.md)
- [simple_reply_baseline_development_20261118_v1](notebook_runs/simple_reply_baseline_development_20261118_v1/REPORT.md)
- [maia_bounded_four_archive_20261117_v1](notebook_runs/maia_bounded_four_archive_20261117_v1/REPORT.md)
- [pre_reply_features_development_20261117_v1](notebook_runs/pre_reply_features_development_20261117_v1/REPORT.md)
- [focused_recovery_classifiers_development_20261118_v1](notebook_runs/focused_recovery_classifiers_development_20261118_v1/REPORT.md)
- [combined_maia_features_development_20261117_v1](notebook_runs/combined_maia_features_development_20261117_v1/REPORT.md)
- [continuous_maia_reply_development_20261117_v1](notebook_runs/continuous_maia_reply_development_20261117_v1/REPORT.md)
- [human_reply_policy_development_20261117_v1](notebook_runs/human_reply_policy_development_20261117_v1/REPORT.md)
- [maia_budget_simulation_20261117_v1](notebook_runs/maia_budget_simulation_20261117_v1/REPORT.md)
- [maia_reply_classification_development_20261117_v1](notebook_runs/maia_reply_classification_development_20261117_v1/REPORT.md)
- [maia_reply_subgroups_development_20261118_v1](notebook_runs/maia_reply_subgroups_development_20261118_v1/REPORT.md)
- [maia_reply_recall_gap_development_20261118_v1](notebook_runs/maia_reply_recall_gap_development_20261118_v1/REPORT.md)
- [maia_reply_probability_plots_development_20261118_v1](notebook_runs/maia_reply_probability_plots_development_20261118_v1/REPORT.md)
- [predictions_within10pp_development_20261117_v1](notebook_runs/predictions_within10pp_development_20261117_v1/REPORT.md)
- [immediate_opponent_reply_development_20261117_v1](notebook_runs/immediate_opponent_reply_development_20261117_v1/REPORT.md)
- [immediate_reply_tables_development_20261117_v1](notebook_runs/immediate_reply_tables_development_20261117_v1/TABLES.md)
- [opponent_reply_targets_development_v1](notebook_runs/opponent_reply_targets_development_v1/)
- [opponent_reply_analysis_development_v1](notebook_runs/opponent_reply_analysis_development_v1/)
- [opponent_reply_models_game_v1](notebook_runs/opponent_reply_models_game_v1/)
- [opponent_reply_models_player_v1](notebook_runs/opponent_reply_models_player_v1/)
- [recovery_threshold_nested_game_v1](notebook_runs/recovery_threshold_nested_game_v1/)
- [recovery_threshold_nested_player_v1](notebook_runs/recovery_threshold_nested_player_v1/)
- [recovery_classifier_nested_game_v1](notebook_runs/recovery_classifier_nested_game_v1/)
- [recovery_classifier_nested_player_v1](notebook_runs/recovery_classifier_nested_player_v1/)
- [xgboost_lightgbm_nested_game_v1](notebook_runs/xgboost_lightgbm_nested_game_v1/)
- [xgboost_lightgbm_nested_game_v2](notebook_runs/xgboost_lightgbm_nested_game_v2/)
- [learning_curves_20261110T222901Z_d3c7e778](notebook_runs/learning_curves_20261110T222901Z_d3c7e778/)
- [learning_curves_20261110T223938Z_6ed4aaf1](notebook_runs/learning_curves_20261110T223938Z_6ed4aaf1/)
- [learning_curves_20261110T224023Z_31ae5f8e](notebook_runs/learning_curves_20261110T224023Z_31ae5f8e/)

</details>

## Final evaluation documents

- [March 2025 evaluation plan](MARCH_2025_EVALUATION_PLAN.md)
- [Technical appendix](MARCH_2025_EVALUATION_APPENDIX.md)

These documents preserve the pre-test plan. The final evaluation was completed on 21 November 2026; results are in [chess_final_results.ipynb](chess_final_results.ipynb). The tables above contain development results only.
